# E23 association collection: 36 previously exposed train videos
Private diagnostic, fixed E23, nine serial groups, no GT reads/training/submission.csv.
Source/model packs by pilkwang (CC0 as recorded in project evidence).
Public-four OFF/ON parity passed; this run is ON-only with per-video frozen reference checks.
Train GT remains on the competition mount: image-only views are not OS isolation.


In [ ]:
import time
D3_STARTED = time.monotonic()


In [ ]:
import os
import numpy as np
from scipy.spatial import cKDTree

BIOHUB_PRESET = 'harmonic_mutual_support_association_fusion_v1'
BIOHUB_SCORE_AXIS = 'weighted harmonic forward/reverse association consensus on the fixed-90 dual-seed baseline'

# Parameters for constrained lineage reconstruction.
os.environ["BIOHUB_OUTPUT_FILTER_SHORT_TRACKS"] = "1"
os.environ["BIOHUB_DET_THRESHOLD"] = "0.96875"
os.environ["BIOHUB_MOTION_RELINK_LEARNED_BONUS"] = '1.0'
os.environ["BIOHUB_ILP_APPEARANCE_WEIGHT"] = "0.0"
os.environ["BIOHUB_ILP_DISAPPEARANCE_WEIGHT"] = "1.5"
os.environ["BIOHUB_ILP_DIVISION_WEIGHT"] = "1.0"  # REVIEW: reverted after testing -- 0.3/1.0/2.0/3.0 all scored 0.915 on the real leaderboard; local proxy dropped at 0.3 (messier graph, not more true divisions), so staying at the neutral, proven value
os.environ["BIOHUB_GAP_CLOSE_MAX_GAP"] = "2"
os.environ["BIOHUB_GAP_CLOSE_UM"] = "5.8"
os.environ["BIOHUB_GAP_DENSITY_ADAPTIVE"] = "1"
os.environ["BIOHUB_GAP_DENSITY_REFERENCE_UM"] = "6.5"
os.environ["BIOHUB_GAP_DENSITY_GAIN"] = "0.040"
os.environ["BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM"] = "0.125"
os.environ["BIOHUB_GAP_DENSITY_NEIGHBORS"] = "3"
os.environ["BIOHUB_OUTPUT_MIN_TRACK_LEN"] = "6"
os.environ["BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS"] = "1"
os.environ["BIOHUB_OUTPUT_GAP2_RECOVERY"] = "0"
# REVIEW: reverted, was "4.66"  -- tested at 7.0, local proxy improved (first non-zero division_jaccard) but real score dropped to 0.914. Likely explained by validator/test-set division-content mismatch, not a validator bug -- back to the proven value.
os.environ["BIOHUB_SAFE_DIV_MAX_UM"] = "8.0"  # REVIEW: widened to match the 0.917 notebook directly. The earlier 7.0 test (pre-divergence-check) added noise that pure geometry couldn't filter out; divergence + mutual-NN are active now and should do that filtering instead.
os.environ["BIOHUB_SAFE_DIV_SISTER_MAX_UM"] = "11.0"  # REVIEW: matches 0.917, was "8.5"
os.environ["BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM"] = "10.0"  # REVIEW: matches 0.917, was "7.65"
os.environ["BIOHUB_SAFE_DIV_FRAME_FRAC_CAP"] = "0.0076"  # unchanged -- never bound at 0.920 (cap_skipped=0), leaving it as the safety net for this test
os.environ["BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP"] = "0.00375"  # unchanged, same reasoning
os.environ["BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE"] = "0"
os.environ["BIOHUB_USE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_REQUIRE_DEEPCENTER_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_EXPECTED_EPOCH"] = "2"  # REVIEW: best.pt is epoch 2, not checkpoint_last.pt's epoch 500
os.environ["BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM"] = "8.5"
os.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt"
os.environ["BIOHUB_DEEPCENTER_GAP_VETO"] = "1"
os.environ["BIOHUB_DEEPCENTER_GAP_THRESHOLD"] = "0.25"
os.environ["BIOHUB_DEEPCENTER_SAFE_DIV_VETO"] = '1'  # REVIEW: was 0; funnel telemetry showed safe-div accepting 81-100% of its own candidates -- turning on the one gate built specifically to catch that
os.environ["BIOHUB_RUN_OUTPUT_DIAGNOSTICS"] = "0"

# Exact candidate-arm settings formerly supplied by the diagnostic harness.
os.environ["BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT"] = "0.30"  # REVIEW: was 0.20 (0.915 reference value, never tuned). Harmonic fusion is the one mechanism with proven benefit (+0.002 in the reference notebook) -- testing whether more weight on the reverse-time evidence extracts more of it, rather than exploring another unproven lever.
os.environ["BIOHUB_BIDIRECTIONAL_FUSION_MODE"] = "harmonic_probability"
os.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"] = "0.90"
os.environ["BIOHUB_DIAGNOSTIC_ARM"] = "harmonic_association_production"

print("BIOHUB_PRESET:", BIOHUB_PRESET)
print("BIOHUB_SCORE_AXIS:", BIOHUB_SCORE_AXIS)

In [ ]:
# Configuration guard: assert the single intended model-level change.
import json as _guard_json
import math as _guard_math
import os as _guard_os

_EXPECTED_NUMERIC = {
    "BIOHUB_DET_THRESHOLD": 0.96875,
    "BIOHUB_ILP_APPEARANCE_WEIGHT": 0.0,
    "BIOHUB_ILP_DISAPPEARANCE_WEIGHT": 1.5,
    "BIOHUB_GAP_CLOSE_UM": 5.8,
    "BIOHUB_OUTPUT_MIN_TRACK_LEN": 6.0,
    "BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT": 0.30,
}

_EXPECTED_TEXT = {
    "BIOHUB_BIDIRECTIONAL_FUSION_MODE": "harmonic_probability",
    "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION": "0.90",
}

_drift = {}
for _key, _want in _EXPECTED_NUMERIC.items():
    _raw = _guard_os.environ.get(_key)
    if _raw is None:
        _drift[_key] = "missing"
        continue
    _got = float(_raw)
    if not _guard_math.isclose(_got, _want, rel_tol=0.0, abs_tol=1e-12):
        _drift[_key] = {"expected": _want, "actual": _got}

for _key, _want in _EXPECTED_TEXT.items():
    _got = _guard_os.environ.get(_key)
    if _got != _want:
        _drift[_key] = {"expected": _want, "actual": _got}

if _drift:
    raise RuntimeError(
        "Configuration drift detected: " + _guard_json.dumps(_drift, sort_keys=True)
    )

print("Configuration guard: PASS")
print("Baseline: fixed-90 dual-seed clean pipeline (public LB 0.913)")
print("Single model-level change: harmonic mutual-support association fusion")
print("Reverse-time association weight: 0.200")

In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
# Hard-bound production input: competition test images only.
TEST_DIR = COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"

METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
EXPERIMENT_TAG = "selected_101_dual_seed_near_balanced_center_confirmed_synthetic_gap"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"

DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))

# Hard-bound full production coverage. Internal GPU sharding remains exhaustive.
SLICE = ""

# If dependencies are not already installed and no offline wheels are attached,
# this controls whether the notebook attempts PyPI installation.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"

# Output-level graph post-processing.
OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))
GAP_DENSITY_ADAPTIVE = os.environ.get("BIOHUB_GAP_DENSITY_ADAPTIVE", "0") != "0"
GAP_DENSITY_REFERENCE_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_REFERENCE_UM", "6.5"))
GAP_DENSITY_GAIN = float(os.environ.get("BIOHUB_GAP_DENSITY_GAIN", "0.040"))
GAP_DENSITY_MAX_STEP_DELTA_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM", "0.125"))
GAP_DENSITY_NEIGHBORS = int(os.environ.get("BIOHUB_GAP_DENSITY_NEIGHBORS", "3"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "1") != "0"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "6"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"
ADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get("BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE", "0") != "0"
SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC", "0.10"))
SHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN", "4"))
SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB", "0.82"))
SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM", "3.25"))
SHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC", "0.018"))
SHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS", "180"))

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "0"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.2"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.4"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0045"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "180"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "8.0"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "11.0"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "10.0"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))

# --- new config: add next to the existing SAFE_DIV_* block ---
SAFE_DIV_DIVERGE_UM = float(os.environ.get("BIOHUB_SAFE_DIV_DIVERGE_UM", "2.25"))
SAFE_DIV_REQUIRE_DIVERGENCE = os.environ.get("BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE", "1") != "0"
SAFE_DIV_REQUIRE_MUTUAL_NN = os.environ.get("BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN", "1") != "0"

# DeepCenter support is retained for compatibility, but this selected run keeps it disabled.
USE_DEEPCENTER_VETO = os.environ.get("BIOHUB_USE_DEEPCENTER_VETO", "1") != "0"
REQUIRE_DEEPCENTER_VETO = os.environ.get("BIOHUB_REQUIRE_DEEPCENTER_VETO", "1") != "0"
DEEPCENTER_MANIFEST_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json",
)
DEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get(
    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT",
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt",
)
DEEPCENTER_RELATIVE = os.environ.get("BIOHUB_DEEPCENTER_RELATIVE", "weights/full_frame_center/best.pt")
DEEPCENTER_GAP_VETO = os.environ.get("BIOHUB_DEEPCENTER_GAP_VETO", "1") != "0"
DEEPCENTER_SAFE_DIV_VETO = os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_VETO", "1") != "0"
DEEPCENTER_GAP_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_THRESHOLD", "0.10"))
DEEPCENTER_EXPECTED_EPOCH = int(os.environ.get("BIOHUB_DEEPCENTER_EXPECTED_EPOCH", "0"))
DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM", "0"))
DEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD", "0.12"))
DEEPCENTER_SCORE_WIN_Z = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_Z", "1"))
DEEPCENTER_SCORE_WIN_YX = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_YX", "2"))
DEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES", "8"))

CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_density_adaptive": GAP_DENSITY_ADAPTIVE,
    "gap_density_reference_um": GAP_DENSITY_REFERENCE_UM,
    "gap_density_gain": GAP_DENSITY_GAIN,
    "gap_density_max_step_delta_um": GAP_DENSITY_MAX_STEP_DELTA_UM,
    "gap_density_neighbors": GAP_DENSITY_NEIGHBORS,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "adaptive_short_track_rescue": ADAPTIVE_SHORT_TRACK_RESCUE,
    "short_track_rescue_trigger_removed_frac": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC,
    "short_track_rescue_min_len": SHORT_TRACK_RESCUE_MIN_LEN,
    "short_track_rescue_min_mean_edge_prob": SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB,
    "short_track_rescue_max_mean_edge_dist_um": SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM,
    "short_track_rescue_max_nodes_frac": SHORT_TRACK_RESCUE_MAX_NODES_FRAC,
    "short_track_rescue_max_nodes_abs": SHORT_TRACK_RESCUE_MAX_NODES_ABS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
    "use_deepcenter_add_only_gate": USE_DEEPCENTER_VETO,
    "deepcenter_gap_add_gate": DEEPCENTER_GAP_VETO,
    "deepcenter_safe_div_add_gate": DEEPCENTER_SAFE_DIV_VETO,
    "deepcenter_gap_threshold": DEEPCENTER_GAP_THRESHOLD,
    "deepcenter_expected_epoch": DEEPCENTER_EXPECTED_EPOCH,
    "deepcenter_gap_confirm_min_span_um": DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM,
    "deepcenter_safe_div_threshold": DEEPCENTER_SAFE_DIV_THRESHOLD,
    "deepcenter_checkpoint_default": DEEPCENTER_CHECKPOINT_DEFAULT,
}

print("Biohub learned UNet + node-transformer + ILP submission")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

In [ ]:
D3_EXPECTED_DATASETS = ['44b6_12dfb391', '44b6_267148e4', '44b6_2a2eff9f', '44b6_341df25f', '44b6_587a1e22', '44b6_5f15d135', '44b6_706092f0', '44b6_74d0c52e', '44b6_7a302da0', '44b6_996155de', '44b6_9be80b04', '44b6_a21120c2', '44b6_aaf8b0ea', '44b6_c50204e0', '44b6_c8e2a523', '44b6_d2f34f90', '44b6_d5e7d891', '44b6_d754aa59', '6bba_062c8d37', '6bba_07e24132', '6bba_085bf656', '6bba_09961292', '6bba_0e7c0d07', '6bba_12665c0e', '6bba_1d0d8384', '6bba_207c6aaf', '6bba_20852818', '6bba_2312ac41', '6bba_268e1230', '6bba_2819ca14', '6bba_32db13fc', '6bba_337b1b3a', '6bba_3abfe10a', '6bba_3c5691b6', '6bba_3db54e20', '6bba_3fda6b25']
D3_TRAIN_IMAGES = COMP_DIR / "train"
assert D3_TRAIN_IMAGES.is_dir()
assert all((D3_TRAIN_IMAGES / (name + ".zarr")).is_dir() for name in D3_EXPECTED_DATASETS)
print("D3 COLLECTION image paths present", len(D3_EXPECTED_DATASETS), flush=True)


In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]

# The safe path for offline reruns is to use attached wheels.
# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))
    _expected_primary_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
    _actual_primary_sha256 = str(manifest_info.get("model", {}).get("weight_sha256", ""))
    if _actual_primary_sha256 != _expected_primary_sha256:
        raise RuntimeError(
            "Primary model checksum mismatch: "
            f"expected {_expected_primary_sha256}, got {_actual_primary_sha256 or 'missing'}"
        )

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)

# Verify every materialized support-repo Python source before the next cell
# performs the sealed dynamic inference patch. Also hash the actual primary and
# DeepCenter checkpoint bytes that the production path will use.
import hashlib as _integrity_hashlib

_support_expected_sha256 = {
    "scripts/augmentations.py": "13db09817bf492f8d0f710a0a4d09776320b262060167055090a303fc6057f4e",
    "scripts/dataspec.py": "e69bf952fb985477ac50ff8598a35020c95d20a035a09b81ab4056e655dd311f",
    "scripts/evaluate.py": "614813cc51c3581c6ccda4bb20725a19da8ecac4a27620654bfca58319cffa3c",
    "scripts/predict_unet_transformer.py": "c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9",
    "scripts/train_unet_transformer.py": "c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea",
    "src/biohub_tracking/__init__.py": "26a18d8da84e40da73281a48ebc3017d847a2e57431ab63e8629d2109e6e8571",
    "src/biohub_tracking/division_metrics.py": "d1cf1e0a43009d02174f1699ce2aa28458a2220ac4b521731d3bcf31cf8c76be",
    "src/biohub_tracking/img_proc.py": "00e8ef0adc8b39f1aaaa547ea6197b906bf9e8c009e339d3e95f8f8dbf31be3f",
    "src/biohub_tracking/io.py": "efae135b088cecaab463d889f16c885ef6da3ad27b0747327d8ddc28d866b7bd",
    "src/biohub_tracking/metrics.py": "31baf45b54c78f68bab4f65dd8f4b38bca702abb644171c6df7c46cdeef55d83",
    "src/biohub_tracking/models/__init__.py": "ab7587ef79856bae50d24b62e5805092d0459ee1c586522b763f9ef70c093e1d",
    "src/biohub_tracking/models/simple_node_transformer.py": "b97209edeb03840e80d903e3e2a8c81c520641c8ef343f6ca2904d0f80db064e",
    "src/biohub_tracking/models/temporal_unet.py": "d809c35d42f504161074ddeaaa7aee5b407e5bca7f9b4e1d5f9b2ff345666cac"
}
_support_expected_manifest_sha256 = "978b626d1fd1e7397435a437dfe68691defe1572fc3c20e61012d7c9b52ed029"
_primary_expected_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
_deepcenter_expected_sha256 = "8040999a92f6b7bbd98fa8cf458141e045c0f9ad7c936bdb3b18e1f7edafe2a0"  # best.pt (epoch 2), not checkpoint_last.pt (epoch 500)


def _integrity_sha256_file(path: Path) -> str:
    digest = _integrity_hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


_support_materialized_paths = {
    path.relative_to(REPO_DIR).as_posix(): path
    for path in REPO_DIR.rglob("*.py")
}
_support_actual_names = set(_support_materialized_paths)
_support_expected_names = set(_support_expected_sha256)
if _support_actual_names != _support_expected_names:
    raise RuntimeError({
        "support_repo_python_files_missing": sorted(
            _support_expected_names - _support_actual_names
        ),
        "support_repo_python_files_extra": sorted(
            _support_actual_names - _support_expected_names
        ),
    })
_support_actual_sha256 = {
    relative: _integrity_sha256_file(_support_materialized_paths[relative])
    for relative in sorted(_support_materialized_paths)
}
if _support_actual_sha256 != _support_expected_sha256:
    raise RuntimeError({
        "support_repo_python_checksum_mismatch": {
            relative: {
                "expected": _support_expected_sha256[relative],
                "actual": _support_actual_sha256[relative],
            }
            for relative in sorted(_support_expected_sha256)
            if _support_actual_sha256[relative]
            != _support_expected_sha256[relative]
        }
    })
_support_manifest_bytes = "".join(
    f"{_support_actual_sha256[relative]}  {relative}\n"
    for relative in sorted(_support_actual_sha256)
).encode("utf-8")
_support_actual_manifest_sha256 = _integrity_hashlib.sha256(
    _support_manifest_bytes
).hexdigest()
if _support_actual_manifest_sha256 != _support_expected_manifest_sha256:
    raise RuntimeError(
        "Support repo manifest checksum mismatch: "
        f"expected {_support_expected_manifest_sha256}, "
        f"got {_support_actual_manifest_sha256}"
    )

_primary_materialized_path = REPO_DIR / WEIGHTS_RELATIVE
_primary_actual_sha256 = _integrity_sha256_file(_primary_materialized_path)
if _primary_actual_sha256 != _primary_expected_sha256:
    raise RuntimeError(
        "Materialized primary model checksum mismatch: "
        f"expected {_primary_expected_sha256}, got {_primary_actual_sha256}"
    )

_deepcenter_candidate_strings = [
    os.environ.get("BIOHUB_DEEPCENTER_CHECKPOINT", "").strip(),
    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/"
    "full_frame_center/best.pt",
    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/"
    "weights/full_frame_center/best.pt",
]
_deepcenter_candidates = []
for _candidate_string in _deepcenter_candidate_strings:
    if not _candidate_string:
        continue
    _candidate_path = Path(_candidate_string)
    if _candidate_path not in _deepcenter_candidates:
        _deepcenter_candidates.append(_candidate_path)
_deepcenter_materialized_path = next(
    (path for path in _deepcenter_candidates if path.is_file()),
    None,
)
if _deepcenter_materialized_path is None:
    raise FileNotFoundError({
        "missing_deepcenter_checkpoint": [str(path) for path in _deepcenter_candidates]
    })
_deepcenter_actual_sha256 = _integrity_sha256_file(
    _deepcenter_materialized_path
)
if _deepcenter_actual_sha256 != _deepcenter_expected_sha256:
    raise RuntimeError(
        "DeepCenter checkpoint checksum mismatch: "
        f"expected {_deepcenter_expected_sha256}, "
        f"got {_deepcenter_actual_sha256}"
    )
os.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = str(
    _deepcenter_materialized_path
)

print("Support repo Python manifest SHA256:", _support_actual_manifest_sha256)
print("Primary materialized SHA256:", _primary_actual_sha256)
print("DeepCenter materialized SHA256:", _deepcenter_actual_sha256)


# Resolve the independent-seed pack by checksum, then materialize only its weights.
import hashlib as _hashlib

_secondary_manifest_explicit = Path(os.environ.get(
    "BIOHUB_SECONDARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-temporal-unet3d-seed314159-v1/ARTIFACT_MANIFEST.json",
))
_secondary_expected_sha256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
_secondary_slug = "biohub-temporal-unet3d-seed314159-v1"


def _find_secondary_artifact_root() -> tuple[Path, dict]:
    candidates = [
        _secondary_manifest_explicit,
        Path(f"/kaggle/input/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
        Path(f"/kaggle/input/datasets/pilkwang/{_secondary_slug}/ARTIFACT_MANIFEST.json"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(input_root.rglob("ARTIFACT_MANIFEST.json"))

    seen = set()
    for manifest_path in candidates:
        manifest_path = manifest_path.expanduser()
        if manifest_path in seen or not manifest_path.is_file():
            continue
        seen.add(manifest_path)
        try:
            info = json.loads(manifest_path.read_text())
        except Exception:
            continue
        sha256 = str(info.get("model", {}).get("weight_sha256", ""))
        if sha256 == _secondary_expected_sha256:
            return manifest_path.parent, info
    raise FileNotFoundError(
        "Could not find the independent-seed artifact with weight SHA256 "
        + _secondary_expected_sha256
    )


SECONDARY_ARTIFACTS, secondary_manifest_info = _find_secondary_artifact_root()
SECONDARY_WEIGHTS_ROOT = WORKING_DIR / "secondary_seed_weights"
copy_or_extract_tree(
    SECONDARY_ARTIFACTS / "weights",
    SECONDARY_ARTIFACTS / "weights.zip",
    SECONDARY_WEIGHTS_ROOT,
)
SECONDARY_WEIGHTS_PATH = (
    SECONDARY_WEIGHTS_ROOT
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)
SECONDARY_CONFIG_PATH = SECONDARY_WEIGHTS_PATH.parent / "config.json"
for _required_secondary_path in (SECONDARY_WEIGHTS_PATH, SECONDARY_CONFIG_PATH):
    if not _required_secondary_path.is_file():
        raise FileNotFoundError(f"Missing secondary model file: {_required_secondary_path}")


def _sha256_file(path: Path) -> str:
    digest = _hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


_secondary_actual_sha256 = _sha256_file(SECONDARY_WEIGHTS_PATH)
if _secondary_actual_sha256 != _secondary_expected_sha256:
    raise RuntimeError(
        "Secondary model checksum mismatch: "
        f"expected {_secondary_expected_sha256}, got {_secondary_actual_sha256}"
    )

os.environ["BIOHUB_SECONDARY_WEIGHTS"] = str(SECONDARY_WEIGHTS_PATH)
os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"] = "0.15"
print("Secondary artifact:", SECONDARY_ARTIFACTS)
print("Secondary weight:", SECONDARY_WEIGHTS_PATH)
print("Secondary SHA256:", _secondary_actual_sha256)
print("Secondary edge-logit weight:", os.environ["BIOHUB_SECONDARY_EDGE_WEIGHT"])

os.environ["BIOHUB_SECONDARY_DETECTION_WEIGHT"] = "0.475"
os.environ["BIOHUB_SECONDARY_LINK_MODE"] = "low_margin_consensus"
os.environ["BIOHUB_SECONDARY_MIX_TEMPERATURE"] = "1"
os.environ["BIOHUB_SECONDARY_LOW_MARGIN_MAX"] = "0.35"
os.environ["BIOHUB_DUAL_SEED_EDGE_THRESHOLD"] = "0.48"

_runtime_integrity_receipt = {
    "status": "complete_label_free_runtime_integrity",
    "verified_before_dynamic_source_patch": True,
    "support_repo_python_file_count": len(_support_actual_sha256),
    "support_repo_python_sha256": _support_actual_sha256,
    "support_repo_python_manifest_sha256": _support_actual_manifest_sha256,
    "checkpoint_sha256": {
        "primary": _primary_actual_sha256,
        "secondary": _secondary_actual_sha256,
        "deepcenter": _deepcenter_actual_sha256,
    },
    "materialized_paths": {
        "primary": str(_primary_materialized_path),
        "secondary": str(SECONDARY_WEIGHTS_PATH),
        "deepcenter": str(_deepcenter_materialized_path),
    },
    "ground_truth_accessed": False,
}
_runtime_integrity_receipt_path = (
    WORKING_DIR / "bidirectional_production_runtime_integrity.json"
)
_runtime_integrity_receipt_path.write_text(
    json.dumps(_runtime_integrity_receipt, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Runtime integrity receipt:", _runtime_integrity_receipt_path)

In [ ]:
# Fail fast instead of silently running volumetric inference on CPU.
import torch as _torch

if not _torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for this notebook. Enable a Kaggle GPU accelerator and commit again."
    )
print("CUDA device:", _torch.cuda.get_device_name(0))

# Apply eight-view planar detection TTA before graph prediction.
_ps = REPO_DIR / "scripts" / "predict_unet_transformer.py"
_s = _ps.read_text()
_old = """        if cfg.det_tta:
            tta_flips = [(-1,), (-2,), (-2, -1)]
            for dims in tta_flips:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
            for f in range(W):
                det_logits[f] = det_logits[f] / 4"""
_new = """        if cfg.det_tta:
            _nv = 1
            for dims in [(-1,), (-2,), (-2, -1)]:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
                _nv += 1
            for _k in (1, 3):
                imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))
                _, det_rot = model.encode(imgs_rot)
                for f in range(W):
                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -_k, dims=(-2, -1))
                del imgs_rot, det_rot
                _nv += 1
            imgs_t = imgs.transpose(-1, -2)
            _, det_t = model.encode(imgs_t)
            for f in range(W):
                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)
            del imgs_t, det_t
            _nv += 1
            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)
            _, det_at = model.encode(imgs_at)
            for f in range(W):
                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))
            del imgs_at, det_at
            _nv += 1
            for f in range(W):
                det_logits[f] = det_logits[f] / _nv"""
if _old in _s:
    _ps.write_text(_s.replace(_old, _new))
    print("TTA patch applied (400ep spatial D4-style)")
else:
    print("TTA WARNING: block not found - using default 4-way")

# Evaluate and calibrate two temporal models on one candidate graph.
_s = _ps.read_text()
_ensemble_replacements = [
    ('    downsample: tuple[int, ...] = (1, 4, 4),\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:', '    downsample: tuple[int, ...] = (1, 4, 4),\n    secondary_model: UNetNodeTransformer | None = None,\n    secondary_edge_weight: float = 0.0,\n    secondary_detection_weight: float = 0.0,\n    secondary_link_mode: str = "raw",\n    secondary_mix_temperature: float = 1.0,\n    secondary_low_margin_max: float = 0.2,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:'),
    ('            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        del imgs', '            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        secondary_unet_out = None\n        if secondary_model is not None:\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\n\n            if secondary_detection_weight > 0.0:\n                if cfg.det_tta:\n                    _secondary_nv = 1\n                    for dims in [(-1,), (-2,), (-2, -1)]:\n                        secondary_imgs_flip = imgs.flip(dims)\n                        _, secondary_det_flip = secondary_model.encode(secondary_imgs_flip)\n                        for f in range(W):\n                            secondary_det_logits[f] = (\n                                secondary_det_logits[f] + secondary_det_flip[f].flip(dims)\n                            )\n                        del secondary_imgs_flip, secondary_det_flip\n                        _secondary_nv += 1\n                    for _k in (1, 3):\n                        secondary_imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\n                        _, secondary_det_rot = secondary_model.encode(secondary_imgs_rot)\n                        for f in range(W):\n                            secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                                secondary_det_rot[f], -_k, dims=(-2, -1)\n                            )\n                        del secondary_imgs_rot, secondary_det_rot\n                        _secondary_nv += 1\n                    secondary_imgs_t = imgs.transpose(-1, -2)\n                    _, secondary_det_t = secondary_model.encode(secondary_imgs_t)\n                    for f in range(W):\n                        secondary_det_logits[f] = (\n                            secondary_det_logits[f] + secondary_det_t[f].transpose(-1, -2)\n                        )\n                    del secondary_imgs_t, secondary_det_t\n                    _secondary_nv += 1\n                    secondary_imgs_at = torch.rot90(\n                        imgs, 1, dims=(-2, -1)\n                    ).transpose(-1, -2)\n                    _, secondary_det_at = secondary_model.encode(secondary_imgs_at)\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                            secondary_det_at[f].transpose(-1, -2),\n                            -1,\n                            dims=(-2, -1),\n                        )\n                    del secondary_imgs_at, secondary_det_at\n                    _secondary_nv += 1\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\n\n                for f in range(W):\n                    primary_det = det_logits[f]\n                    secondary_det = secondary_det_logits[f]\n                    primary_mean = primary_det.mean()\n                    secondary_mean = secondary_det.mean()\n                    primary_scale = primary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    secondary_scale = secondary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_det_aligned = (\n                        (secondary_det - secondary_mean) * scale_ratio + primary_mean\n                    )\n                    det_logits[f] = (\n                        (1.0 - secondary_detection_weight) * primary_det\n                        + secondary_detection_weight * secondary_det_aligned\n                    )\n\n            del secondary_det_logits\n\n        del imgs'),
    ('            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            raw = edge_logits_pair[0]', '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            if secondary_model is not None:\n                if secondary_unet_out is None:\n                    raise RuntimeError("Secondary model is loaded but its feature map is missing")\n                secondary_feat_src = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx], p_coords_src, p_mask_src,\n                )\n                secondary_feat_tgt = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx + 1], p_coords_tgt, p_mask_tgt,\n                )\n                secondary_logits_pair = secondary_model.predict_edges(\n                    secondary_feat_src, secondary_feat_tgt,\n                    p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                    p_pos_src, p_pos_tgt,\n                    p_mask_src, p_mask_tgt,\n                )\n\n                if secondary_link_mode == "raw":\n                    secondary_for_mix = secondary_logits_pair\n                    blend_weight = secondary_edge_weight\n                elif secondary_link_mode in {\n                    "calibrated", "adaptive", "low_margin_consensus"\n                }:\n                    primary_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    primary_scale = edge_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_center = secondary_logits_pair.mean(dim=1, keepdim=True)\n                    secondary_scale = secondary_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_for_mix = (\n                        (secondary_logits_pair - secondary_center) * secondary_scale_ratio\n                        + primary_center\n                    )\n                    if secondary_link_mode == "calibrated":\n                        blend_weight = secondary_edge_weight\n                    elif secondary_link_mode == "adaptive":\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            secondary_margin = secondary_top2.values[0] - secondary_top2.values[1]\n                            local_weight = (\n                                secondary_edge_weight + secondary_margin - primary_margin\n                            ).clamp(0.15, 0.75)\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            local_weight = torch.where(\n                                same_parent,\n                                torch.maximum(\n                                    local_weight,\n                                    torch.full_like(local_weight, secondary_edge_weight),\n                                ),\n                                local_weight,\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = secondary_edge_weight\n                    else:\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            uncertainty = (\n                                (secondary_low_margin_max - primary_margin)\n                                / secondary_low_margin_max\n                            ).clamp(0.0, 1.0)\n                            local_weight = secondary_edge_weight * uncertainty\n                            local_weight = torch.where(\n                                same_parent,\n                                local_weight,\n                                torch.zeros_like(local_weight),\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = 0.0\n                else:\n                    raise ValueError(f"Unsupported secondary link mode: {secondary_link_mode}")\n\n                edge_logits_pair = (\n                    (1.0 - blend_weight) * edge_logits_pair\n                    + blend_weight * secondary_for_mix\n                )\n                if secondary_mix_temperature != 1.0:\n                    mixed_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    edge_logits_pair = mixed_center + (\n                        edge_logits_pair - mixed_center\n                    ) / secondary_mix_temperature\n\n            raw = edge_logits_pair[0]'),
    ('        del unet_out\n', '        del unet_out\n        if secondary_unet_out is not None:\n            del secondary_unet_out\n'),
    ('    model, window_size, downsample = load_model(weights_path, device)\n    print(', '    model, window_size, downsample = load_model(weights_path, device)\n\n    secondary_model = None\n    secondary_weights_text = os.environ.get("BIOHUB_SECONDARY_WEIGHTS", "").strip()\n    secondary_edge_weight = float(os.environ.get("BIOHUB_SECONDARY_EDGE_WEIGHT", "0"))\n    secondary_detection_weight = float(\n        os.environ.get("BIOHUB_SECONDARY_DETECTION_WEIGHT", "0")\n    )\n    secondary_link_mode = os.environ.get("BIOHUB_SECONDARY_LINK_MODE", "raw").strip()\n    secondary_mix_temperature = float(\n        os.environ.get("BIOHUB_SECONDARY_MIX_TEMPERATURE", "1")\n    )\n    secondary_low_margin_max = float(\n        os.environ.get("BIOHUB_SECONDARY_LOW_MARGIN_MAX", "0.2")\n    )\n    edge_candidate_threshold = float(\n        os.environ.get("BIOHUB_DUAL_SEED_EDGE_THRESHOLD", str(cfg.threshold))\n    )\n    if secondary_weights_text:\n        if not 0.0 < secondary_edge_weight < 1.0:\n            raise ValueError("BIOHUB_SECONDARY_EDGE_WEIGHT must be strictly between 0 and 1")\n        if not 0.0 <= secondary_detection_weight < 1.0:\n            raise ValueError(\n                "BIOHUB_SECONDARY_DETECTION_WEIGHT must be in the half-open interval [0, 1)"\n            )\n        if secondary_link_mode not in {\n            "raw", "calibrated", "adaptive", "low_margin_consensus"\n        }:\n            raise ValueError(\n                "BIOHUB_SECONDARY_LINK_MODE must be raw, calibrated, adaptive, "\n                "or low_margin_consensus"\n            )\n        if not 0.5 <= secondary_mix_temperature <= 2.0:\n            raise ValueError("BIOHUB_SECONDARY_MIX_TEMPERATURE must be in [0.5, 2.0]")\n        if not 0.0 < edge_candidate_threshold < 1.0:\n            raise ValueError("BIOHUB_DUAL_SEED_EDGE_THRESHOLD must be strictly between 0 and 1")\n        if not 0.0 < secondary_low_margin_max <= 1.0:\n            raise ValueError("BIOHUB_SECONDARY_LOW_MARGIN_MAX must be in (0, 1]")\n        secondary_model, secondary_window_size, secondary_downsample = load_model(\n            Path(secondary_weights_text), device,\n        )\n        if secondary_window_size != window_size or secondary_downsample != downsample:\n            raise ValueError(\n                "Primary and secondary models have incompatible inference grids: "\n                f"primary=(window={window_size}, downsample={downsample}), "\n                f"secondary=(window={secondary_window_size}, downsample={secondary_downsample})"\n            )\n        cfg.threshold = edge_candidate_threshold\n        print(\n            f"Secondary model: {secondary_weights_text} | "\n            f"edge weight={secondary_edge_weight:.3f} | "\n            f"detection weight={secondary_detection_weight:.3f} | "\n            f"link mode={secondary_link_mode} | "\n            f"temperature={secondary_mix_temperature:.3f} | "\n            f"low-margin max={secondary_low_margin_max:.3f} | "\n            f"edge threshold={cfg.threshold:.3f}",\n            flush=True,\n        )\n\n    print('),
    ('                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n            )', '                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n                secondary_model=secondary_model,\n                secondary_edge_weight=secondary_edge_weight,\n                secondary_detection_weight=secondary_detection_weight,\n                secondary_link_mode=secondary_link_mode,\n                secondary_mix_temperature=secondary_mix_temperature,\n                secondary_low_margin_max=secondary_low_margin_max,\n            )'),
]
for _patch_index, (_ensemble_old, _ensemble_new) in enumerate(
    _ensemble_replacements, start=1
):
    _ensemble_count = _s.count(_ensemble_old)
    if _ensemble_count != 1:
        raise RuntimeError(
            f'Calibrated dual-seed patch {_patch_index} expected one match, '
            f'found {_ensemble_count}'
        )
    _s = _s.replace(_ensemble_old, _ensemble_new, 1)
compile(_s, str(_ps), 'exec')
_ps.write_text(_s)
print('Calibrated dual-seed runtime patch applied')


# Frozen label-free frame retention guard.
os.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"] = "0.90"
for _guard_old_log in WORKING_DIR.glob("retention_guard_*.jsonl"):
    _guard_old_log.unlink()

_s = _ps.read_text()
_guard_old = """                    det_logits[f] = (
                        (1.0 - secondary_detection_weight) * primary_det
                        + secondary_detection_weight * secondary_det_aligned
                    )"""
_guard_new = """                    blended_det = (
                        (1.0 - secondary_detection_weight) * primary_det
                        + secondary_detection_weight * secondary_det_aligned
                    )
                    primary_candidates = len(_detect_cells_pooled(
                        primary_det[0],
                        int(frame_indices[f]),
                        cfg.det_threshold,
                        pool_k,
                    ))
                    blended_candidates = len(_detect_cells_pooled(
                        blended_det[0],
                        int(frame_indices[f]),
                        cfg.det_threshold,
                        pool_k,
                    ))
                    minimum_retention = float(os.environ.get(
                        "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION",
                        "0.90",
                    ))
                    candidate_retention = (
                        blended_candidates / primary_candidates
                        if primary_candidates
                        else 1.0
                    )
                    use_primary_detection = bool(
                        primary_candidates > 0
                        and candidate_retention < minimum_retention
                    )
                    det_logits[f] = (
                        primary_det if use_primary_detection else blended_det
                    )
                    if int(frame_indices[f]) not in seen_frames:
                        shard = os.environ.get(
                            "BIOHUB_GPU_SHARD", "single"
                        ).replace("/", "_")
                        guard_log = (
                            Path("/kaggle/working")
                            / f"retention_guard_{shard}.jsonl"
                        )
                        guard_record = {
                            "dataset": ds_path.stem,
                            "frame": int(frame_indices[f]),
                            "primary_candidates": int(primary_candidates),
                            "blended_candidates": int(blended_candidates),
                            "retention": float(candidate_retention),
                            "minimum_retention": float(minimum_retention),
                            "use_primary": bool(use_primary_detection),
                        }
                        with guard_log.open("a") as guard_handle:
                            guard_handle.write(
                                json.dumps(guard_record, sort_keys=True)
                                + "\\n"
                            )
                        if use_primary_detection:
                            print(
                                "BIOHUB_RETENTION_GUARD "
                                + json.dumps(guard_record, sort_keys=True),
                                flush=True,
                            )"""
_guard_matches = _s.count(_guard_old)
if _guard_matches != 1:
    raise RuntimeError(
        f"Retention guard expected one blend block, found {_guard_matches}"
    )
_s = _s.replace(_guard_old, _guard_new, 1)
compile(_s, str(_ps), "exec")
_ps.write_text(_s)
print(
    "Frozen frame retention guard applied at "
    + os.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"]
)

# Fixed public reverse-time primary-association experiment. Detection, the
# secondary low-margin mix, ILP, and graph post-processing are unchanged.
import math as _bidirectional_math

_bidirectional_weight_guard = float(
    os.environ.get("BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT", "0")
)
if not _bidirectional_math.isclose(
    _bidirectional_weight_guard, 0.30, rel_tol=0.0, abs_tol=1e-12
):
    raise ValueError({
        "expected_bidirectional_weight": 0.30,
        "actual_bidirectional_weight": _bidirectional_weight_guard,
    })

_s = _ps.read_text()
_bi_old = '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            if secondary_model is not None:\n'
_bi_new = '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            _bidirectional_weight = float(\n                os.environ.get("BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT", "0")\n            )\n            if _bidirectional_weight > 0.0:\n                reverse_logits_native = model.predict_edges(\n                    unet_feat_tgt, unet_feat_src,\n                    p_coords_tgt * ds_arr_t, p_coords_src * ds_arr_t,\n                    p_pos_tgt, p_pos_src,\n                    p_mask_tgt, p_mask_src,\n                )  # (1, n_tgt, n_src)\n                reverse_logits_pair = reverse_logits_native.transpose(1, 2)\n\n                forward_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                forward_scale = edge_logits_pair.float().std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                reverse_center = reverse_logits_pair.mean(dim=1, keepdim=True)\n                reverse_scale = reverse_logits_pair.float().std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                reverse_scale_ratio = (forward_scale / reverse_scale).clamp(0.5, 2.0)\n                reverse_scale_ratio = reverse_scale_ratio.to(reverse_logits_pair.dtype)\n                reverse_aligned = (\n                    (reverse_logits_pair - reverse_center) * reverse_scale_ratio\n                    + forward_center\n                )\n                # Biohub 145: require mutual forward/reverse support in probability space.\n                # The harmonic mean penalizes a candidate when either temporal direction\n                # assigns it very low probability, while calibration preserves the forward\n                # logit scale used by the unchanged downstream candidate threshold and ILP.\n                forward_prob = torch.softmax(edge_logits_pair.float(), dim=1).clamp_min(1e-8)\n                reverse_prob = torch.softmax(reverse_aligned.float(), dim=1).clamp_min(1e-8)\n                harmonic_prob = 1.0 / (\n                    (1.0 - _bidirectional_weight) / forward_prob\n                    + _bidirectional_weight / reverse_prob\n                )\n                harmonic_prob = harmonic_prob / harmonic_prob.sum(\n                    dim=1, keepdim=True\n                ).clamp_min(1e-8)\n                harmonic_logits = torch.log(harmonic_prob.clamp_min(1e-8))\n                harmonic_center = harmonic_logits.mean(dim=1, keepdim=True)\n                harmonic_scale = harmonic_logits.std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                harmonic_scale_ratio = (forward_scale / harmonic_scale).clamp(0.5, 2.0)\n                edge_logits_pair = (\n                    (harmonic_logits - harmonic_center) * harmonic_scale_ratio\n                    + forward_center\n                ).to(reverse_aligned.dtype)\n                del (\n                    reverse_logits_native,\n                    reverse_logits_pair,\n                    reverse_aligned,\n                    forward_prob,\n                    reverse_prob,\n                    harmonic_prob,\n                    harmonic_logits,\n                )\n            if secondary_model is not None:\n'
_bi_count = _s.count(_bi_old)
if _bi_count != 1:
    raise RuntimeError(
        f"Bidirectional edge patch expected one transformed block, found {_bi_count}"
    )
_s = _s.replace(_bi_old, _bi_new, 1)

_coordinate_manifest_old = '    coords = coords.astype(np.int16)\n    return coords, all_edges'
_coordinate_manifest_new = '    coords = coords.astype(np.int16)\n\n    # Label-free, pre-ILP detector-coordinate manifest. This executes inside\n    # predict_video, before build_graph and the ILP call in predict().\n    _coordinate_manifest_arm = os.environ.get(\n        "BIOHUB_DIAGNOSTIC_ARM", ""\n    ).strip()\n    if _coordinate_manifest_arm:\n        import hashlib as _coordinate_hashlib\n\n        _coordinate_shard = os.environ.get(\n            "BIOHUB_GPU_SHARD", "single"\n        ).replace("/", "_")\n        _coordinate_array = np.ascontiguousarray(\n            coords.astype("<i2", copy=False)\n        )\n        _coordinate_frame_counts = [\n            [int(_coordinate_t), int((_coordinate_array[:, 0] == _coordinate_t).sum())]\n            for _coordinate_t in np.unique(_coordinate_array[:, 0])\n        ]\n        _coordinate_record = {\n            "columns": ["t", "z", "y", "x"],\n            "coordinate_sha256": _coordinate_hashlib.sha256(\n                _coordinate_array.tobytes(order="C")\n            ).hexdigest(),\n            "dataset": ds_path.stem,\n            "dtype": "<i2",\n            "frame_counts": _coordinate_frame_counts,\n            "rows": int(len(_coordinate_array)),\n            "stage": "post_detection_pre_graph_pre_ilp",\n        }\n        _coordinate_manifest_path = (\n            Path("/kaggle/working")\n            / f"detector_coordinates_{_coordinate_manifest_arm}_"\n            f"{_coordinate_shard}.jsonl"\n        )\n        with _coordinate_manifest_path.open("a") as _coordinate_handle:\n            _coordinate_handle.write(\n                json.dumps(_coordinate_record, sort_keys=True) + "\\n"\n            )\n\n    return coords, all_edges'
_coordinate_manifest_count = _s.count(_coordinate_manifest_old)
if _coordinate_manifest_count != 1:
    raise RuntimeError(
        "Coordinate-manifest patch expected one pre-return block, found "
        f"{_coordinate_manifest_count}"
    )
_s = _s.replace(
    _coordinate_manifest_old, _coordinate_manifest_new, 1
)
compile(_s, str(_ps), "exec")
_ps.write_text(_s)
print(
    "Bidirectional harmonic-probability association fusion applied | weight=",
    _bidirectional_weight_guard,
)
print("Pre-ILP detector-coordinate manifest hook applied")



In [ ]:
D3_PAYLOAD = WORKING_DIR / "collection_payload"
assert not D3_PAYLOAD.exists(), "fresh payload required"
D3_FILES = {'src/biohub/association_capture.py': '"""Lossless observation packets for the fixed E23 association path, not a predictor.\n\nThe caller supplies detached CPU NumPy arrays. No torch/model imports, network,\nlabels, score modification, implicit precision conversion, or top-k selection.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\nMATRICES = (\n    "primary_forward_logits", "primary_reverse_logits", "secondary_forward_logits",\n    "mixed_logits", "mixed_probabilities",\n)\nFEATURES = ("primary_source_features", "primary_target_features",\n            "secondary_source_features", "secondary_target_features")\nCOORDINATES = ("source_indices", "target_indices", "source_coords_grid", "target_coords_grid")\n\n\n@dataclass(frozen=True)\nclass CaptureLimits:\n    nodes_per_side: int = 2048\n    pairs_per_packet: int = 1_000_000\n    pairs_per_run: int = 450_000_000\n    output_bytes: int = 12 * 1024**3\n    writer_seconds: float = 1200.0\n\n\ndef _require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef _sha(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024**2), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _json_bytes(value):\n    return (json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False) + "\\n").encode()\n\n\nclass PairCapture:\n    """Fresh-run writer with literal per-video detector counts and full-pair coverage.\n\n`primary_reverse_logits` must already be transposed into (source, target)\norientation by the observer. Original-space coordinates are grid * downsample;\nphysical coordinates additionally multiply scale. Neither conversion alters input.\n"""\n\n    def __init__(self, root: Path, frame_counts: dict[str, list[int]], *,\n                 downsample=(1, 4, 4), scale=(1.625, 0.40625, 0.40625), limits=None):\n        if limits is None:\n            limits = CaptureLimits()\n        _require(isinstance(limits, CaptureLimits), "invalid limits")\n        _require(all(type(getattr(limits, k)) is int and getattr(limits, k) > 0\n                     for k in ("nodes_per_side", "pairs_per_packet", "pairs_per_run", "output_bytes")),\n                 "limits must be positive integers")\n        _require(type(limits.writer_seconds) in (int, float) and np.isfinite(limits.writer_seconds)\n                 and limits.writer_seconds > 0, "invalid writer time limit")\n        _require(len(downsample) == 3 and all(type(v) is int and v > 0 for v in downsample),\n                 "invalid downsample")\n        _require(len(scale) == 3 and all(type(v) in (int, float) and np.isfinite(v) and v > 0 for v in scale),\n                 "invalid scale")\n        _require(bool(frame_counts), "empty dataset plan")\n        self.frame_counts = {}\n        self.offsets = {}\n        self.expected = set()\n        expected_pairs = 0\n        for name, counts in frame_counts.items():\n            _require(isinstance(name, str) and re.fullmatch(r"[A-Za-z0-9_-]+", name), "unsafe dataset name")\n            _require(len(counts) >= 2 and all(type(n) is int and 0 <= n <= limits.nodes_per_side for n in counts),\n                     "invalid detector count")\n            _require(len(counts) < 32768, "frame index exceeds int16")\n            self.frame_counts[name] = tuple(counts)\n            self.offsets[name] = np.cumsum([0, *counts], dtype=np.int64)\n            for t in range(len(counts) - 1):\n                size = counts[t] * counts[t + 1]\n                _require(size <= limits.pairs_per_packet, "packet pair budget exceeded")\n                expected_pairs += size\n                self.expected.add((name, t))\n        _require(expected_pairs <= limits.pairs_per_run, "run pair budget exceeded")\n        self.expected_pairs = expected_pairs\n        self.downsample, self.scale, self.limits = tuple(downsample), tuple(scale), limits\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=False)\n        self.records = {}\n        self.output_bytes = 0\n        self.writer_seconds = 0.0\n        self.failed = False\n        self.closed = False\n\n    def _available(self):\n        _require(not self.failed and not self.closed, "capture already failed or closed")\n\n    def _failure(self, exc):\n        self.failed = True\n        path = self.root / "ERROR.json"\n        if not path.exists():\n            with path.open("xb") as stream:\n                stream.write(_json_bytes({"status": "ERROR", "type": type(exc).__name__, "error": str(exc),\n                                          "submission_authorized": False}))\n\n    def _validate(self, dataset, t_source, arrays):\n        _require(type(t_source) is int and (dataset, t_source) in self.expected, "unexpected frame pair")\n        _require((dataset, t_source) not in self.records, "duplicate frame pair")\n        ns, nt = self.frame_counts[dataset][t_source:t_source + 2]\n        empty = ns == 0 or nt == 0\n        keys = set(COORDINATES) if empty else set(COORDINATES + MATRICES + FEATURES)\n        _require(set(arrays) == keys, "packet key set mismatch")\n        for key, a in arrays.items():\n            _require(isinstance(a, np.ndarray) and not a.dtype.hasobject, f"not a numeric ndarray: {key}")\n            if key in MATRICES:\n                shape, dtype = (ns, nt), np.dtype("float32")\n            else:\n                n = ns if "source" in key else nt\n                shape = (n,) if key.endswith("indices") else (n, 4) if key.endswith("grid") else (n, 32)\n                dtype = np.dtype("int64") if key.endswith("indices") else (\n                    np.dtype("int16") if key.endswith("grid") else np.dtype("float32"))\n            _require(a.shape == shape and a.dtype == dtype, f"shape/dtype mismatch: {key}")\n            _require(np.isfinite(a).all(), f"nonfinite: {key}")\n        for side, t in (("source", t_source), ("target", t_source + 1)):\n            start, end = self.offsets[dataset][t:t + 2]\n            _require(np.array_equal(arrays[f"{side}_indices"], np.arange(start, end, dtype=np.int64)),\n                     f"global detector ID mapping mismatch: {side}")\n            coords = arrays[f"{side}_coords_grid"]\n            _require(np.all(coords[:, 0] == t) and np.all(coords[:, 1:] >= 0), f"invalid t/grid coords: {side}")\n        if not empty:\n            probs = arrays["mixed_probabilities"]\n            _require(np.all((probs >= 0) & (probs <= 1)), "probability out of range")\n            _require(np.allclose(probs.sum(axis=0, dtype=np.float64), 1., atol=2e-5, rtol=0),\n                     "probabilities not normalised over source axis")\n        return ns, nt, empty\n\n    def write_pair(self, dataset: str, t_source: int, arrays: dict[str, np.ndarray]):\n        self._available()\n        started = time.perf_counter()\n        try:\n            ns, nt, empty = self._validate(dataset, t_source, arrays)\n            metadata = {"schema_version": "biohub.e23.association_pair.v1", "dataset": dataset,\n                        "t_source": t_source, "t_target": t_source + 1, "window_size": 2,\n                        "downsample_zyx": self.downsample, "scale_zyx_um": self.scale,\n                        "matrix_axes": ["source", "target"], "probability_normalisation_axis": "source",\n                        "status": "SKIPPED_EMPTY" if empty else "OBSERVED",\n                        "n_source": ns, "n_target": nt, "dense_pairs": ns * nt,\n                        "arrays": {k: {"shape": list(a.shape), "dtype": a.dtype.str,\n                                       "sha256": hashlib.sha256(a.tobytes(order="C")).hexdigest()}\n                                   for k, a in arrays.items()}, "submission_authorized": False}\n            folder = self.root / dataset\n            folder.mkdir(exist_ok=True)\n            path = folder / f"pair_{t_source:04d}.npz"\n            with path.open("xb") as stream:\n                np.savez_compressed(stream, metadata=np.array(_json_bytes(metadata).decode()), **arrays)\n            with np.load(path, allow_pickle=False) as saved:\n                _require(set(saved.files) == set(arrays) | {"metadata"}, "roundtrip key mismatch")\n                _require(json.loads(str(saved["metadata"])) == json.loads(json.dumps(metadata)),\n                         "roundtrip metadata mismatch")\n                for key, original in arrays.items():\n                    actual = saved[key]\n                    _require(actual.dtype == original.dtype and actual.shape == original.shape and\n                             actual.tobytes(order="C") == original.tobytes(order="C"),\n                             f"lossy roundtrip: {key}")\n            digest = _sha(path)\n            self.output_bytes += path.stat().st_size\n            self.writer_seconds += time.perf_counter() - started\n            _require(self.output_bytes <= self.limits.output_bytes, "output byte budget exceeded")\n            _require(self.writer_seconds <= self.limits.writer_seconds, "writer time budget exceeded")\n            self.records[dataset, t_source] = {"path": path.relative_to(self.root).as_posix(),\n                                             "bytes": path.stat().st_size, "sha256": digest, **metadata}\n            return json.loads(json.dumps(self.records[dataset, t_source]))\n        except Exception as exc:\n            self._failure(exc)\n            raise\n\n    def finish(self):\n        self._available()\n        started = time.perf_counter()\n        try:\n            _require(set(self.records) == self.expected, "incomplete frame-pair capture")\n            _require(sum(r["dense_pairs"] for r in self.records.values()) == self.expected_pairs,\n                     "dense pair count mismatch")\n            for record in self.records.values():\n                path = self.root / record["path"]\n                _require(path.stat().st_size == record["bytes"] and _sha(path) == record["sha256"],\n                         "saved packet changed")\n            self.writer_seconds += time.perf_counter() - started\n            _require(self.writer_seconds <= self.limits.writer_seconds, "writer time budget exceeded")\n            result = {"schema_version": "biohub.e23.association_capture.v1", "status": "PAIR_CAPTURE_COMPLETE",\n                      "capture_only": True, "submission_authorized": False,\n                      "graph_ID_mapping_complete": False, "inference_parity_verified": False,\n                      "frame_counts": self.frame_counts, "dense_pairs": self.expected_pairs,\n                      "writer_seconds": self.writer_seconds, "packet_bytes": self.output_bytes,\n                      "records": [self.records[key] for key in sorted(self.records)]}\n            encoded = _json_bytes(result)\n            _require(self.output_bytes + len(encoded) <= self.limits.output_bytes, "manifest byte budget exceeded")\n            with (self.root / "MANIFEST.json").open("xb") as stream:\n                stream.write(encoded)\n            self.closed = True\n            return result\n        except Exception as exc:\n            self._failure(exc)\n            raise\n', 'src/biohub/association_observer.py': '"""Sequential E23 observation adapter; no predictor, GT, model load or network."""\n\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom functools import wraps\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom biohub.association_capture import PairCapture, _json_bytes, _require, _sha\n\n\ndef observed(method):\n    """Account synchronous capture work at call boundaries; never swallow errors."""\n    @wraps(method)\n    def call(self, *args, **kwargs):\n        _require(not self.failed and not self.closed, "observer failed or closed")\n        started = time.perf_counter()\n        try:\n            result = method(self, *args, **kwargs)\n            self.seconds += time.perf_counter() - started\n            _require(self.seconds <= self.pairs.limits.writer_seconds, "observer time budget exceeded")\n            _require(self.pairs.output_bytes + self.graph_bytes <= self.pairs.limits.output_bytes,\n                     "observer byte budget exceeded")\n            return result\n        except Exception as exc:\n            self.failed = True\n            path = self.root / "ERROR.json"\n            if not path.exists():\n                with path.open("xb") as stream:\n                    stream.write(_json_bytes({"status": "ERROR", "error": str(exc),\n                                              "submission_authorized": False}))\n            raise\n    return call\n\n\ndef _copy_tensor(value, *, batched=True):\n    # No .float(), autocast, RNG, or model invocation. PairCapture checks actual dtype.\n    array = value.detach().cpu().numpy()\n    if batched:\n        _require(array.ndim == 3 and array.shape[0] == 1, "expected batch size one")\n        array = array[0]\n    return array.copy()\n\n\nclass AssociationObserver:\n    def __init__(self, root: Path, frame_counts, *, limits=None, downsample=(1, 4, 4),\n                 scale=(1.625, .40625, .40625)):\n        self.root = Path(root)\n        # PairCapture validates the plan before creating any directory.\n        _require(not self.root.exists(), "observer output already exists")\n        self.pairs = PairCapture(self.root / "pairs", frame_counts, limits=limits,\n                                 downsample=downsample, scale=scale)\n        self.failed = self.closed = False\n        self.seconds = self.graph_bytes = 0\n        self.dataset = None\n        self.pending = None\n        self.stage = "idle"\n        self.completed = set()\n        self.artifacts = []\n        self.coords = self.edges = self.before = None\n\n    @observed\n    def start_video(self, name, frames, window_size, downsample, scale, activation, *,\n                    threshold=.48, max_parents=None, max_children=None):\n        _require(self.dataset is None and name not in self.completed, "video overlap or duplicate")\n        _require(name in self.pairs.frame_counts and frames == len(self.pairs.frame_counts[name]),\n                 "video plan mismatch")\n        _require(window_size == 2 and tuple(downsample) == self.pairs.downsample\n                 and tuple(scale) == self.pairs.scale and activation == "softmax", "inference geometry mismatch")\n        _require(threshold == .48 and max_parents is None and max_children is None,\n                 "E23 candidate filtering mismatch")\n        self.threshold = threshold\n        self.dataset, self.stage = name, "pair"\n\n    @observed\n    def start_pair(self, t, s_src, e_src, s_tgt, e_tgt, coords):\n        _require(self.stage == "pair" and self.pending is None, "pair stage order mismatch")\n        _require((self.dataset, t) in self.pairs.expected, "unexpected pair")\n        offsets = self.pairs.offsets[self.dataset]\n        _require((s_src, e_src, s_tgt, e_tgt) == (offsets[t], offsets[t + 1], offsets[t + 1], offsets[t + 2]),\n                 "detector offsets mismatch")\n        self.t = t\n        self.pending = {\n            "source_indices": np.arange(s_src, e_src, dtype=np.int64),\n            "target_indices": np.arange(s_tgt, e_tgt, dtype=np.int64),\n            "source_coords_grid": coords[s_src:e_src].copy(),\n            "target_coords_grid": coords[s_tgt:e_tgt].copy(),\n        }\n        if s_src == e_src or s_tgt == e_tgt:\n            self.pairs.write_pair(self.dataset, t, self.pending)\n            self.pending = None\n        else:\n            self.stage = "primary"\n\n    @observed\n    def primary(self, logits, source, target):\n        _require(self.stage == "primary", "primary stage order mismatch")\n        self.pending.update(primary_forward_logits=_copy_tensor(logits),\n                            primary_source_features=_copy_tensor(source),\n                            primary_target_features=_copy_tensor(target))\n        self.stage = "reverse"\n\n    @observed\n    def reverse(self, transposed_logits):\n        _require(self.stage == "reverse", "reverse stage order mismatch")\n        self.pending["primary_reverse_logits"] = _copy_tensor(transposed_logits)\n        self.stage = "secondary"\n\n    @observed\n    def secondary(self, logits, source, target):\n        _require(self.stage == "secondary", "secondary stage order mismatch")\n        self.pending.update(secondary_forward_logits=_copy_tensor(logits),\n                            secondary_source_features=_copy_tensor(source),\n                            secondary_target_features=_copy_tensor(target))\n        self.stage = "mixed"\n\n    @observed\n    def mixed(self, raw, probabilities):\n        _require(self.stage == "mixed", "mixed stage order mismatch")\n        self.pending.update(mixed_logits=_copy_tensor(raw, batched=False),\n                            mixed_probabilities=probabilities.copy())\n        self.pairs.write_pair(self.dataset, self.t, self.pending)\n        self.pending, self.stage = None, "pair"\n\n    @observed\n    def end_video(self, coords, edges):\n        _require(self.stage == "pair" and self.pending is None, "incomplete pair")\n        name = self.dataset\n        _require({(name, t) for t in range(len(self.pairs.frame_counts[name]) - 1)}\n                 <= set(self.pairs.records), "incomplete video")\n        _require(coords.dtype == np.int16 and coords.shape == (sum(self.pairs.frame_counts[name]), 4),\n                 "returned coordinates shape/dtype mismatch")\n        by_time = {t: {} for t in range(len(self.pairs.frame_counts[name]) - 1)}\n        for s, target, probability, distance in edges:\n            _require(type(s) is int and type(target) is int and 0 <= s < len(coords)\n                     and 0 <= target < len(coords), "invalid returned edge index")\n            t = int(coords[s, 0])\n            _require(t in by_time and coords[target, 0] == t + 1, "invalid returned edge time")\n            _require((s, target) not in by_time[t], "duplicate returned edge")\n            by_time[t][s, target] = (probability, distance)\n        for t in range(len(self.pairs.frame_counts[name]) - 1):\n            record = self.pairs.records[name, t]\n            with np.load(self.pairs.root / record["path"], allow_pickle=False) as packet:\n                for side in ("source", "target"):\n                    grid = packet[f"{side}_coords_grid"].astype(np.int64)\n                    grid[:, 1:] *= self.pairs.downsample\n                    _require(np.array_equal(coords[packet[f"{side}_indices"]], grid),\n                             "returned coordinates differ from observed detections")\n                expected = {}\n                if record["status"] != "SKIPPED_EMPTY":\n                    probabilities = packet["mixed_probabilities"]\n                    source_ids, target_ids = packet["source_indices"], packet["target_indices"]\n                    source_grid, target_grid = packet["source_coords_grid"], packet["target_coords_grid"]\n                    for i, j in zip(*np.where(probabilities > self.threshold), strict=True):\n                        delta = source_grid[i, 1:].astype(np.float32) - target_grid[j, 1:].astype(np.float32)\n                        expected[int(source_ids[i]), int(target_ids[j])] = (\n                            float(probabilities[i, j]), float(np.linalg.norm(delta)))\n                _require(expected == by_time[t], "returned candidates differ from observed probability/geometry")\n        self.coords = coords.copy()\n        self.edges = tuple(tuple(edge) for edge in edges)\n        self.stage = "pre_graph"\n\n    def _save_arrays(self, label, arrays):\n        _require(all(isinstance(a, np.ndarray) and not a.dtype.hasobject for a in arrays.values()),\n                 "graph array type mismatch")\n        path = self.root / f"{self.dataset}_{label}.npz"\n        with path.open("xb") as stream:\n            np.savez_compressed(stream, **arrays)\n        with np.load(path, allow_pickle=False) as saved:\n            _require(set(saved.files) == set(arrays), "graph roundtrip keys mismatch")\n            for key, original in arrays.items():\n                actual = saved[key]\n                _require(actual.dtype == original.dtype and actual.shape == original.shape\n                         and actual.tobytes() == original.tobytes(), "graph roundtrip mismatch")\n        self.graph_bytes += path.stat().st_size\n        self.artifacts.append({"path": path.name, "bytes": path.stat().st_size, "sha256": _sha(path),\n                               "dataset": self.dataset, "stage": label})\n\n    @staticmethod\n    def _graph_tables(graph):\n        nodes = graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])\n        edge_keys = ["edge_id", "source_id", "target_id"]\n        edges = graph.edge_attrs(attr_keys=edge_keys + (["edge_prob", "edge_dist"] if graph.num_edges() else []))\n        return nodes, edges\n\n    @observed\n    def pre_graph(self, coords, edges, node_ids, graph):\n        _require(self.stage == "pre_graph", "pre-graph stage order mismatch")\n        _require(np.array_equal(coords, self.coords) and tuple(tuple(e) for e in edges) == self.edges,\n                 "build_graph inputs changed")\n        _require(len(node_ids) == len(coords) and len(set(node_ids)) == len(coords)\n                 and all(isinstance(i, (int, np.integer)) and not isinstance(i, bool) for i in node_ids),\n                 "nonbijective graph ID mapping")\n        nodes, edge_table = self._graph_tables(graph)\n        node_map = {r["node_id"]: r for r in nodes.iter_rows(named=True)}\n        _require(len(node_map) == len(node_ids), "pre-graph node count mismatch")\n        for row, node_id in enumerate(node_ids):\n            _require(node_id in node_map and\n                     tuple(node_map[node_id][k] for k in ("t", "z", "y", "x")) == tuple(coords[row]),\n                     "graph ID/coordinate mismatch")\n        expected = {}\n        for s, t, prob, dist in edges:\n            _require(type(s) is int and type(t) is int and 0 <= s < len(coords) and 0 <= t < len(coords),\n                     "invalid candidate index")\n            _require(coords[t, 0] == coords[s, 0] + 1 and np.isfinite(prob) and np.isfinite(dist)\n                     and 0 <= prob <= 1 and dist >= 0, "invalid candidate edge")\n            key = (node_ids[s], node_ids[t])\n            _require(key not in expected, "duplicate candidate edge")\n            expected[key] = (prob, dist)\n        actual = {(r["source_id"], r["target_id"]): (r["edge_prob"], r["edge_dist"])\n                  for r in edge_table.iter_rows(named=True)}\n        _require(edge_table.height == len(actual) and actual == expected, "pre-graph candidate mismatch")\n        arrays = {"detector_indices": np.arange(len(coords), dtype=np.int64),\n                  "graph_node_ids": np.asarray(node_ids, dtype=np.int64)}\n        for prefix, table in (("node", nodes), ("edge", edge_table)):\n            arrays.update({f"{prefix}_{column}": table[column].to_numpy() for column in table.columns})\n        self._save_arrays("pre_ilp", arrays)\n        self.before = (node_map, actual)\n        self.stage = "selected_graph"\n\n    @observed\n    def selected_graph(self, graph):\n        _require(self.stage == "selected_graph", "selected-graph stage order mismatch")\n        nodes, edges = self._graph_tables(graph)\n        before_nodes, before_edges = self.before\n        node_ids = nodes["node_id"].to_list()\n        _require(len(node_ids) == len(set(node_ids)), "duplicate selected node")\n        for row in nodes.iter_rows(named=True):\n            _require(before_nodes.get(row["node_id"]) == row, "selected node changed")\n        selected_edges = set()\n        selected_ids = set(node_ids)\n        for row in edges.iter_rows(named=True):\n            key = (row["source_id"], row["target_id"])\n            _require(key not in selected_edges and all(i in selected_ids for i in key)\n                     and before_edges.get(key) == (row["edge_prob"], row["edge_dist"]),\n                     "selected edge changed or dangling")\n            selected_edges.add(key)\n        self._save_arrays("post_ilp", {f"{prefix}_{column}": table[column].to_numpy()\n                                       for prefix, table in (("node", nodes), ("edge", edges))\n                                       for column in table.columns})\n        self.completed.add(self.dataset)\n        self.dataset, self.stage = None, "idle"\n        self.coords = self.edges = self.before = None\n\n    @observed\n    def verify_complete(self):\n        _require(self.dataset is None and self.completed == set(self.pairs.frame_counts), "incomplete graph capture")\n        self.pairs.finish()\n        for item in self.artifacts:\n            path = self.root / item["path"]\n            _require(path.stat().st_size == item["bytes"] and _sha(path) == item["sha256"], "graph artifact changed")\n        self.graph_bytes += (self.pairs.root / "MANIFEST.json").stat().st_size\n\n    def finish(self):\n        self.verify_complete()\n        result = {"status": "ASSOCIATION_OBSERVATION_COMPLETE_NOT_PARITY", "capture_only": True,\n                  "graph_ID_mapping_complete": True, "inference_parity_verified": False,\n                  "submission_authorized": False, "datasets": sorted(self.completed),\n                  "candidate_threshold": .48, "candidate_limits": {"parents": None, "children": None},\n                  "observer_seconds_excluding_final_manifest": self.seconds,\n                  "time_limit_enforcement": "synchronous_call_boundaries",\n                  "pair_manifest_sha256": _sha(self.pairs.root / "MANIFEST.json"),\n                  "graphs": self.artifacts}\n        encoded = _json_bytes(result)\n        try:\n            _require(self.pairs.output_bytes + self.graph_bytes + len(encoded) <= self.pairs.limits.output_bytes,\n                     "observer final manifest byte budget exceeded")\n            with (self.root / "MANIFEST.json").open("xb") as stream:\n                stream.write(encoded)\n        except Exception as exc:\n            self.failed = True\n            with (self.root / "ERROR.json").open("xb") as stream:\n                stream.write(_json_bytes({"status": "ERROR", "error": str(exc), "submission_authorized": False}))\n            raise\n        self.closed = True\n        return json.loads(encoded)\n', 'src/biohub/association_instrumentation.py': '"""Static-only, reversible observer insertion into the pinned E23 reconstruction."""\n\nimport ast\nimport hashlib\n\nSOURCE_SHA256 = "8e7ac19e8b436d6777576b7ea2269464c4ed8842990b45448d40bf178f8f62de"\n\n# Anchors are never replaced. Each addition is removable byte-for-byte.\nINSERTIONS = (\n    ("    edges: list[tuple[int, int, float, float]],\\n", "    association_observer=None,\\n"),\n    ("    secondary_low_margin_max: float = 0.2,\\n", "    association_observer=None,\\n"),\n    ("    evaluate: bool = False,\\n", "    association_observer=None,\\n"),\n    ("    pool_k = pool_kernel_from_um(cfg.pool_kernel_um, voxel_size)\\n",\n     "    if association_observer is not None:\\n"\n     "        association_observer.start_video(\\n"\n     "            ds_path.stem, T, W, downsample, ds.scale, cfg.edge_activation,\\n"\n     "            threshold=cfg.threshold, max_parents=cfg.max_parents_per_node,\\n"\n     "            max_children=cfg.max_children_per_node)\\n"),\n    ("            s_tgt, e_tgt = coord_offset[t_tgt]\\n",\n     "            if association_observer is not None:\\n"\n     "                association_observer.start_pair(t_src, s_src, e_src, s_tgt, e_tgt, coords_so_far)\\n"),\n    ("            )  # (1, n_src, n_tgt)\\n",\n     "            if association_observer is not None:\\n"\n     "                association_observer.primary(edge_logits_pair, unet_feat_src, unet_feat_tgt)\\n"),\n    ("                reverse_logits_pair = reverse_logits_native.transpose(1, 2)\\n",\n     "                if association_observer is not None:\\n"\n     "                    association_observer.reverse(reverse_logits_pair)\\n"),\n    ("                    p_mask_src, p_mask_tgt,\\n                )\\n",\n     "                if association_observer is not None:\\n"\n     "                    association_observer.secondary(\\n"\n     "                        secondary_logits_pair, secondary_feat_src, secondary_feat_tgt)\\n"),\n    ("                probs = torch.sigmoid(raw).cpu().numpy()\\n",\n     "            if association_observer is not None:\\n"\n     "                association_observer.mixed(raw, probs)\\n"),\n    ("    return coords, all_edges\\n",\n     "    if association_observer is not None:\\n"\n     "        association_observer.end_video(coords, all_edges)\\n"),\n    ("    return graph\\n",\n     "    if association_observer is not None:\\n"\n     "        association_observer.pre_graph(coords, edges, node_ids, graph)\\n"),\n    ("                secondary_low_margin_max=secondary_low_margin_max,\\n",\n     "                association_observer=association_observer,\\n"),\n    ("        graph = build_graph(coords, edges)\\n",\n     ""),\n    ("        save_graph(graph, output_dir / f\\"{name}.geff\\")\\n",\n     "        if association_observer is not None:\\n"\n     "            association_observer.selected_graph(graph)\\n"),\n)\n\n\ndef instrument_source(source: str) -> tuple[str, dict]:\n    """Return text and a receipt. Never imports or executes the supplied source.\n\n    The build_graph call needs a keyword, so insert it before the closing paren\n    instead of replacing the call. All original executable AST nodes survive.\n    """\n    digest = hashlib.sha256(source.encode()).hexdigest()\n    if digest != SOURCE_SHA256:\n        raise ValueError("E23 reconstructed source SHA mismatch")\n    patched = source\n    additions = []\n    for index, (anchor, body) in enumerate(INSERTIONS):\n        if patched.count(anchor) != 1:\n            raise ValueError(f"observer anchor {index} is not unique")\n        if not body:  # build_graph keyword insertion, not a replacement function\n            original = "build_graph(coords, edges)"\n            addition = ", association_observer=association_observer"\n            patched = patched.replace(original, original[:-1] + addition + ")", 1)\n            additions.append((original[:-1] + addition + ")", original))\n            continue\n        # End-of-function and post-solver hooks must precede return/save.\n        before = anchor.lstrip().startswith(("return ", "save_graph("))\n        replacement = body + anchor if before else anchor + body\n        patched = patched.replace(anchor, replacement, 1)\n        additions.append((replacement, anchor))\n    restored = patched\n    for inserted, original in reversed(additions):\n        if restored.count(inserted) != 1:\n            raise ValueError("observer removal is ambiguous")\n        restored = restored.replace(inserted, original, 1)\n    if restored != source or ast.dump(ast.parse(restored)) != ast.dump(ast.parse(source)):\n        raise ValueError("original E23 program changed")\n    ast.parse(patched)\n    receipt = {"status": "STATIC_OBSERVER_INSERTION_ONLY", "original_sha256": digest,\n               "instrumented_sha256": hashlib.sha256(patched.encode()).hexdigest(),\n               "restored_bytes_equal": True, "restored_ast_equal": True,\n               "insertions": len(additions), "source_executed": False,\n               "inference_parity_verified": False, "submission_authorized": False}\n    return patched, receipt\n', 'src/biohub/association_parity.py': '"""Common OFF/ON inference bridge and exact observation-parity comparison."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport os\nimport random\nimport resource\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom biohub.association_capture import _json_bytes, _require, _sha\nfrom biohub.association_observer import AssociationObserver\n\nE23_ENV = {\n    "BIOHUB_SECONDARY_EDGE_WEIGHT": "0.15", "BIOHUB_SECONDARY_DETECTION_WEIGHT": "0.475",\n    "BIOHUB_SECONDARY_LINK_MODE": "low_margin_consensus", "BIOHUB_SECONDARY_MIX_TEMPERATURE": "1",\n    "BIOHUB_SECONDARY_LOW_MARGIN_MAX": "0.35", "BIOHUB_DUAL_SEED_EDGE_THRESHOLD": "0.48",\n    "BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT": "0.30", "BIOHUB_BIDIRECTIONAL_FUSION_MODE": "harmonic_probability",\n    "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION": "0.90",\n}\nCFG = {"det_threshold": .96875, "use_ilp": True, "ilp_edge_weight": -1.,\n       "ilp_appearance_weight": 0., "ilp_disappearance_weight": 1.5, "ilp_division_weight": 1.}\n\n\ndef array_signatures(arrays):\n    return {k: {"dtype": a.dtype.str, "shape": list(a.shape),\n                "sha256": hashlib.sha256(a.tobytes(order="C")).hexdigest()} for k, a in sorted(arrays.items())}\n\n\ndef graph_arrays(graph):\n    nodes = graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"]).sort("node_id")\n    edge_keys = ["edge_id", "source_id", "target_id"]\n    if graph.num_edges():\n        edge_keys += ["edge_prob", "edge_dist"]\n    edges = graph.edge_attrs(attr_keys=edge_keys).sort("source_id", "target_id", "edge_id")\n    return {f"{prefix}_{column}": table[column].to_numpy()\n            for prefix, table in (("node", nodes), ("edge", edges)) for column in table.columns}\n\n\ndef semantic_graph_signature(arrays):\n    """GEFF has endpoint pairs rather than edge IDs; compare lossless numeric values."""\n    keys = ("node_node_id", "node_t", "node_z", "node_y", "node_x",\n            "edge_source_id", "edge_target_id", "edge_edge_prob", "edge_edge_dist")\n    canonical = {}\n    for k in keys:\n        raw = arrays.get(k, np.empty(0, dtype=np.float64))\n        dtype = np.int64 if k.endswith("_id") or k == "node_t" else np.float64\n        canonical[k] = raw.astype(dtype)\n        _require(np.array_equal(canonical[k], raw), f"lossy graph canonicalization: {k}")\n    return array_signatures(canonical)\n\n\ndef compare_results(off, on):\n    for result, arm in ((off, "off"), (on, "on")):\n        _require(result["status"] == "ARM_COMPLETE" and result["arm"] == arm, "incomplete/wrong arm")\n        names = result["dataset_order"]\n        _require(len(names) == 4 and len(set(names)) == 4 and names == sorted(names), "incomplete dataset coverage")\n        records = result["records"]\n        _require(len(records) == 12 and {(r["dataset"], r["stage"]) for r in records}\n                 == {(name, stage) for name in names for stage in ("returned", "pre", "post")},\n                 "incomplete trace coverage")\n    common = ("dataset_order", "input_inventory", "dependencies", "environment", "model_config", "rng_after")\n    for key in common:\n        _require(off[key] == on[key], f"OFF/ON {key} mismatch")\n    _require(off["plan_sha256"] == on["plan_sha256"], "OFF/ON plan mismatch")\n    _require(off["records"] == on["records"], "OFF/ON prediction/graph arrays mismatch")\n    _require(on["observation_complete"], "ON observer incomplete")\n    return {"status": "PUBLIC4_ASSOCIATION_OBSERVER_PARITY_PASS", "datasets": off["dataset_order"],\n            "reference_raw_matched": True, "submission_authorized": False,\n            "full36_capture_complete": False, "generalization_evidence": False}\n\n\ndef input_inventory(root, datasets):\n    result = []\n    for name in datasets:\n        folder = root / f"{name}.zarr"\n        _require(folder.is_dir(), "missing planned image dataset")\n        for path in sorted(folder.rglob("*")):\n            if path.is_file():\n                result.append({"path": path.relative_to(root).as_posix(),\n                               "bytes": path.stat().st_size, "sha256": _sha(path)})\n    return result\n\n\ndef _rng_digest(torch):\n    numpy_state = np.random.get_state()\n    return {"python": hashlib.sha256(_json_bytes(random.getstate())).hexdigest(),\n            "numpy": hashlib.sha256(numpy_state[1].tobytes() + _json_bytes(\n                [numpy_state[0], *numpy_state[2:]])).hexdigest(),\n            "torch_cpu": hashlib.sha256(torch.get_rng_state().numpy().tobytes()).hexdigest(),\n            "torch_cuda": [hashlib.sha256(s.cpu().numpy().tobytes()).hexdigest()\n                           for s in torch.cuda.get_rng_state_all()]}\n\n\ndef capture_module(module, plan, root, arm):\n    """Run one prepared module. Synthetic tests supply a module-shaped fixture."""\n    _require(arm in ("off", "on"), "unknown arm")\n    names = sorted(plan["datasets"])\n    observer = AssociationObserver(root / "observation", {\n        name: plan["datasets"][name]["frame_counts"] for name in names}) if arm == "on" else None\n    original_video, original_build, original_save = module.predict_video, module.build_graph, module.save_graph\n    records = {}\n    current = None\n    stage = "video"\n    order = []\n\n    def save_record(name, label, arrays):\n        _require((name, label) not in records, "duplicate trace record")\n        _require(all(isinstance(a, np.ndarray) and not a.dtype.hasobject and np.isfinite(a).all()\n                     for a in arrays.values()), "invalid trace array")\n        path = root / f"{name}_{label}.npz"\n        with path.open("xb") as stream:\n            np.savez_compressed(stream, **arrays)\n        with np.load(path, allow_pickle=False) as saved:\n            _require(array_signatures({k: saved[k] for k in saved.files}) == array_signatures(arrays),\n                     "trace roundtrip mismatch")\n        records[name, label] = {"dataset": name, "stage": label, "arrays": array_signatures(arrays)}\n\n    def video(*args, **kwargs):\n        nonlocal current, stage\n        _require(stage == "video", "overlapping prediction video")\n        name = Path(args[1]).stem\n        _require(name in names and name not in order, "unplanned or duplicate prediction video")\n        current = name\n        coords, edges = original_video(*args, **kwargs)\n        expected = plan["datasets"][name]\n        _require(coords.dtype == np.dtype("int16") and coords.shape == (sum(expected["frame_counts"]), 4),\n                 "reference detection shape/dtype mismatch")\n        _require(hashlib.sha256(coords.astype("<i2", copy=False).tobytes()).hexdigest()\n                 == expected["coordinate_sha256"], "reference detection coordinate mismatch")\n        _require([int(np.count_nonzero(coords[:, 0] == t)) for t in range(100)] == expected["frame_counts"],\n                 "reference detector counts mismatch")\n        save_record(name, "returned", {"coords": coords, "edges": np.array(edges, dtype=np.float64).reshape(-1, 4)})\n        order.append(name)\n        stage = "pre"\n        return coords, edges\n\n    def build(*args, **kwargs):\n        nonlocal stage\n        _require(stage == "pre", "unexpected graph build")\n        graph = original_build(*args, **kwargs)\n        save_record(current, "pre", graph_arrays(graph))\n        stage = "post"\n        return graph\n\n    def save(graph, path, *args, **kwargs):\n        nonlocal stage\n        _require(stage == "post" and Path(path).stem == current, "unexpected graph save")\n        arrays = graph_arrays(graph)\n        _require(semantic_graph_signature(arrays) == plan["datasets"][current]["raw_graph_signature"],\n                 "reference selected raw graph mismatch")\n        save_record(current, "post", arrays)\n        result = original_save(graph, path, *args, **kwargs)\n        stage = "video"\n        return result\n\n    module.predict_video, module.build_graph, module.save_graph = video, build, save\n    try:\n        kwargs = {"data_dir": Path(plan["data_dir"]), "fold": 0, "splits_file": Path(plan["splits_file"]),\n                  "weights_path": Path(plan["primary_weights"]), "cfg": module.PredictConfig(**CFG),\n                  "method": "unet_transformer", "unet_batch_size": 4, "evaluate": False}\n        if observer is not None:\n            kwargs["association_observer"] = observer\n        module.predict(**kwargs)\n        _require(stage == "video" and order == names and len(records) == len(names) * 3, "incomplete trace")\n        if observer is not None:\n            observer.finish()\n        return {"dataset_order": order, "records": [records[key] for key in sorted(records)],\n                "observation_complete": observer is not None}\n    finally:\n        module.predict_video, module.build_graph, module.save_graph = original_video, original_build, original_save\n\n\ndef run_arm(plan_path, source, root, arm):\n    """Kaggle-only entry. Fail closed before importing a source with unverified bytes."""\n    _require(Path("/kaggle/working").is_dir(), "real model execution is Kaggle-only")\n    plan = json.loads(plan_path.read_text())\n    _require(plan["schema_version"] == "biohub.association.public4.v1" and len(plan["datasets"]) == 4,\n             "invalid public4 plan")\n    _require(arm in ("off", "on") and _sha(source) == plan["source_sha256"][arm], "source SHA mismatch")\n    _require({key: os.environ.get(key) for key in E23_ENV} == E23_ENV, "effective E23 environment mismatch")\n    _require(set(p.stem for p in Path(plan["data_dir"]).glob("*.zarr")) == set(plan["datasets"]),\n             "public dataset set mismatch")\n    for path, expected in plan["runtime_file_sha256"].items():\n        _require(_sha(Path(path)) == expected, f"runtime file changed: {path}")\n    for record in plan["source_inventory"]:\n        _require(_sha(Path(record["path"])) == record["sha256"], "source inventory changed")\n    root.mkdir(parents=True, exist_ok=False)\n    started = time.perf_counter()\n    try:\n        inputs = input_inventory(Path(plan["data_dir"]), sorted(plan["datasets"]))\n        import torch\n        _require(torch.cuda.is_available() and torch.cuda.device_count() == 1, "expected one visible CUDA device")\n        # Common diagnostic seed; E23 saved raw parity must still pass, no baseline replacement.\n        random.seed(23826)\n        np.random.seed(23826)\n        torch.manual_seed(23826)\n        torch.cuda.manual_seed_all(23826)\n        dependencies = {name: importlib.metadata.version(name) for name in\n                        ("numpy", "torch", "polars", "tracksdata", "pyscipopt", "zarr", "geff", "scipy")}\n        spec = importlib.util.spec_from_file_location("biohub_e23_target_predict", source)\n        module = importlib.util.module_from_spec(spec)\n        sys.modules[spec.name] = module\n        spec.loader.exec_module(module)\n        import dataspec\n        # The original predictor deletes an existing output; point it only at this fresh arm.\n        dataspec.PREDICTIONS_PATH = root / "raw_predictions"\n        torch.cuda.synchronize()\n        result = capture_module(module, plan, root, arm)\n        torch.cuda.synchronize()\n        _require(inputs == input_inventory(Path(plan["data_dir"]), sorted(plan["datasets"])), "image bytes changed")\n        for path, expected in plan["runtime_file_sha256"].items():\n            _require(_sha(Path(path)) == expected, "runtime file changed after inference")\n        for record in plan["source_inventory"]:\n            _require(_sha(Path(record["path"])) == record["sha256"], "source changed after inference")\n        _require(_sha(source) == plan["source_sha256"][arm], "prediction source changed")\n        result.update(status="ARM_COMPLETE", arm=arm, input_inventory=inputs, dependencies=dependencies,\n                      environment=E23_ENV, model_config=CFG, rng_after=_rng_digest(torch),\n                      plan_sha256=_sha(plan_path), source_sha256=_sha(source), submission_authorized=False,\n                      wall_seconds=time.perf_counter() - started,\n                      peak_cuda_allocated_bytes=torch.cuda.max_memory_allocated(),\n                      peak_cuda_reserved_bytes=torch.cuda.max_memory_reserved(),\n                      peak_self_rss_bytes=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024)\n        _require(result["wall_seconds"] <= 3600 and result["peak_self_rss_bytes"] <= 24 * 1024**3,\n                 "arm runtime/RSS budget exceeded")\n        _require(sum(p.stat().st_size for p in root.rglob("*") if p.is_file()) <= 12 * 1024**3,\n                 "arm output budget exceeded")\n        with (root / "RESULT.json").open("xb") as stream:\n            stream.write(_json_bytes(result))\n        print(json.dumps({"status": result["status"], "arm": arm, "wall_seconds": result["wall_seconds"]}), flush=True)\n        return result\n    except Exception as exc:\n        with (root / "ERROR.json").open("xb") as stream:\n            stream.write(_json_bytes({"status": "ERROR", "error": str(exc), "submission_authorized": False}))\n        raise\n', 'src/biohub/association_supervise.py': '"""Serial bounded Kaggle OFF/ON supervision; never submits or retries."""\n\nimport json\nimport os\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom biohub.association_capture import _json_bytes, _require, _sha\nfrom biohub.association_parity import compare_results\n\n\ndef process_tree_rss(pid, proc_root=Path("/proc")):\n    """Linux resident-page sampling, following children of every thread, no extra package."""\n    pending, seen, pages = [pid], set(), 0\n    while pending:\n        current = pending.pop()\n        if current in seen:\n            continue\n        seen.add(current)\n        folder = proc_root / str(current)\n        try:\n            pages += int((folder / "statm").read_text().split()[1])\n            for children in (folder / "task").glob("*/children"):\n                pending.extend(int(value) for value in children.read_text().split())\n        except FileNotFoundError:\n            continue  # Process/thread ended between samples; exit status is checked separately.\n    return pages * os.sysconf("SC_PAGE_SIZE")\n\n\ndef supervise(plan_path, root, payload, deadline):\n    _require(Path("/kaggle/working").is_dir(), "supervisor is Kaggle-only")\n    root.mkdir(parents=True, exist_ok=False)\n    _require(shutil.disk_usage(root).free >= 16 * 1024**3, "less than 16 GiB free")\n    plan = json.loads(plan_path.read_text())\n    completed = {}\n    process_records = {}\n    tokens = [t.strip() for t in os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",") if t.strip()]\n    _require(tokens and tokens[0] != "-1", "no CUDA token")\n    plan_sha = _sha(plan_path)\n    for arm in ("off", "on"):\n        _require(time.monotonic() < deadline, "diagnostic wall budget exceeded before arm")\n        command = [sys.executable, str(payload / "scripts/e23_association_bridge.py"),\n                   "--plan", str(plan_path), "--source", plan["source_paths"][arm],\n                   "--root", str(root / arm), "--arm", arm]\n        env = {**os.environ, "PYTHONPATH": str(payload / "src") + os.pathsep + str(Path(plan["repo_dir"]) / "src"),\n               "CUDA_VISIBLE_DEVICES": tokens[0], "BIOHUB_GPU_SHARD": "single",\n               "PYTHONHASHSEED": "0"}\n        started, peak_rss = time.monotonic(), 0\n        error = None\n        with (root / f"{arm}.log").open("xb") as log:\n            process = subprocess.Popen(command, cwd=plan["repo_dir"], env=env, stdout=log,\n                                       stderr=subprocess.STDOUT, start_new_session=True)\n            print(f"D3 ARM_STARTED {arm} PID {process.pid}", flush=True)\n            try:\n                while process.poll() is None:\n                    if time.monotonic() >= min(started + 3600, deadline):\n                        raise RuntimeError("arm/diagnostic wall budget exceeded")\n                    peak_rss = max(peak_rss, process_tree_rss(process.pid))\n                    _require(peak_rss <= 24 * 1024**3, "process-tree RSS budget exceeded")\n                    size = sum(p.stat().st_size for p in (root / arm).rglob("*") if p.is_file())\n                    _require(size <= 12 * 1024**3, "arm output budget exceeded")\n                    time.sleep(1.)\n            except Exception as exc:\n                error = str(exc)\n                if process.poll() is None:\n                    os.killpg(process.pid, signal.SIGTERM)\n                    try:\n                        process.wait(timeout=15)\n                    except subprocess.TimeoutExpired:\n                        os.killpg(process.pid, signal.SIGKILL)\n                        process.wait()\n            code = process.wait()\n        process_records[arm] = {"returncode": code, "wall_seconds": time.monotonic() - started,\n                                "peak_sampled_tree_rss_bytes": peak_rss, "error": error}\n        with (root / f"{arm}_PROCESS.json").open("xb") as stream:\n            stream.write(_json_bytes(process_records[arm]))\n        print(f"D3 ARM_FINISHED {arm} exit={code}", flush=True)\n        _require(code == 0 and error is None, "arm failed; inspect retained log/PROCESS/ERROR")\n        _require(_sha(plan_path) == plan_sha, "plan changed during diagnostic")\n        completed[arm] = json.loads((root / arm / "RESULT.json").read_text())\n    result = compare_results(completed["off"], completed["on"])\n    result.update(processes=process_records, plan_sha256=plan_sha,\n                  arm_result_sha256={arm: _sha(root / arm / "RESULT.json") for arm in completed})\n    _require(time.monotonic() <= deadline, "diagnostic deadline exceeded")\n    with (root / "PARITY_RESULT.json").open("xb") as stream:\n        stream.write(_json_bytes(result))\n    print(json.dumps(result, sort_keys=True), flush=True)\n    return result\n', 'scripts/e23_association_bridge.py': '"""Run one frozen public-four E23 observation arm, only inside Kaggle."""\n\nimport argparse\nfrom pathlib import Path\n\nfrom biohub.association_parity import run_arm\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--plan", type=Path, required=True)\n    parser.add_argument("--source", type=Path, required=True)\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--arm", choices=("off", "on"), required=True)\n    args = parser.parse_args()\n    run_arm(args.plan, args.source, args.root, args.arm)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/biohub/association_collection.py': '"""Kaggle-only collection groups, reusing the frozen E23 trace without changing it."""\n\nimport importlib.metadata\nimport importlib.util\nimport json\nimport os\nimport random\nimport resource\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom biohub.association_capture import _json_bytes, _require, _sha\nfrom biohub.association_parity import CFG, E23_ENV, _rng_digest, capture_module, input_inventory\n\nREFERENCE_SHA = "08395ca8615b6eeaaaeec521cd937c62d2212b7a20cf18be1a27478afd4af7ac"\nGROUP_SCHEMA = "biohub.association.collection36.group.v1"\n\n\ndef load_reference(path):\n    _require(_sha(path) == REFERENCE_SHA, "collection reference SHA mismatch")\n    reference = json.loads(path.read_text())\n    _require(reference["schema_version"] == "biohub.association.collection36.preparation.v1"\n             and len(reference["groups"]) == 9 and len(reference["datasets"]) == 36,\n             "invalid collection reference")\n    return reference\n\n\ndef validate_plan(plan):\n    _require(plan["schema_version"] == GROUP_SCHEMA and plan["submission_authorized"] is False,\n             "invalid collection group schema")\n    reference = load_reference(Path(plan["reference_path"]))\n    groups = {g["group_id"]: g for g in reference["groups"]}\n    _require(plan["group_id"] in groups, "unplanned collection group")\n    group = groups[plan["group_id"]]\n    expected = {n: reference["datasets"][n] for n in group["datasets"]}\n    _require(plan["datasets"] == expected and len(expected) == 4, "collection dataset/reference mismatch")\n    _require(plan["source_sha256"] == reference["source_sha256"]["on"], "ON source contract mismatch")\n    _require({key: os.environ.get(key) for key in E23_ENV} == E23_ENV, "effective E23 environment mismatch")\n    image_root = Path(plan["data_dir"])\n    _require(set(p.name for p in image_root.iterdir()) == {f"{n}.zarr" for n in expected},\n             "image-only view contains unplanned entries")\n    _require(all((image_root / f"{n}.zarr").is_dir() for n in expected), "missing image directory")\n    split = json.loads(Path(plan["splits_file"]).read_text())\n    _require(split == [{"split": 0, "train": [], "test": sorted(expected)}], "collection split mismatch")\n    _require(bool(plan["runtime_file_sha256"]) and bool(plan["source_inventory"]), "unbound runtime")\n    _require(plan["primary_weights"] in plan["runtime_file_sha256"]\n             and plan["splits_file"] in plan["runtime_file_sha256"], "missing required runtime binding")\n    for path, digest in plan["runtime_file_sha256"].items():\n        _require(_sha(Path(path)) == digest, f"runtime changed: {path}")\n    for entry in plan["source_inventory"]:\n        _require(_sha(Path(entry["path"])) == entry["sha256"], "source inventory changed")\n    _require(_sha(Path(plan["source_path"])) == plan["source_sha256"], "prediction source changed")\n    return reference, group\n\n\ndef validate_group_result(result, plan, group, plan_sha):\n    names = group["datasets"]\n    _require(result["status"] == "COLLECTION_GROUP_COMPLETE" and result["group_id"] == group["group_id"],\n             "incomplete/wrong group")\n    _require(result["plan_sha256"] == plan_sha and result["source_sha256"] == plan["source_sha256"],\n             "group provenance mismatch")\n    _require(result["dataset_order"] == names and result["observation_complete"], "group coverage incomplete")\n    _require(len(result["records"]) == 12 and {(r["dataset"], r["stage"]) for r in result["records"]}\n             == {(n, s) for n in names for s in ("returned", "pre", "post")}, "trace coverage incomplete")\n    _require(result["pair_count"] == 396 and result["dense_pairs"] == group["dense_pairs"], "pair coverage incomplete")\n    _require(result["environment"] == E23_ENV and result["model_config"] == CFG, "group configuration drift")\n    _require(result["submission_authorized"] is False and result["generalization_evidence"] is False,\n             "wrong collection claims")\n    for key in ("wall_seconds", "writer_seconds", "observer_seconds", "peak_self_rss_bytes"):\n        _require(type(result[key]) in (int, float) and np.isfinite(result[key]) and result[key] >= 0,\n                 f"invalid resource value: {key}")\n    _require(result["wall_seconds"] <= 3600 and result["peak_self_rss_bytes"] <= 24 * 1024**3,\n             "group runtime/RSS budget exceeded")\n\n\ndef run_group(plan_path, root):\n    _require(Path("/kaggle/working").is_dir(), "model execution is Kaggle-only")\n    root.mkdir(parents=True, exist_ok=False)\n    started = time.perf_counter()\n    try:\n        plan_sha = _sha(plan_path)\n        plan = json.loads(plan_path.read_text())\n        _, group = validate_plan(plan)\n        names = group["datasets"]\n        inputs = input_inventory(Path(plan["data_dir"]), names)\n        _require(bool(inputs), "empty image inventory")\n        import torch\n\n        _require(torch.cuda.is_available() and torch.cuda.device_count() == 1, "expected one visible CUDA device")\n        random.seed(23826)\n        np.random.seed(23826)\n        torch.manual_seed(23826)\n        torch.cuda.manual_seed_all(23826)\n        dependencies = {n: importlib.metadata.version(n) for n in\n                        ("numpy", "torch", "polars", "tracksdata", "pyscipopt", "zarr", "geff", "scipy")}\n        spec = importlib.util.spec_from_file_location("biohub_e23_collection_predict", plan["source_path"])\n        module = importlib.util.module_from_spec(spec)\n        sys.modules[spec.name] = module\n        spec.loader.exec_module(module)\n        import dataspec\n\n        dataspec.PREDICTIONS_PATH = root / "raw_predictions"\n        torch.cuda.synchronize()\n        result = capture_module(module, plan, root, "on")\n        torch.cuda.synchronize()\n        _require(_sha(plan_path) == plan_sha, "group plan changed")\n        validate_plan(plan)\n        _require(inputs == input_inventory(Path(plan["data_dir"]), names), "image bytes changed")\n        observer_path = root / "observation/MANIFEST.json"\n        pairs_path = root / "observation/pairs/MANIFEST.json"\n        observer, pairs = (json.loads(p.read_text()) for p in (observer_path, pairs_path))\n        _require(observer["pair_manifest_sha256"] == _sha(pairs_path), "pair manifest changed")\n        _require(observer["graph_ID_mapping_complete"] and observer["datasets"] == names\n                 and observer["status"] == "ASSOCIATION_OBSERVATION_COMPLETE_NOT_PARITY"\n                 and pairs["status"] == "PAIR_CAPTURE_COMPLETE", "observation incomplete")\n        result.update(status="COLLECTION_GROUP_COMPLETE", group_id=group["group_id"], plan_sha256=plan_sha,\n                      source_sha256=plan["source_sha256"], input_inventory=inputs, dependencies=dependencies,\n                      environment=E23_ENV, model_config=CFG, rng_after=_rng_digest(torch),\n                      reference_sha256=REFERENCE_SHA, reference_raw_matched=True, submission_authorized=False,\n                      generalization_evidence=False, paired_off_on_this_group=False,\n                      pair_count=len(pairs["records"]), dense_pairs=pairs["dense_pairs"],\n                      packet_bytes=pairs["packet_bytes"], writer_seconds=pairs["writer_seconds"],\n                      observer_seconds=observer["observer_seconds_excluding_final_manifest"],\n                      observer_manifest_sha256=_sha(observer_path), pair_manifest_sha256=_sha(pairs_path),\n                      wall_seconds=time.perf_counter() - started,\n                      peak_self_rss_bytes=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024,\n                      peak_cuda_allocated_bytes=torch.cuda.max_memory_allocated(),\n                      peak_cuda_reserved_bytes=torch.cuda.max_memory_reserved())\n        validate_group_result(result, plan, group, plan_sha)\n        encoded = _json_bytes(result)\n        _require(sum(p.stat().st_size for p in root.rglob("*") if p.is_file()) + len(encoded) <= 12 * 1024**3,\n                 "group output budget exceeded")\n        with (root / "RESULT.json").open("xb") as stream:\n            stream.write(encoded)\n        print(json.dumps({"status": result["status"], "group_id": group["group_id"],\n                          "wall_seconds": result["wall_seconds"]}), flush=True)\n        return result\n    except Exception as exc:\n        with (root / "ERROR.json").open("xb") as stream:\n            stream.write(_json_bytes({"status": "ERROR", "error": str(exc), "submission_authorized": False}))\n        raise\n', 'src/biohub/association_collection_supervise.py': '"""Nine serial collection groups, cumulative budgets, no retries or submission."""\n\nimport json\nimport os\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom biohub.association_capture import _json_bytes, _require, _sha\nfrom biohub.association_collection import load_reference, validate_group_result\nfrom biohub.association_supervise import process_tree_rss\n\n\ndef output_bytes(root):\n    return sum(p.stat().st_size for p in root.rglob("*") if p.is_file())\n\n\ndef record(path, value):\n    with path.open("xb") as stream:\n        stream.write(_json_bytes(value))\n\n\ndef check_cumulative(results, limits):\n    totals = {key: sum(r[key] for r in results) for key in\n              ("pair_count", "dense_pairs", "packet_bytes", "writer_seconds", "observer_seconds")}\n    _require(totals["writer_seconds"] <= limits["whole_writer_seconds"], "cumulative writer time exceeded")\n    _require(totals["observer_seconds"] <= limits["whole_observer_seconds"], "cumulative observer time exceeded")\n    _require(totals["dense_pairs"] <= limits["dense_pairs_per_run"], "cumulative dense pair budget exceeded")\n    return totals\n\n\ndef supervise_collection(master_path, root, payload, notebook_started):\n    _require(Path("/kaggle/working").is_dir(), "collection supervisor is Kaggle-only")\n    root.mkdir(parents=True, exist_ok=False)\n    completed, results = [], []\n    try:\n        master_sha = _sha(master_path)\n        master = json.loads(master_path.read_text())\n        _require(master["schema_version"] == "biohub.association.collection36.master.v1"\n                 and master["submission_authorized"] is False, "invalid collection master")\n        reference = load_reference(Path(master["reference_path"]))\n        limits = reference["limits"]\n        groups = reference["groups"]\n        _require([r["group_id"] for r in master["group_plans"]] == [g["group_id"] for g in groups],\n                 "master group order/coverage mismatch")\n        _require(len({r["path"] for r in master["group_plans"]}) == 9, "duplicate group plan")\n        _require(shutil.disk_usage(root).free >= limits["start_free_disk_bytes"], "less than 16 GiB free")\n        deadline = notebook_started + limits["whole_job_wall_seconds"]\n        tokens = [t.strip() for t in os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",") if t.strip()]\n        _require(tokens and tokens[0] != "-1", "no CUDA token")\n        for group, bound in zip(groups, master["group_plans"], strict=True):\n            gid = group["group_id"]\n            plan_path = Path(bound["path"])\n            _require(_sha(master_path) == master_sha and _sha(plan_path) == bound["sha256"], "plan changed")\n            plan = json.loads(plan_path.read_text())\n            _require(plan["group_id"] == gid and plan["reference_path"] == master["reference_path"],\n                     "wrong group plan")\n            _require(plan["datasets"] == {n: reference["datasets"][n] for n in group["datasets"]},\n                     "group references changed")\n            _require(time.monotonic() < deadline, "whole deadline before group")\n            _require(output_bytes(root) <= limits["whole_output_bytes"], "whole output budget exceeded")\n            command = [sys.executable, str(payload / "scripts/e23_association_collect.py"),\n                       "--plan", str(plan_path), "--root", str(root / gid)]\n            env = {**os.environ,\n                   "PYTHONPATH": str(payload / "src") + os.pathsep + str(Path(master["repo_dir"]) / "src"),\n                   "CUDA_VISIBLE_DEVICES": tokens[0], "BIOHUB_GPU_SHARD": "single", "PYTHONHASHSEED": "0"}\n            started, peak_rss, error = time.monotonic(), 0, None\n            with (root / f"{gid}.log").open("xb") as log:\n                process = subprocess.Popen(command, cwd=master["repo_dir"], env=env, stdout=log,\n                                           stderr=subprocess.STDOUT, start_new_session=True)\n                print(f"D3 COLLECTION_STARTED {gid} PID {process.pid}", flush=True)\n                try:\n                    while process.poll() is None:\n                        _require(time.monotonic() < min(started + limits["group_wall_seconds"], deadline),\n                                 "group/whole wall budget exceeded")\n                        peak_rss = max(peak_rss, process_tree_rss(process.pid))\n                        _require(peak_rss <= limits["rss_bytes"], "process-tree RSS budget exceeded")\n                        _require(output_bytes(root) <= limits["whole_output_bytes"], "whole output budget exceeded")\n                        time.sleep(1.)\n                except Exception as exc:\n                    error = str(exc)\n                    if process.poll() is None:\n                        try:\n                            os.killpg(process.pid, signal.SIGTERM)\n                        except ProcessLookupError:\n                            pass\n                        try:\n                            process.wait(timeout=15)\n                        except subprocess.TimeoutExpired:\n                            os.killpg(process.pid, signal.SIGKILL)\n                            process.wait()\n                code = process.wait()\n            process_record = {"returncode": code, "error": error, "wall_seconds": time.monotonic() - started,\n                              "peak_sampled_tree_rss_bytes": peak_rss}\n            record(root / f"{gid}_PROCESS.json", process_record)\n            print(f"D3 COLLECTION_FINISHED {gid} exit={code}", flush=True)\n            _require(code == 0 and error is None, "collection group failed; retained PROCESS/log/ERROR")\n            _require(process_record["wall_seconds"] <= limits["group_wall_seconds"]\n                     and time.monotonic() <= deadline, "completed group exceeded wall budget")\n            _require(_sha(master_path) == master_sha and _sha(plan_path) == bound["sha256"], "plan changed after group")\n            result_path = root / gid / "RESULT.json"\n            result = json.loads(result_path.read_text())\n            validate_group_result(result, plan, group, bound["sha256"])\n            _require(result["dependencies"] == master["expected_dependencies"], "dependencies differ from public4")\n            for filename, key in (("observation/MANIFEST.json", "observer_manifest_sha256"),\n                                  ("observation/pairs/MANIFEST.json", "pair_manifest_sha256")):\n                _require(_sha(root / gid / filename) == result[key], "observation manifest changed")\n            results.append(result)\n            totals = check_cumulative(results, limits)\n            completed.append({"group_id": gid, "datasets": group["datasets"],\n                              "result_sha256": _sha(result_path), "process": process_record})\n            record(root / f"{gid}_PROGRESS.json", {"status": "COLLECTION_PARTIAL", "completed_groups": completed,\n                                                  "totals": totals, "submission_authorized": False})\n        names = [n for group in completed for n in group["datasets"]]\n        _require(len(names) == len(set(names)) == 36 and set(names) == set(reference["datasets"]),\n                 "incomplete final video coverage")\n        _require(totals["pair_count"] == reference["expected_pair_packets"]\n                 and totals["dense_pairs"] == reference["expected_dense_pairs"], "incomplete final pair coverage")\n        result = {"status": "COLLECTION36_COMPLETE_REFERENCE_MATCHED", "completed_groups": completed,\n                  "totals": totals, "master_sha256": master_sha,\n                  "reference_sha256": _sha(Path(master["reference_path"])),\n                  "wall_seconds_including_setup": time.monotonic() - notebook_started,\n                  "submission_authorized": False, "generalization_evidence": False,\n                  "paired_off_on_all36": False, "gt_scored": False, "training_started": False}\n        _require(time.monotonic() <= deadline and output_bytes(root) + len(_json_bytes(result))\n                 <= limits["whole_output_bytes"], "final time/output budget exceeded")\n        record(root / "RESULT.json", result)\n        print(json.dumps(result, sort_keys=True), flush=True)\n        return result\n    except Exception as exc:\n        record(root / "ERROR.json", {"status": "ERROR", "error": str(exc),\n                                     "completed_groups": completed, "submission_authorized": False})\n        raise\n', 'scripts/e23_association_collect.py': '"""Run one literal collection36 group, only inside Kaggle."""\n\nimport argparse\nfrom pathlib import Path\n\nfrom biohub.association_collection import run_group\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--plan", type=Path, required=True)\n    parser.add_argument("--root", type=Path, required=True)\n    args = parser.parse_args()\n    run_group(args.plan, args.root)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/biohub/__init__.py': '# Diagnostic-only payload namespace.\n'}
for relative, text in D3_FILES.items():
    target = D3_PAYLOAD / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("x") as stream:
        stream.write(text)
sys.path.insert(0, str(D3_PAYLOAD / "src"))
from biohub.association_capture import _sha
from biohub.association_instrumentation import instrument_source
from biohub.association_collection import load_reference
patched, receipt = instrument_source(_ps.read_text())
assert receipt["original_sha256"] == '8e7ac19e8b436d6777576b7ea2269464c4ed8842990b45448d40bf178f8f62de'
assert receipt["instrumented_sha256"] == 'e2fda47e41d970a2c650bbbbb5fe068b6e1b9667fdd91cb59cc2a3ac8ba95a9c'
D3_SOURCES = {arm: REPO_DIR / "scripts" / f"e23_observer_{arm}.py" for arm in ("off", "on")}
for arm, text in (("off", _ps.read_text()), ("on", patched)):
    with D3_SOURCES[arm].open("x") as stream:
        stream.write(text)
D3_REFERENCE_PATH = WORKING_DIR / "association_collection_reference.json"
with D3_REFERENCE_PATH.open("xb") as stream:
    stream.write(b'{\n  "schema_version": "biohub.association.collection36.preparation.v1",\n  "status": "REFERENCES_PREPARED_NOT_DISPATCHED",\n  "datasets": {\n    "44b6_12dfb391": {\n      "frame_counts": [\n        438,\n        435,\n        444,\n        433,\n        430,\n        446,\n        458,\n        462,\n        449,\n        443,\n        453,\n        455,\n        449,\n        445,\n        447,\n        455,\n        445,\n        459,\n        453,\n        452,\n        455,\n        470,\n        454,\n        452,\n        468,\n        467,\n        462,\n        465,\n        476,\n        489,\n        475,\n        473,\n        486,\n        490,\n        491,\n        469,\n        480,\n        488,\n        493,\n        502,\n        513,\n        533,\n        520,\n        523,\n        515,\n        515,\n        503,\n        501,\n        504,\n        504,\n        519,\n        532,\n        516,\n        505,\n        513,\n        500,\n        486,\n        484,\n        476,\n        495,\n        487,\n        491,\n        505,\n        507,\n        500,\n        505,\n        529,\n        521,\n        519,\n        564,\n        493,\n        506,\n        516,\n        521,\n        527,\n        559,\n        534,\n        532,\n        541,\n        587,\n        569,\n        558,\n        527,\n        494,\n        521,\n        530,\n        533,\n        556,\n        579,\n        568,\n        552,\n        554,\n        571,\n        554,\n        539,\n        555,\n        546,\n        539,\n        531,\n        523\n      ],\n      "coordinate_sha256": "999928e29423851389b740449a257e2c518d0d9e45630df50caf613b2cbee977",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            41908\n          ],\n          "sha256": "fdf2b963d1a4b88e60f3ba5c6fe6b682faed7693495568f4d22c0626c66a6492"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            41908\n          ],\n          "sha256": "a1187164daafdcbdbfdc0772f108d557e40a954caacb514bcd0e13ac69077681"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            41908\n          ],\n          "sha256": "1b48ae384bebcf2018e446b927854e879906cd57115d6b363aabad3f666b46df"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            41908\n          ],\n          "sha256": "a03eea83a3ff08cbf80cb08c2ca9d11c861564e36109fc3a6ca43ac7ca381551"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            44958\n          ],\n          "sha256": "310e8dd52f8ef9645a7d01a2fa5daca7b0d179129acab793c1ed69258397a258"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            44958\n          ],\n          "sha256": "48d56248f5326deefa9086f1718dc2167c521cfce0baba140406da832b8df1c9"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            44958\n          ],\n          "sha256": "f7fd39729680bedc9ac2e00af3607b1c152912ef9a0b95a634b49bb9c2f4ca60"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            44958\n          ],\n          "sha256": "9426947caa41a2fa12930afa0d61f3e4488baf6f34d2209250f34dbad936c001"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            44958\n          ],\n          "sha256": "3540afe6a26c4fa6aee4941436ac937a3e299786719d788ff0e55711a7b15b09"\n        }\n      }\n    },\n    "44b6_267148e4": {\n      "frame_counts": [\n        189,\n        183,\n        191,\n        192,\n        204,\n        199,\n        207,\n        217,\n        216,\n        220,\n        228,\n        230,\n        232,\n        238,\n        247,\n        251,\n        252,\n        260,\n        255,\n        267,\n        276,\n        260,\n        276,\n        279,\n        287,\n        284,\n        291,\n        289,\n        287,\n        305,\n        305,\n        310,\n        293,\n        302,\n        306,\n        307,\n        313,\n        329,\n        319,\n        312,\n        321,\n        321,\n        315,\n        313,\n        308,\n        261,\n        313,\n        237,\n        328,\n        334,\n        317,\n        322,\n        314,\n        309,\n        305,\n        305,\n        314,\n        307,\n        301,\n        311,\n        288,\n        298,\n        298,\n        295,\n        296,\n        270,\n        282,\n        286,\n        265,\n        268,\n        254,\n        253,\n        256,\n        256,\n        246,\n        249,\n        261,\n        254,\n        256,\n        249,\n        241,\n        229,\n        222,\n        226,\n        213,\n        216,\n        221,\n        218,\n        222,\n        217,\n        205,\n        212,\n        197,\n        210,\n        204,\n        189,\n        178,\n        171,\n        181,\n        184\n      ],\n      "coordinate_sha256": "572e58e3e8dd7854e13a6a3970c7ea4abcb0d7a147ca7df3838c90e26ad3a7e0",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            20933\n          ],\n          "sha256": "ffce4b32246838badb265de3932accb4c213a0a5772eb5e796f365e1facc6b8e"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            20933\n          ],\n          "sha256": "4e5e7cb7f4dfd6e148e61ca9a32b21f3fb373bf05a819d56a5b325f736b73c85"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            20933\n          ],\n          "sha256": "a232fc0e0f8d44b6bf5fdb5c7f2cc1a0075432f39834292ec78df1c242211b31"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            20933\n          ],\n          "sha256": "f5f9e4110838c2bdc8392f4f284b7741f9ea9c82ba3470d4650e8cf31ce026e0"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            22577\n          ],\n          "sha256": "3f5aeaea153e81544e87d0e6d2960ec192029854a5d13af1cdb2a2d56786c34e"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            22577\n          ],\n          "sha256": "a23ddaffb38190f313a5016faa05fb74962c9c33b29fae4347d2d3f231322651"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            22577\n          ],\n          "sha256": "e17f314697bfe3e919fde3a0f09f73d6f88b1b7bdc4f41039a06866fdf48321c"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            22577\n          ],\n          "sha256": "54a6cf28da467fccf1539e3eacfacb09d00dfa39fbf02d19aca16313c483c18a"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            22577\n          ],\n          "sha256": "e2d24084bbdf1bc7dfd40c1591763c360e12d9c87296330524008304996df675"\n        }\n      }\n    },\n    "44b6_2a2eff9f": {\n      "frame_counts": [\n        555,\n        485,\n        489,\n        476,\n        490,\n        492,\n        543,\n        526,\n        522,\n        485,\n        513,\n        491,\n        464,\n        455,\n        496,\n        436,\n        474,\n        453,\n        436,\n        472,\n        500,\n        450,\n        466,\n        424,\n        467,\n        446,\n        437,\n        468,\n        410,\n        465,\n        456,\n        453,\n        459,\n        450,\n        467,\n        459,\n        446,\n        388,\n        430,\n        384,\n        422,\n        394,\n        381,\n        414,\n        437,\n        423,\n        432,\n        444,\n        416,\n        407,\n        404,\n        402,\n        401,\n        412,\n        385,\n        401,\n        416,\n        443,\n        420,\n        462,\n        430,\n        449,\n        450,\n        461,\n        461,\n        490,\n        451,\n        451,\n        449,\n        455,\n        450,\n        451,\n        470,\n        425,\n        444,\n        447,\n        449,\n        442,\n        494,\n        496,\n        466,\n        460,\n        457,\n        429,\n        456,\n        471,\n        483,\n        472,\n        441,\n        465,\n        467,\n        467,\n        460,\n        483,\n        489,\n        475,\n        462,\n        438,\n        487,\n        446\n      ],\n      "coordinate_sha256": "5a55c6728cabf8776d788cff2db2a25c0957de7614934245c028205c6f38d099",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            34322\n          ],\n          "sha256": "dfee4787a43c75e88117fd492b9ff294d29d7632d231472d19f8137c7015fda6"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            34322\n          ],\n          "sha256": "6cb735d95140339fcb66ccca82d296335df560d06e84a4b47f2af1ead71e7d9b"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            34322\n          ],\n          "sha256": "8032fd990b2920f5bc9f8f5d10078e3051d48e257718c114d20e54f878eb6ef0"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            34322\n          ],\n          "sha256": "d38b192fa2ac2fdd51fea27fed179087e706f7f8fdb6fc9f49250d308a92889d"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            37614\n          ],\n          "sha256": "3fd1f3fbba684947cb31bccf94e9c7938f477df3f0c5fbc255eabe1aa8783e20"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            37614\n          ],\n          "sha256": "9d68cbca8d287adce3a3a175e76d8a0a411633686d14634c4e66632dae6db7ac"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            37614\n          ],\n          "sha256": "6b3ed21a47fa2bb67fdf2332766efe62bb58334be400a4e1b9880a39a542809b"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            37614\n          ],\n          "sha256": "b2a6d05ff631ab0217678f8724a735c5e0f70b7fa0251a05d4911c7adb168ba5"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            37614\n          ],\n          "sha256": "a007b707075964cd2591c65864b2e00750c076fc9ad79fda68b4c427f5548a7e"\n        }\n      }\n    },\n    "44b6_341df25f": {\n      "frame_counts": [\n        105,\n        107,\n        106,\n        101,\n        106,\n        101,\n        100,\n        108,\n        110,\n        105,\n        111,\n        104,\n        102,\n        101,\n        104,\n        104,\n        102,\n        105,\n        107,\n        107,\n        108,\n        104,\n        102,\n        102,\n        94,\n        95,\n        95,\n        95,\n        98,\n        97,\n        91,\n        92,\n        88,\n        89,\n        85,\n        82,\n        88,\n        90,\n        88,\n        93,\n        91,\n        94,\n        88,\n        86,\n        91,\n        91,\n        87,\n        90,\n        86,\n        88,\n        88,\n        95,\n        89,\n        86,\n        84,\n        88,\n        87,\n        84,\n        87,\n        88,\n        88,\n        84,\n        87,\n        82,\n        81,\n        86,\n        84,\n        83,\n        86,\n        87,\n        90,\n        88,\n        88,\n        87,\n        82,\n        87,\n        83,\n        85,\n        82,\n        84,\n        80,\n        81,\n        80,\n        78,\n        80,\n        79,\n        79,\n        78,\n        79,\n        82,\n        82,\n        77,\n        78,\n        74,\n        76,\n        79,\n        77,\n        76,\n        73,\n        74\n      ],\n      "coordinate_sha256": "061647cf695f52f8482c09f1885c5620ca7331db65974dba9d5dcdcee3174842",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            8167\n          ],\n          "sha256": "2601051ecc9eb735dd26c20a428a0f7ecbbf4b95ea974614dcd6444cb0e56d2a"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            8167\n          ],\n          "sha256": "ae32a3674d0aa4585a0ca4808a02b261cbbcdebabe4b6a45e30ac3284e6c23fd"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            8167\n          ],\n          "sha256": "68e743fde8a357c2d51578d333aa6a720784f38df9a3934886bfad3cefcdc30b"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            8167\n          ],\n          "sha256": "0487ffb8261a3e88e79895026bd4178ad4a34f74495627a2bd24a9b8c6d8372c"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            8539\n          ],\n          "sha256": "31cc2181677551286284dea2cb73a90727b52fc97a7936e82b77560db3f8a156"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            8539\n          ],\n          "sha256": "17f2b94524c764f819c3373bb9b20803bef2c32c16ef527d29c9707e0893183a"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            8539\n          ],\n          "sha256": "b12030cd9ffacc0e6ef21cba409ac29fa6af2c97ce5b6df977feaee58c90bb90"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            8539\n          ],\n          "sha256": "6a0906e337fe06db60a6750d105614654022dd4c3a5aecf19142981b86a93458"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            8539\n          ],\n          "sha256": "d8a6623cd10aeec7b11292b02f1755f47615748e466d9b5821436c962f049fe1"\n        }\n      }\n    },\n    "44b6_587a1e22": {\n      "frame_counts": [\n        207,\n        206,\n        208,\n        215,\n        213,\n        214,\n        215,\n        219,\n        216,\n        222,\n        221,\n        226,\n        228,\n        228,\n        229,\n        233,\n        233,\n        236,\n        237,\n        236,\n        237,\n        238,\n        241,\n        243,\n        254,\n        257,\n        252,\n        244,\n        240,\n        240,\n        240,\n        241,\n        245,\n        235,\n        230,\n        231,\n        227,\n        237,\n        233,\n        239,\n        232,\n        233,\n        237,\n        234,\n        228,\n        232,\n        233,\n        236,\n        230,\n        228,\n        220,\n        233,\n        226,\n        229,\n        224,\n        228,\n        231,\n        224,\n        224,\n        215,\n        217,\n        210,\n        206,\n        199,\n        193,\n        193,\n        194,\n        190,\n        186,\n        181,\n        179,\n        177,\n        176,\n        170,\n        168,\n        159,\n        164,\n        157,\n        146,\n        148,\n        143,\n        137,\n        134,\n        126,\n        122,\n        125,\n        127,\n        131,\n        122,\n        125,\n        118,\n        116,\n        115,\n        117,\n        119,\n        109,\n        107,\n        106,\n        106,\n        106\n      ],\n      "coordinate_sha256": "1f3cdf9d6831ec1db7ddb940c3381c39bf9829328f39b1530600c36c1e49329f",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            18236\n          ],\n          "sha256": "1cd67faba2c3ac70f8fd10c6f0708cf0719cf1581801e8edbe6866128eb4d1a0"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            18236\n          ],\n          "sha256": "f603eec7ae28795dfef284a67a6750a3d8d3a87f9aedf35849a4d80b8dae8e09"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            18236\n          ],\n          "sha256": "dbb4712366b00726e4e9a48ba633533767990def3ad2b605915ac35438cf15d3"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            18236\n          ],\n          "sha256": "708c94bf0eff1568e9ba7436d7ac07d318f3412f5965f1492e71695ee11aa5c8"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            19096\n          ],\n          "sha256": "735e438b323016ee69ac8db2521b5f895412c0eb2bfaaa64ad1758fbd505c77e"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            19096\n          ],\n          "sha256": "71a12e3cd4dc6b51d94e7afb187b4da08469deb025e441ff603e885bc164e24a"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            19096\n          ],\n          "sha256": "4383aeb71e6ca308008155ff1dbc88eef7f6d675a1affbea930f9dca3d5e0a75"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            19096\n          ],\n          "sha256": "4571a69276287bddfadb318f5ff142805a6523b0e0b2a84802379bf23207c464"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            19096\n          ],\n          "sha256": "c1e0f3bbf0a20740cafbf974e3b4d2a61f8a6fd0476ee9dc2a8324479fd6ba1b"\n        }\n      }\n    },\n    "44b6_5f15d135": {\n      "frame_counts": [\n        420,\n        398,\n        365,\n        384,\n        348,\n        334,\n        301,\n        301,\n        312,\n        318,\n        345,\n        341,\n        333,\n        337,\n        339,\n        346,\n        339,\n        355,\n        329,\n        317,\n        313,\n        326,\n        320,\n        296,\n        282,\n        304,\n        317,\n        285,\n        302,\n        288,\n        309,\n        297,\n        268,\n        283,\n        248,\n        287,\n        263,\n        265,\n        294,\n        308,\n        343,\n        346,\n        321,\n        336,\n        319,\n        333,\n        343,\n        326,\n        323,\n        304,\n        316,\n        332,\n        341,\n        334,\n        322,\n        332,\n        295,\n        291,\n        273,\n        279,\n        308,\n        267,\n        266,\n        255,\n        251,\n        260,\n        269,\n        268,\n        256,\n        241,\n        221,\n        212,\n        231,\n        232,\n        215,\n        200,\n        190,\n        204,\n        212,\n        231,\n        221,\n        243,\n        255,\n        209,\n        187,\n        196,\n        203,\n        223,\n        164,\n        193,\n        190,\n        186,\n        173,\n        186,\n        197,\n        167,\n        180,\n        214,\n        193,\n        201\n      ],\n      "coordinate_sha256": "4d3199f25d6603394eb803cad01172aa5df0a473c1c7aba20f66284dc128b9e7",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            20139\n          ],\n          "sha256": "deee7648008fb72959e50808cde6a38f88c95ac1b68b9d5b403287f177b4fee5"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            20139\n          ],\n          "sha256": "06bf2cc95f269bff93eaa488e6475a898ab6d467add5d86a1198145f1b2be372"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            20139\n          ],\n          "sha256": "1b41af19285a3dd1ed7a49fecf40e47b424dad8d477152e26bb60aead59f8ee0"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            20139\n          ],\n          "sha256": "321d8b91c2573c6983e955d7f48cecace30fbee18bf164253c907502a40ed7d0"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            22191\n          ],\n          "sha256": "fae2de37c42260925abd755e1ad2b2a9e1d938d9ab407f4f38efb360bb333c91"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            22191\n          ],\n          "sha256": "3477aeee57203c3c8dac2425476f40641c0a407e208f41fd29ffa131bd5f7a61"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            22191\n          ],\n          "sha256": "b479fe158036e12f0ab529de12d9fb5f8232e6003d353ff8c7fedfd049e25b2d"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            22191\n          ],\n          "sha256": "729e5546d4e3e485cf614bfc0daca1648f78d484699a3c93960927339c7eec4d"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            22191\n          ],\n          "sha256": "b5fdbb1ebc4686b07a28127d887a5907c337e2e2d42ecaff3ac708904410e1fb"\n        }\n      }\n    },\n    "44b6_706092f0": {\n      "frame_counts": [\n        361,\n        351,\n        370,\n        400,\n        383,\n        364,\n        368,\n        362,\n        366,\n        353,\n        332,\n        363,\n        339,\n        346,\n        336,\n        330,\n        340,\n        351,\n        329,\n        333,\n        345,\n        344,\n        356,\n        338,\n        310,\n        333,\n        327,\n        344,\n        352,\n        362,\n        367,\n        370,\n        349,\n        346,\n        348,\n        377,\n        375,\n        398,\n        379,\n        385,\n        357,\n        360,\n        374,\n        363,\n        359,\n        386,\n        344,\n        329,\n        346,\n        342,\n        341,\n        376,\n        363,\n        359,\n        371,\n        343,\n        351,\n        357,\n        338,\n        413,\n        388,\n        393,\n        376,\n        381,\n        400,\n        376,\n        346,\n        348,\n        352,\n        344,\n        356,\n        368,\n        345,\n        364,\n        370,\n        365,\n        371,\n        381,\n        361,\n        376,\n        371,\n        368,\n        373,\n        347,\n        345,\n        345,\n        363,\n        378,\n        369,\n        362,\n        354,\n        356,\n        398,\n        419,\n        357,\n        354,\n        318,\n        410,\n        419,\n        352\n      ],\n      "coordinate_sha256": "f8d1725c8de72873666bdb4ff8701029462372f56c47f8571de5f42c64ceb74f",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            23995\n          ],\n          "sha256": "47c4fdadb9d12f63116923d67a920485ae6927276ad1adb3e7b02c4fb999a2f1"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            23995\n          ],\n          "sha256": "ad00cc1619314acf34e3ca92b864c0fb3fae0215f234161e5a014b60066a618f"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            23995\n          ],\n          "sha256": "695d67c5410292e7c1f3472242770b4e1bcab62e522f3c449e5e088cbd6bac24"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            23995\n          ],\n          "sha256": "6e676069d4fcfe32bc85acdc2c647ef0ced7e8454fcdacb59b245187e5ac3d51"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            26713\n          ],\n          "sha256": "0b6b8e9f99c46f356d207b9e6e3789cb3cea824872f8cc5b0e5b99523a5f8ee6"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            26713\n          ],\n          "sha256": "501314012a1bb48df8e101dfa1bd0daa87a01c85ca90bcd183781978c4881d73"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            26713\n          ],\n          "sha256": "f6c6bbe8bd56897ac92e8b2a438fddaea44e0fe27151178f934cd35fd9a44db1"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            26713\n          ],\n          "sha256": "4c474114abe353987fbf8b454affe220f1f5baf8c98f6b4d5b77c998a1732a4f"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            26713\n          ],\n          "sha256": "b91d380872fe0171d6e1e6117a3a77b39bdf8441e6fac55dff01d15378ca805d"\n        }\n      }\n    },\n    "44b6_74d0c52e": {\n      "frame_counts": [\n        98,\n        94,\n        94,\n        93,\n        84,\n        83,\n        86,\n        91,\n        80,\n        93,\n        96,\n        99,\n        97,\n        99,\n        95,\n        113,\n        112,\n        106,\n        104,\n        107,\n        97,\n        95,\n        95,\n        96,\n        105,\n        110,\n        128,\n        104,\n        108,\n        117,\n        97,\n        123,\n        128,\n        111,\n        107,\n        105,\n        109,\n        115,\n        119,\n        119,\n        106,\n        114,\n        110,\n        110,\n        111,\n        102,\n        111,\n        117,\n        122,\n        117,\n        117,\n        115,\n        116,\n        110,\n        114,\n        110,\n        107,\n        114,\n        113,\n        104,\n        109,\n        103,\n        110,\n        122,\n        118,\n        113,\n        108,\n        112,\n        104,\n        99,\n        103,\n        107,\n        108,\n        106,\n        105,\n        93,\n        82,\n        91,\n        87,\n        84,\n        96,\n        99,\n        101,\n        104,\n        97,\n        81,\n        81,\n        78,\n        84,\n        72,\n        80,\n        79,\n        83,\n        80,\n        83,\n        90,\n        99,\n        84,\n        90,\n        87\n      ],\n      "coordinate_sha256": "e9ba4dcb01bfa0dfc4783c7fdc8598a72445e76c5d1232fa2e35096dd91fc8b6",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            7859\n          ],\n          "sha256": "4a17699490c93f69de5c4eb53e11a4e11f7851832cb50f6889324a12af932f03"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            7859\n          ],\n          "sha256": "19357df0dffddabb5b82f9de5955401be08ed661a1bb5fa7bfa1869d5e14ce95"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            7859\n          ],\n          "sha256": "3b24818d18f8909b58dfe7399386ffc944a3d6cffee0219e6a37644f3f57abfe"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            7859\n          ],\n          "sha256": "625bb8aec29d1e06a9ef68991b5e513ed36c34dd1cc99550cc98df2d307ece7b"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            8561\n          ],\n          "sha256": "50ca9b98d5146513301caafdedb20b52b4c7920205e02377779bffca95c73902"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            8561\n          ],\n          "sha256": "faf6076addc4dc532b327a600963f3f538b0015f7e7ff00edada29a707a07aac"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            8561\n          ],\n          "sha256": "52a3a6b83b7b748c680a6e2a17c88990e37578f7946866ea6ca1a32a307f7c13"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            8561\n          ],\n          "sha256": "6fd8470f2101f0391992eac5559a961f896bd47faf787dd247ea6e9e6589ae85"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            8561\n          ],\n          "sha256": "d5dc20265672406afd5f95dae7cf08dc94a3b0d47d467c32e85f4b475f326614"\n        }\n      }\n    },\n    "44b6_7a302da0": {\n      "frame_counts": [\n        553,\n        553,\n        548,\n        550,\n        538,\n        547,\n        559,\n        578,\n        583,\n        578,\n        560,\n        569,\n        581,\n        583,\n        559,\n        557,\n        553,\n        533,\n        540,\n        567,\n        572,\n        589,\n        598,\n        633,\n        588,\n        553,\n        548,\n        517,\n        542,\n        563,\n        570,\n        556,\n        548,\n        575,\n        555,\n        570,\n        572,\n        538,\n        533,\n        556,\n        557,\n        546,\n        550,\n        513,\n        534,\n        512,\n        495,\n        512,\n        512,\n        535,\n        539,\n        540,\n        493,\n        518,\n        540,\n        556,\n        548,\n        563,\n        509,\n        492,\n        508,\n        546,\n        543,\n        495,\n        511,\n        518,\n        470,\n        505,\n        487,\n        508,\n        541,\n        560,\n        553,\n        545,\n        531,\n        529,\n        560,\n        498,\n        523,\n        522,\n        515,\n        481,\n        502,\n        497,\n        523,\n        528,\n        535,\n        494,\n        487,\n        536,\n        522,\n        551,\n        548,\n        538,\n        576,\n        572,\n        542,\n        539,\n        561,\n        595\n      ],\n      "coordinate_sha256": "91cfe94b870f38fdcb8d93c040966cef20a8eff42cd7416723c2d22ed003a169",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            40778\n          ],\n          "sha256": "4fe118a17ddd57c4cd17539fb9e011d44ccf9bdf87819a81af350ecb40b6d742"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            40778\n          ],\n          "sha256": "074de8c8b388b60f5eb5447749190dbde3895d5da60ec091111c78bab33a1ef9"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            40778\n          ],\n          "sha256": "42f4b9d827a4483d7c878aac2cfc8a214bf21f8ddbb7a6dcfb3b7aecd1311655"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            40778\n          ],\n          "sha256": "2b7f8ce82efac3466680216f8062cb36bc85347179fa0f852749665d8e7d1f9a"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            44580\n          ],\n          "sha256": "2411da7daa3cfc5603bcc3ec0066b3b7745674a47f47c01e12427ff9b8a4f79b"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            44580\n          ],\n          "sha256": "801b6bd2f8ca776b1458a553e0b3cafdf33c46d0bc2b79d651daace657e49cf1"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            44580\n          ],\n          "sha256": "1e562de5ec2401fbc061f0497b9fad1ceb29ee43e04e0e6ff4fb900193886f84"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            44580\n          ],\n          "sha256": "f580833d1c072c9ca999cd35860b4fcc5ec758960db8d35e80c35b4e22e3895b"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            44580\n          ],\n          "sha256": "688d00c496911492c44e1c333f1dac894965753ac016be40e4d2ab0383e2ee78"\n        }\n      }\n    },\n    "44b6_996155de": {\n      "frame_counts": [\n        80,\n        84,\n        90,\n        88,\n        90,\n        91,\n        89,\n        96,\n        89,\n        85,\n        89,\n        88,\n        92,\n        88,\n        87,\n        87,\n        89,\n        94,\n        90,\n        91,\n        99,\n        103,\n        98,\n        104,\n        100,\n        96,\n        89,\n        94,\n        94,\n        95,\n        106,\n        100,\n        94,\n        97,\n        99,\n        98,\n        93,\n        97,\n        99,\n        88,\n        95,\n        96,\n        95,\n        94,\n        94,\n        92,\n        91,\n        96,\n        91,\n        93,\n        95,\n        95,\n        99,\n        98,\n        96,\n        100,\n        97,\n        96,\n        108,\n        114,\n        108,\n        112,\n        116,\n        117,\n        118,\n        118,\n        115,\n        114,\n        106,\n        117,\n        118,\n        115,\n        105,\n        110,\n        104,\n        108,\n        108,\n        120,\n        115,\n        119,\n        124,\n        121,\n        125,\n        133,\n        129,\n        118,\n        123,\n        128,\n        130,\n        131,\n        125,\n        139,\n        140,\n        135,\n        139,\n        132,\n        136,\n        132,\n        130,\n        137\n      ],\n      "coordinate_sha256": "11ddab0bf7ca8a713df247a8dce4e5fb46d622329cd9f78a66985cb30851e7b4",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            8788\n          ],\n          "sha256": "84851715fd06e058b043f0129157321b073e94ae28e2649d4674673e663ade03"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            8788\n          ],\n          "sha256": "ce45cda02a17ce7383493e5475cfdb227640542c06bfe1140e2ad786241e1ffd"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            8788\n          ],\n          "sha256": "51111bc8225f29a550c7fb274cb850ee116a57a6f6467371a3c36ee060c00cfd"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            8788\n          ],\n          "sha256": "627b641239ef793ebb13d4d304eda6c75a020ea3632a99565bc7b06161d60abf"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            9309\n          ],\n          "sha256": "d3ddd8c09b29458a8af1a1373f3aa08b78e53cfa2cf7acc559b6d289391b17cb"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            9309\n          ],\n          "sha256": "70328f948823371565833f8c66af36d4c18460c45dc2bda015342e526921e7ba"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            9309\n          ],\n          "sha256": "048df50f9784ede646e84b8e73f1ed0f68366e01aaa9f81a5a275f12a6fdadf0"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            9309\n          ],\n          "sha256": "ced0110dbc50fd1d87a9058360c8e57be5fc824912ff2638c33e1bf9a1a4a4cb"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            9309\n          ],\n          "sha256": "9eb6b042eb81f04a15772a4e1294fc3eedd5c0abdfb3eed245a932ea58828c46"\n        }\n      }\n    },\n    "44b6_9be80b04": {\n      "frame_counts": [\n        357,\n        355,\n        308,\n        337,\n        365,\n        354,\n        364,\n        348,\n        345,\n        358,\n        321,\n        328,\n        362,\n        363,\n        352,\n        361,\n        343,\n        307,\n        297,\n        310,\n        300,\n        284,\n        329,\n        314,\n        290,\n        291,\n        273,\n        306,\n        291,\n        296,\n        320,\n        333,\n        345,\n        320,\n        361,\n        331,\n        336,\n        342,\n        350,\n        328,\n        357,\n        368,\n        337,\n        354,\n        348,\n        327,\n        315,\n        315,\n        315,\n        316,\n        311,\n        301,\n        294,\n        309,\n        306,\n        285,\n        275,\n        275,\n        291,\n        312,\n        299,\n        306,\n        304,\n        297,\n        288,\n        295,\n        311,\n        306,\n        291,\n        313,\n        307,\n        301,\n        311,\n        316,\n        309,\n        311,\n        296,\n        295,\n        312,\n        298,\n        300,\n        275,\n        287,\n        323,\n        299,\n        270,\n        316,\n        291,\n        298,\n        293,\n        311,\n        271,\n        314,\n        322,\n        346,\n        355,\n        311,\n        345,\n        341,\n        344\n      ],\n      "coordinate_sha256": "f6ab7012c9203b98586a89f532cf72fb5fb9ad99c13ecf2aace5982588ccf231",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            22489\n          ],\n          "sha256": "0fd0c2bf387f4d8151924b9b56dffbed7d1f7cac9b7478c4def7a867f868b9b5"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            22489\n          ],\n          "sha256": "90bde3f5dee42d87aa1f7b35a2d5678f048e849935f847bca3f2af350013aba8"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            22489\n          ],\n          "sha256": "53d43737575636c3ec0fd79d5020e2e759b13f6175795f4af2b74909cc0fd32a"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            22489\n          ],\n          "sha256": "4ba205d52358f697c9266733c2b63c802fc1f725f957fc3c95f710e4d3b2f3d5"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            24945\n          ],\n          "sha256": "d8b296c1e9653bc23d37fcbbc2ed9643516a4a3efa6fe4318f68e30122c1293e"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            24945\n          ],\n          "sha256": "52bc6b8cd1e4cd4c5f3a3c7f537acfe66340d3fbc324caab192ae8f8527cf04c"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            24945\n          ],\n          "sha256": "53dfbff257dd88c30091c3869696539b0f92d6b6e9bdeb5f6a6a59256a576372"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            24945\n          ],\n          "sha256": "5652fee7295f5f1046a2ba0be4c27c758f4c5e56c9cf3468f7c5c23fd8787911"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            24945\n          ],\n          "sha256": "8fd9797e6e1413d143ca0abef4944db5b90f827519669393850ed1515c2fada6"\n        }\n      }\n    },\n    "44b6_a21120c2": {\n      "frame_counts": [\n        237,\n        236,\n        232,\n        233,\n        235,\n        234,\n        239,\n        246,\n        245,\n        248,\n        249,\n        250,\n        243,\n        238,\n        238,\n        238,\n        240,\n        243,\n        249,\n        242,\n        246,\n        242,\n        252,\n        242,\n        244,\n        245,\n        243,\n        246,\n        252,\n        241,\n        243,\n        243,\n        253,\n        251,\n        249,\n        246,\n        240,\n        252,\n        250,\n        251,\n        260,\n        267,\n        269,\n        270,\n        270,\n        258,\n        261,\n        252,\n        256,\n        257,\n        253,\n        248,\n        249,\n        250,\n        254,\n        248,\n        248,\n        252,\n        245,\n        251,\n        257,\n        257,\n        252,\n        243,\n        243,\n        238,\n        233,\n        230,\n        224,\n        226,\n        222,\n        221,\n        217,\n        216,\n        211,\n        203,\n        200,\n        193,\n        196,\n        183,\n        183,\n        172,\n        180,\n        173,\n        175,\n        171,\n        177,\n        175,\n        167,\n        164,\n        168,\n        172,\n        171,\n        171,\n        161,\n        162,\n        163,\n        163,\n        160,\n        152\n      ],\n      "coordinate_sha256": "b080488f9151d51c1a0e57baa12a77bcbef253da84b241a51688dbda45a3b0c8",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            21255\n          ],\n          "sha256": "e544370629c584ebf156de0177b848b4d39fbe8ffe491112f33cf23dfa4f6c39"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            21255\n          ],\n          "sha256": "6cf35d78c0586e6edc019d214ec9f1889f69c1c2b4f03abf39ff4cff4c4b7d4f"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            21255\n          ],\n          "sha256": "34a827cc9653d679ce6f6f9a27cc247dd5e8597724598ffa1799752b4f0c8b9a"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            21255\n          ],\n          "sha256": "6813d31867d4a4fe5f703cb7efb25ddfe4411bb3aa299ed0d97e838de2e77960"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            22136\n          ],\n          "sha256": "40018cfc26f75ac45e89482293905d0b499e63abe3050b36a43aa06c545df0fc"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            22136\n          ],\n          "sha256": "5826490ea2e42df9133b9c071467f3595cbced75eb4bcf1d51d085efddef5ec2"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            22136\n          ],\n          "sha256": "1ebb2bac5bfaeda859c17049866c65ed3a0aabaf5d76f9857d2510f2abdb8378"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            22136\n          ],\n          "sha256": "a3916a50e25277d5f2ee6dee9e8b23d7c9eb70f3f72be924c5e16877312b327b"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            22136\n          ],\n          "sha256": "7fee0ec28b95518a076871212002277ce1c30c329af6620a2246e3b792f7a573"\n        }\n      }\n    },\n    "44b6_aaf8b0ea": {\n      "frame_counts": [\n        184,\n        190,\n        199,\n        192,\n        179,\n        170,\n        184,\n        182,\n        194,\n        181,\n        173,\n        163,\n        173,\n        160,\n        161,\n        172,\n        180,\n        187,\n        176,\n        182,\n        174,\n        174,\n        183,\n        183,\n        193,\n        173,\n        184,\n        194,\n        188,\n        176,\n        179,\n        193,\n        166,\n        165,\n        177,\n        183,\n        171,\n        173,\n        180,\n        178,\n        184,\n        171,\n        183,\n        182,\n        192,\n        195,\n        174,\n        181,\n        184,\n        183,\n        180,\n        162,\n        159,\n        167,\n        178,\n        190,\n        171,\n        182,\n        187,\n        184,\n        160,\n        173,\n        182,\n        165,\n        178,\n        177,\n        179,\n        181,\n        205,\n        204,\n        193,\n        181,\n        177,\n        169,\n        173,\n        185,\n        185,\n        192,\n        186,\n        188,\n        182,\n        194,\n        212,\n        190,\n        202,\n        199,\n        188,\n        171,\n        164,\n        194,\n        194,\n        197,\n        187,\n        187,\n        194,\n        196,\n        200,\n        193,\n        189,\n        214\n      ],\n      "coordinate_sha256": "e17d2ec65008dd1d5623ed96713a0e694a1dd3580bce3edd7cbef02e82281b25",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            13862\n          ],\n          "sha256": "8562786e3dae783902aaf7fc81712052b879b41ae09b51f36a46d902b8bdd74a"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            13862\n          ],\n          "sha256": "5124d0e6f8caf43ad6cbdaabd8c0ab13a0d3a44af3296582204eb1bafec476a2"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            13862\n          ],\n          "sha256": "57d8dffca9c8d2e89bca701dec1294e9876995b64e6e4cc9d68f27b34476726a"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            13862\n          ],\n          "sha256": "30f5521955bd934b0dfca2032734e88285a81566690b7ce6c35d0ec993ed3434"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            15054\n          ],\n          "sha256": "4da386a72593f0465f9e47206a8ff8af2a771a8fbbfd0bae2c84ef4934068683"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            15054\n          ],\n          "sha256": "1cc2e4e02a7c7f429ee38dd2b3fa4cb0f0b12b07879755d22e2ec620fdf1406d"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            15054\n          ],\n          "sha256": "5e6d185ecddfaf3704362faec8e791b97383ab2541f94dbb184443ecacd3894b"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            15054\n          ],\n          "sha256": "2e85e796962b9854467e1d0c46ffa1a4bfb2ec625cd56e58cbf305526338f462"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            15054\n          ],\n          "sha256": "5abda2e2ccea675ac23a877f2531e5b99e3c91e8b05be5daa7baadd87d1ac657"\n        }\n      }\n    },\n    "44b6_c50204e0": {\n      "frame_counts": [\n        266,\n        316,\n        317,\n        319,\n        308,\n        350,\n        390,\n        394,\n        384,\n        397,\n        408,\n        409,\n        406,\n        390,\n        384,\n        394,\n        394,\n        415,\n        406,\n        428,\n        438,\n        440,\n        437,\n        441,\n        465,\n        456,\n        459,\n        473,\n        490,\n        512,\n        500,\n        542,\n        549,\n        530,\n        551,\n        521,\n        536,\n        517,\n        492,\n        562,\n        563,\n        561,\n        567,\n        580,\n        573,\n        587,\n        592,\n        624,\n        613,\n        586,\n        601,\n        553,\n        529,\n        579,\n        570,\n        548,\n        552,\n        588,\n        610,\n        550,\n        509,\n        545,\n        564,\n        559,\n        543,\n        555,\n        552,\n        548,\n        522,\n        529,\n        504,\n        507,\n        505,\n        536,\n        543,\n        546,\n        555,\n        562,\n        524,\n        524,\n        533,\n        543,\n        557,\n        551,\n        529,\n        543,\n        517,\n        543,\n        525,\n        542,\n        544,\n        552,\n        503,\n        519,\n        546,\n        456,\n        476,\n        546,\n        533,\n        526\n      ],\n      "coordinate_sha256": "bf110a6256f4334ab325576fb187e96505a35c314dc2b99f5fc8c4138b9cfe49",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            35575\n          ],\n          "sha256": "8db1fcc4693be2f16692945af4aab12d64310eaaee81d285ec7185421f14713a"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            35575\n          ],\n          "sha256": "8f1988f133fdfbc941aa540105abd781555b26470ab879f22ea9502692fba524"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            35575\n          ],\n          "sha256": "44454cb9705d7addd2d528ffcb63f081540068341fdfd9261dcb1d15042bfef0"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            35575\n          ],\n          "sha256": "86982e0890865024cecb990af5183532520d5159054dd7350bb57c867ced7d74"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            39305\n          ],\n          "sha256": "a8c2a4ca76a1e90f2acbd3c109a9979359970016ea0c7306ad47eae58d77863a"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            39305\n          ],\n          "sha256": "27a99917713ab0e482e365db69859671d7c9541b138d8ef3e81dcc0b4e3f9e79"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            39305\n          ],\n          "sha256": "b4a5e9bd314127c1fdcc2934d4a22a3467d92d349b4390c82c3ef81f750cb77d"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            39305\n          ],\n          "sha256": "5e70efdc5dc60d6d59437a14d578e6e9f7048f2d149386d03bce41ffb8212ccd"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            39305\n          ],\n          "sha256": "191dce67f92d323497a3d1f77841c221080efd738e7111a552a99ed42b64eec2"\n        }\n      }\n    },\n    "44b6_c8e2a523": {\n      "frame_counts": [\n        250,\n        249,\n        245,\n        249,\n        256,\n        246,\n        280,\n        273,\n        275,\n        271,\n        268,\n        270,\n        276,\n        275,\n        283,\n        286,\n        288,\n        286,\n        286,\n        284,\n        279,\n        284,\n        283,\n        292,\n        290,\n        289,\n        289,\n        295,\n        297,\n        300,\n        299,\n        291,\n        294,\n        289,\n        294,\n        301,\n        301,\n        306,\n        299,\n        300,\n        299,\n        304,\n        300,\n        308,\n        311,\n        313,\n        311,\n        305,\n        298,\n        300,\n        318,\n        310,\n        329,\n        310,\n        312,\n        305,\n        305,\n        309,\n        315,\n        309,\n        310,\n        318,\n        327,\n        330,\n        319,\n        325,\n        319,\n        328,\n        324,\n        332,\n        328,\n        338,\n        334,\n        339,\n        339,\n        343,\n        344,\n        332,\n        327,\n        331,\n        319,\n        328,\n        321,\n        310,\n        314,\n        315,\n        306,\n        296,\n        308,\n        299,\n        297,\n        302,\n        299,\n        297,\n        293,\n        292,\n        290,\n        291,\n        284,\n        291\n      ],\n      "coordinate_sha256": "4810024bf05234d3676628071dd664100a35e3a14b581f676db76b107f3757ff",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            27481\n          ],\n          "sha256": "0110d80d831ee0873597467ed2adab2729fe85f39bc3d7c6e2697237f2c735c2"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            27481\n          ],\n          "sha256": "ac8fe8c184a0d5efc06e3718787e43ca309c2f48ddb27c43cfc7b30329dbb6df"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            27481\n          ],\n          "sha256": "eb509f6ffa4cb7aad7ed3bac12448728815c384f356a55bbfccfb7817bbfaa4c"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            27481\n          ],\n          "sha256": "45e31d597bd6194d356f666b74caa6a4e0fe1cd873cdccf577c71b3e2e358ddb"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            28862\n          ],\n          "sha256": "bd59db4281d45f8511c2b3ade809e611418c2316aaadc7aac81f8280934c9db8"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            28862\n          ],\n          "sha256": "91c15147ed7f2c6d95b8ae45e03e66f76279a9018e0ab2dd281f5d5cc9c6503e"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            28862\n          ],\n          "sha256": "c420ea4f2b8f1bec9834f4e58034afe00b23dbff70845ca2bc6bd1165e9b1f6b"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            28862\n          ],\n          "sha256": "88e8a3803d04f9c9575c34e0e4ecc974188975de48c06498708eb0f93b5cff29"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            28862\n          ],\n          "sha256": "1ba6d898577f4118d7700fa342c3797e89ca56983d09e55c4b794d1e0275a831"\n        }\n      }\n    },\n    "44b6_d2f34f90": {\n      "frame_counts": [\n        195,\n        183,\n        162,\n        162,\n        168,\n        155,\n        173,\n        151,\n        152,\n        142,\n        168,\n        166,\n        183,\n        166,\n        161,\n        171,\n        174,\n        183,\n        211,\n        215,\n        193,\n        187,\n        184,\n        179,\n        183,\n        206,\n        227,\n        222,\n        218,\n        231,\n        225,\n        198,\n        195,\n        193,\n        177,\n        181,\n        205,\n        196,\n        182,\n        187,\n        169,\n        188,\n        205,\n        209,\n        207,\n        202,\n        210,\n        189,\n        174,\n        179,\n        194,\n        175,\n        197,\n        191,\n        200,\n        177,\n        195,\n        226,\n        230,\n        246,\n        206,\n        200,\n        211,\n        211,\n        224,\n        249,\n        236,\n        226,\n        222,\n        210,\n        203,\n        185,\n        210,\n        186,\n        204,\n        182,\n        183,\n        171,\n        187,\n        159,\n        187,\n        197,\n        191,\n        149,\n        153,\n        173,\n        196,\n        181,\n        171,\n        192,\n        198,\n        181,\n        173,\n        186,\n        171,\n        178,\n        172,\n        160,\n        166,\n        174\n      ],\n      "coordinate_sha256": "58f94c125a0a248b17f1f344e12dd8b732b1dcde9286d6f2f66fee14721c8d1c",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            13805\n          ],\n          "sha256": "dce84791bc33d172273a69dfc8f03e4f954033f88b162bcb06adc17fa0bdb39d"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            13805\n          ],\n          "sha256": "7cee37ef08f4f4c8b3e88a22d00bdf9f4b155c4c653bf3537797f62195d7ff55"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            13805\n          ],\n          "sha256": "5b75da33085dc70b5a70745b485ea60696ea0d2beb2525632d4dec8e90f7f67d"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            13805\n          ],\n          "sha256": "7ac9df3794bd36742e76fd371dc4db74c4573045b383934cba1aef19212f2764"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            15137\n          ],\n          "sha256": "f131f17c04f38d110242c8c4d2dc70b626fe002713065e3f51504417e7a8b5a9"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            15137\n          ],\n          "sha256": "5fffcd8fd974f63de3bedad13d8edcfb12c87eca23b2f800af3141eb9b36e222"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            15137\n          ],\n          "sha256": "65e1be66056185c49d505e70fcffa0c8dadee962d48cdcb9f30143e54a6f9f88"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            15137\n          ],\n          "sha256": "1eaa390218de34dcb34b2506e35de8becef9cab29660ac48fe2e7f4e99254be0"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            15137\n          ],\n          "sha256": "4a2dd94d53e66a921a058d6e76261afeb837342a9445811392cc03093a7eeecc"\n        }\n      }\n    },\n    "44b6_d5e7d891": {\n      "frame_counts": [\n        321,\n        328,\n        344,\n        348,\n        356,\n        354,\n        355,\n        357,\n        367,\n        371,\n        361,\n        365,\n        390,\n        393,\n        402,\n        390,\n        384,\n        415,\n        429,\n        437,\n        436,\n        441,\n        430,\n        441,\n        452,\n        447,\n        428,\n        434,\n        452,\n        474,\n        475,\n        488,\n        494,\n        498,\n        504,\n        489,\n        505,\n        478,\n        495,\n        491,\n        501,\n        501,\n        517,\n        532,\n        520,\n        518,\n        530,\n        536,\n        532,\n        543,\n        539,\n        539,\n        545,\n        548,\n        509,\n        520,\n        532,\n        545,\n        557,\n        556,\n        534,\n        548,\n        550,\n        578,\n        591,\n        574,\n        579,\n        611,\n        578,\n        593,\n        598,\n        592,\n        578,\n        645,\n        631,\n        650,\n        631,\n        648,\n        658,\n        688,\n        727,\n        683,\n        654,\n        652,\n        567,\n        585,\n        622,\n        616,\n        606,\n        641,\n        611,\n        605,\n        634,\n        677,\n        671,\n        665,\n        670,\n        694,\n        678,\n        652\n      ],\n      "coordinate_sha256": "2e00710f41f263412abd9e28b44d24f1e2cbefdd5bc141bf0f189dfeec1921b7",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            41401\n          ],\n          "sha256": "ecba9c6749538e2ef1d2d4c4b2036b42b9f1856d3e2ce9839c67b0200a8827b1"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            41401\n          ],\n          "sha256": "fcf7e3201ee9f9b224a00c94688825dd2439ed4069f26eea565ef7bf0b74d8bf"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            41401\n          ],\n          "sha256": "e71689203cfe64b5dca59c6bf15ad48f4759ca1755820332841eb1b574414bd7"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            41401\n          ],\n          "sha256": "d26da3183ae5039f009576a6425f2baec5183ad828b525577b9e6e8fb685fbd7"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            44891\n          ],\n          "sha256": "9ad1e7c9b21a1b2babf17a600130da6e0e7ecc68f298018279a5876b526da241"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            44891\n          ],\n          "sha256": "56abee6f31bb4bae2114877cd13a6777809afc4ed591dc85d916717926a0ef5c"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            44891\n          ],\n          "sha256": "b2f39a3b39e57390d0595ddc038ac5c703549e12a6704e7baafaa1c316e76e4f"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            44891\n          ],\n          "sha256": "594d331dc4b8d6ca9a5eb825a49ffcffde2617b17e274b163e39ee0035e19111"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            44891\n          ],\n          "sha256": "e467de86cb3470bc8470458ec78a07de1130ee112a104da3d1c82f0b97650340"\n        }\n      }\n    },\n    "44b6_d754aa59": {\n      "frame_counts": [\n        68,\n        63,\n        62,\n        62,\n        67,\n        68,\n        61,\n        63,\n        60,\n        62,\n        62,\n        66,\n        62,\n        62,\n        66,\n        65,\n        62,\n        62,\n        68,\n        59,\n        63,\n        64,\n        59,\n        56,\n        54,\n        54,\n        54,\n        57,\n        55,\n        57,\n        58,\n        57,\n        54,\n        60,\n        58,\n        55,\n        54,\n        54,\n        57,\n        57,\n        59,\n        57,\n        56,\n        56,\n        59,\n        55,\n        57,\n        53,\n        56,\n        55,\n        54,\n        55,\n        54,\n        65,\n        52,\n        56,\n        65,\n        61,\n        56,\n        56,\n        62,\n        58,\n        62,\n        62,\n        58,\n        63,\n        67,\n        70,\n        65,\n        62,\n        62,\n        64,\n        65,\n        64,\n        67,\n        69,\n        65,\n        65,\n        62,\n        64,\n        66,\n        66,\n        77,\n        74,\n        74,\n        71,\n        81,\n        77,\n        82,\n        77,\n        78,\n        77,\n        73,\n        74,\n        67,\n        67,\n        72,\n        72,\n        72,\n        73\n      ],\n      "coordinate_sha256": "7104780ca4463f0b48c06f37f8dcfb831bc3c99767c51dcf07fff95330de130e",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            5424\n          ],\n          "sha256": "a8af430990c53d3913f40e73346d6164c59f0dbc6db76c34f5f4ec3b1fd2ebf4"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            5424\n          ],\n          "sha256": "85e17a8c95ecbb1fcb0ae8ddc129555e0866031ad76c59d5378390abaf41903d"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            5424\n          ],\n          "sha256": "575d500930ee12c35417067cccda1cc3d7a65d06516ed0f87139a8d8c8226735"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            5424\n          ],\n          "sha256": "fe68d51e8a8824f1931999ac31d58aa3638a32ac718f44b42502f9af3eaf7c1f"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            5699\n          ],\n          "sha256": "35c609e8c88f4e2b7755e9845911ffb73763ef0158f0c8dbb6c454f8051b1f71"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            5699\n          ],\n          "sha256": "ab7c9f297cc35c140d619086ab22d669718afa4a551029685751ccc02b607591"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            5699\n          ],\n          "sha256": "72eae05168fb9f63409652d0c3d0f7de9ee08600e7fa503fa827b5d55aac55b9"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            5699\n          ],\n          "sha256": "f34c6c8700c101acaf08a8a08f0137494e846d07691fa44c72726639dd862788"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            5699\n          ],\n          "sha256": "4068a6ff335749887bc147d3f90f27e560da8f1d817f05ec948fbcca8ccdccd5"\n        }\n      }\n    },\n    "6bba_062c8d37": {\n      "frame_counts": [\n        89,\n        88,\n        81,\n        74,\n        72,\n        73,\n        73,\n        75,\n        76,\n        71,\n        67,\n        65,\n        67,\n        66,\n        64,\n        64,\n        64,\n        67,\n        67,\n        68,\n        68,\n        64,\n        63,\n        62,\n        60,\n        63,\n        64,\n        62,\n        61,\n        60,\n        56,\n        60,\n        62,\n        61,\n        68,\n        66,\n        72,\n        67,\n        67,\n        67,\n        113,\n        64,\n        58,\n        57,\n        64,\n        62,\n        63,\n        64,\n        59,\n        60,\n        63,\n        61,\n        61,\n        64,\n        65,\n        65,\n        63,\n        60,\n        67,\n        63,\n        64,\n        61,\n        66,\n        69,\n        65,\n        65,\n        67,\n        66,\n        70,\n        68,\n        64,\n        62,\n        65,\n        67,\n        64,\n        64,\n        69,\n        65,\n        65,\n        63,\n        61,\n        62,\n        59,\n        62,\n        66,\n        68,\n        66,\n        66,\n        63,\n        59,\n        62,\n        61,\n        59,\n        60,\n        61,\n        62,\n        57,\n        63,\n        62,\n        68\n      ],\n      "coordinate_sha256": "009a798919f8a0461977798024dde40b7e8acd9be89a360e847fa129e4b5811e",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            5698\n          ],\n          "sha256": "e679850b45916972d481e8807f89e9e1cd5accfc6442db5a62def0819ff4c7a6"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            5698\n          ],\n          "sha256": "6d16c64f960ea0cacabf15719e1c8c1235b389c0f9b702eb5fd76f1858cff69d"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            5698\n          ],\n          "sha256": "f2761f2f6b69beccf57a250542ac6ff8692e76ce7d2233f95842bc146375b2f9"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            5698\n          ],\n          "sha256": "6e4627ed58e23ae20025507e64c4e1dc3cc06d2a012e7e9965833186a4fede90"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            6050\n          ],\n          "sha256": "1c18299c846efb2cbfec24957205237335b2781acc027243041f1547e0357cf4"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            6050\n          ],\n          "sha256": "b5da974bd7116a2d86955b9f4fc2d8ed8418c0d44441c84be34ad5192ddd173e"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            6050\n          ],\n          "sha256": "83959d8f4a605974da8592ece800089dc4bd269ae182b4ea3c0d692e2666e21c"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            6050\n          ],\n          "sha256": "be35083165f7e52f92bad75539f67079cf8e7e794fbcfb0cb5592f38c9fce6ab"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            6050\n          ],\n          "sha256": "f1fc51db0d426ccef5e1ae726fdedc06e84b309fc74690245e340775fd4e282e"\n        }\n      }\n    },\n    "6bba_07e24132": {\n      "frame_counts": [\n        421,\n        414,\n        443,\n        419,\n        403,\n        417,\n        406,\n        408,\n        403,\n        428,\n        428,\n        428,\n        419,\n        411,\n        407,\n        395,\n        393,\n        367,\n        394,\n        384,\n        379,\n        425,\n        397,\n        349,\n        363,\n        337,\n        379,\n        373,\n        359,\n        385,\n        380,\n        353,\n        351,\n        342,\n        355,\n        347,\n        370,\n        377,\n        370,\n        368,\n        347,\n        372,\n        341,\n        322,\n        309,\n        308,\n        308,\n        296,\n        291,\n        310,\n        351,\n        331,\n        328,\n        324,\n        371,\n        340,\n        382,\n        346,\n        344,\n        347,\n        351,\n        325,\n        325,\n        313,\n        306,\n        292,\n        337,\n        322,\n        327,\n        322,\n        311,\n        332,\n        301,\n        375,\n        350,\n        333,\n        418,\n        433,\n        375,\n        353,\n        384,\n        340,\n        367,\n        390,\n        365,\n        330,\n        342,\n        322,\n        325,\n        322,\n        347,\n        325,\n        301,\n        315,\n        329,\n        321,\n        322,\n        304,\n        319,\n        316\n      ],\n      "coordinate_sha256": "371ff9f27fa42187ae4d5ca0f267d0c8bc5e8f0d71f2a782a891073f7eb25f45",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            25237\n          ],\n          "sha256": "b6811d1c19abf4679818c3cd02afa83963b39767b0ffce3baca7032971c7c5a7"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            25237\n          ],\n          "sha256": "3631bdd9fe676280f56e7af70986a387a0f54c39bb55c2e1026b2a9b3a477421"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            25237\n          ],\n          "sha256": "4351c1f8a46095a34f45a12b28b67a80e45e917da93ae2484e1538fcfb2adc3c"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            25237\n          ],\n          "sha256": "dff7ec7fe6ba3ff6835c17c73c89a2dc7491f4edd1d4427f1d2271f784d8c7ed"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            27399\n          ],\n          "sha256": "64f8a3a25e48a0347806882fab2ff594a2835ce31c7a5f5b45b863d8130f338c"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            27399\n          ],\n          "sha256": "957c2d95b5b2f2ac0ac85ff072fc33e21cc031bc4528ea7ef1fc8b32acccca10"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            27399\n          ],\n          "sha256": "0fbebc2cdc1cc902d27af26347659b88c49d4a485f7bbd50e78dbb4aa2422672"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            27399\n          ],\n          "sha256": "3578b485a106d82b898ea176c688dc487ca85f1e77b6994b19374728c921d7ef"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            27399\n          ],\n          "sha256": "7b9b9bbb728a95e508203f2083509a525581cfd08a1efa438d79757d98d0f74b"\n        }\n      }\n    },\n    "6bba_085bf656": {\n      "frame_counts": [\n        153,\n        147,\n        143,\n        137,\n        138,\n        136,\n        128,\n        130,\n        129,\n        129,\n        125,\n        127,\n        127,\n        129,\n        135,\n        130,\n        133,\n        130,\n        128,\n        121,\n        112,\n        114,\n        112,\n        112,\n        108,\n        108,\n        110,\n        105,\n        106,\n        104,\n        104,\n        107,\n        105,\n        102,\n        103,\n        99,\n        99,\n        95,\n        94,\n        93,\n        100,\n        89,\n        92,\n        91,\n        90,\n        87,\n        89,\n        89,\n        92,\n        89,\n        92,\n        87,\n        84,\n        84,\n        82,\n        80,\n        84,\n        83,\n        82,\n        79,\n        82,\n        83,\n        83,\n        79,\n        74,\n        72,\n        73,\n        73,\n        73,\n        73,\n        71,\n        71,\n        70,\n        68,\n        66,\n        65,\n        63,\n        60,\n        62,\n        60,\n        64,\n        62,\n        62,\n        65,\n        64,\n        64,\n        64,\n        63,\n        57,\n        57,\n        55,\n        54,\n        54,\n        58,\n        56,\n        54,\n        55,\n        52,\n        52,\n        52\n      ],\n      "coordinate_sha256": "cb746df8679fb0cb282a6964fc03324e2cb85b36237fa5a1716ad8499b12c849",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            8162\n          ],\n          "sha256": "affa8677b0cf002011e4b00aba8b70694db5dc9b086ccd7035592854c43dc9ee"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            8162\n          ],\n          "sha256": "bc1e0563bdc5acb623b444c5d072c1658ae7a76a535fa4ae1953549ca31ca856"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            8162\n          ],\n          "sha256": "e4d38ece615ac3735196d42607c1410ca8f5a7ed8aa9e3393a964ad2f9c21e57"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            8162\n          ],\n          "sha256": "bd773ab12698b2365e65422c25bbeeffe4ab48fd543218dc9b330396ca41bc93"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            8505\n          ],\n          "sha256": "356e7497ed88d0d2de4cd3ab179403c1d03bbf110fc1efb9c0c6df2091052755"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            8505\n          ],\n          "sha256": "726a89fe0e6cd56a9f0d5a48b064c437e2aff92be21eecc64e1e32ff23cf637a"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            8505\n          ],\n          "sha256": "827810c44df2c4fddb5552350246c482f9c89f2e88686c0ad32bed4cb2377b8e"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            8505\n          ],\n          "sha256": "e269c8f07ceb2f39dbed057d3a83609baab3401202e73fa9e7a7ccbb44345578"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            8505\n          ],\n          "sha256": "fee389f0cc459636c764ddca5130db84fe9ba9d230b71357e8fa08edd8c2f5b2"\n        }\n      }\n    },\n    "6bba_09961292": {\n      "frame_counts": [\n        359,\n        361,\n        356,\n        361,\n        359,\n        368,\n        362,\n        361,\n        345,\n        355,\n        361,\n        351,\n        343,\n        335,\n        334,\n        342,\n        336,\n        327,\n        330,\n        330,\n        327,\n        331,\n        331,\n        329,\n        322,\n        316,\n        324,\n        321,\n        312,\n        308,\n        304,\n        311,\n        307,\n        316,\n        318,\n        318,\n        310,\n        308,\n        315,\n        245,\n        294,\n        293,\n        303,\n        298,\n        310,\n        306,\n        314,\n        318,\n        311,\n        304,\n        304,\n        311,\n        307,\n        294,\n        307,\n        315,\n        314,\n        319,\n        314,\n        317,\n        318,\n        310,\n        318,\n        318,\n        324,\n        321,\n        312,\n        319,\n        306,\n        309,\n        315,\n        310,\n        310,\n        320,\n        325,\n        324,\n        305,\n        307,\n        299,\n        302,\n        298,\n        300,\n        294,\n        300,\n        309,\n        308,\n        297,\n        299,\n        299,\n        297,\n        312,\n        307,\n        301,\n        266,\n        305,\n        304,\n        311,\n        313,\n        301,\n        294\n      ],\n      "coordinate_sha256": "e3b14e55a82d762cd29ab7cb89ac69f0db7f049ec62fcce8f7601c2ed42df8c1",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            28597\n          ],\n          "sha256": "8ec84918955e67923b5cd9f3d4bdc06a651fb72744b000a3b80b8de2ef85164a"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            28597\n          ],\n          "sha256": "8a5f2d0220acaa962bbdcd052f4821bef86722c8b7f6c4ad697e3e43889ecc33"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            28597\n          ],\n          "sha256": "4a212432e09e5149dadce9d530d78efded4eeb51ae03cccf49cffa719c329b7d"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            28597\n          ],\n          "sha256": "1a68ed6fef93798bd9aeae08dd0eca97618ff1a473019e513a390012b201e471"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            29985\n          ],\n          "sha256": "dde125b5c518a89d103c4e0c142025c9b78d0905a570020c7e5efb22f98488d3"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            29985\n          ],\n          "sha256": "7c1c0e04e8772cc2e451168411656b9852e004e06714caadce1bc747fe61b7d8"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            29985\n          ],\n          "sha256": "84713f2c1f499b5179667e65214327b3dd29014527c991c1d4235c2c55e313fc"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            29985\n          ],\n          "sha256": "cd66e5bf90030b862b22c01ac4d5165d429bf1a56894ebb3cf5d1246e6493a11"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            29985\n          ],\n          "sha256": "9cce16c1665abed3b51e854e05619266eb536aa254db3e3a455058fabe4c083c"\n        }\n      }\n    },\n    "6bba_0e7c0d07": {\n      "frame_counts": [\n        28,\n        30,\n        29,\n        30,\n        28,\n        29,\n        30,\n        28,\n        31,\n        30,\n        30,\n        31,\n        34,\n        33,\n        32,\n        36,\n        37,\n        36,\n        37,\n        36,\n        39,\n        40,\n        40,\n        51,\n        49,\n        52,\n        55,\n        61,\n        60,\n        107,\n        109,\n        117,\n        122,\n        135,\n        138,\n        157,\n        171,\n        188,\n        201,\n        253,\n        256,\n        274,\n        286,\n        300,\n        309,\n        323,\n        342,\n        342,\n        375,\n        376,\n        382,\n        404,\n        426,\n        444,\n        446,\n        462,\n        470,\n        467,\n        478,\n        484,\n        494,\n        508,\n        525,\n        523,\n        531,\n        530,\n        533,\n        524,\n        530,\n        525,\n        517,\n        515,\n        509,\n        495,\n        484,\n        441,\n        462,\n        448,\n        442,\n        431,\n        416,\n        409,\n        399,\n        398,\n        384,\n        387,\n        324,\n        324,\n        314,\n        292,\n        307,\n        297,\n        295,\n        265,\n        266,\n        271,\n        252,\n        245,\n        234,\n        228\n      ],\n      "coordinate_sha256": "c95635e8cc0d0bc1d6c76336afabcd392c6a3d637a197082ccef39c8792e343e",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            21686\n          ],\n          "sha256": "39464a72fb43d76c72be27a8aa658f6262de213ce69a275430ccc73b8ae9cf71"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            21686\n          ],\n          "sha256": "0ef516e3779fcf62b193c26737d06a5afbec7a0730c7aec3b04f838b1d7c551a"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            21686\n          ],\n          "sha256": "9101f71a0ec38b273b54e6cbe151c0b3b6ed2f266cfb70bff7f9309ea6c73250"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            21686\n          ],\n          "sha256": "abf48f822ce2954e73f7348990e60db0849e707bca03bfab32c017db8b31c312"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            23616\n          ],\n          "sha256": "7472c570312b9513758d526d4882e4c484bed8123a6d6b7faf355108b9daeabb"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            23616\n          ],\n          "sha256": "ef783b193bef235ea9ef506035ed03d833efac08c63a08534126094580c90a4d"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            23616\n          ],\n          "sha256": "179238141b748e98a392c7877e656981b943eed0a53483209257e2c92260c61c"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            23616\n          ],\n          "sha256": "9954f97bd646afeebfacbcf10ce9c4b43291bf5361461d9ea3bdc0562fab5640"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            23616\n          ],\n          "sha256": "1c8dbbc03351130b31ec039dd2c06bb82087c76002322e1fca55ade873a678cd"\n        }\n      }\n    },\n    "6bba_12665c0e": {\n      "frame_counts": [\n        177,\n        174,\n        164,\n        172,\n        171,\n        168,\n        165,\n        156,\n        151,\n        152,\n        152,\n        144,\n        142,\n        140,\n        143,\n        139,\n        137,\n        140,\n        126,\n        132,\n        129,\n        122,\n        120,\n        118,\n        113,\n        116,\n        116,\n        116,\n        117,\n        112,\n        111,\n        105,\n        106,\n        100,\n        95,\n        95,\n        91,\n        88,\n        83,\n        85,\n        87,\n        87,\n        76,\n        71,\n        72,\n        69,\n        68,\n        67,\n        62,\n        70,\n        64,\n        68,\n        71,\n        62,\n        61,\n        63,\n        61,\n        60,\n        58,\n        54,\n        54,\n        60,\n        61,\n        58,\n        51,\n        50,\n        49,\n        49,\n        49,\n        49,\n        59,\n        57,\n        59,\n        58,\n        58,\n        54,\n        55,\n        55,\n        59,\n        54,\n        54,\n        55,\n        53,\n        51,\n        50,\n        48,\n        53,\n        53,\n        58,\n        53,\n        53,\n        61,\n        56,\n        59,\n        58,\n        57,\n        57,\n        56,\n        56,\n        58\n      ],\n      "coordinate_sha256": "e612913d6763d2738e43e5893a6bfd4249edf2e30707f6f68286b4e0ab4702bf",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            7799\n          ],\n          "sha256": "94bcbf1ccab20f7fe04c385997ee7193c5fb390cf7c7cc76f44abe457c29f01b"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            7799\n          ],\n          "sha256": "cc32ea50a68f00cbb0d6624752da923ef9eb7666fa5efd5380299717ada87535"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            7799\n          ],\n          "sha256": "3cad94492c63fa48a20ceea1017ef0363e09878601d03275e21f8d08d79eed0d"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            7799\n          ],\n          "sha256": "695cbea2550a4702ffcad89fe32b9dba234f42267f2811f770605444ce62d256"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            8202\n          ],\n          "sha256": "1a829195558d59d582d7a8be8cb2f75334fb49a48f538714a3fa3be7d932b207"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            8202\n          ],\n          "sha256": "477275292f27384a6929fbdf418797d10b6b3da5db5d0b0c89aa63696d9f8cfb"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            8202\n          ],\n          "sha256": "806ebed5690ad9b963f0fadd5e58b7defcb4e12c9e4aea5a84c047501f2b5277"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            8202\n          ],\n          "sha256": "a073de4a719a106f5875ef7eea929d16a65213a9c0c6a73abaf24d5fc3c0645b"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            8202\n          ],\n          "sha256": "908c8312e86dac9902b91d2391be2e4ccfa065c0200f90639e0c6a9e4db00bdd"\n        }\n      }\n    },\n    "6bba_1d0d8384": {\n      "frame_counts": [\n        318,\n        317,\n        321,\n        318,\n        321,\n        334,\n        324,\n        312,\n        320,\n        314,\n        319,\n        317,\n        323,\n        305,\n        300,\n        304,\n        307,\n        315,\n        312,\n        308,\n        298,\n        280,\n        278,\n        272,\n        278,\n        272,\n        270,\n        261,\n        264,\n        259,\n        260,\n        263,\n        269,\n        254,\n        256,\n        235,\n        235,\n        246,\n        236,\n        244,\n        224,\n        218,\n        210,\n        211,\n        204,\n        187,\n        181,\n        187,\n        191,\n        174,\n        186,\n        172,\n        171,\n        164,\n        172,\n        171,\n        177,\n        170,\n        154,\n        155,\n        148,\n        148,\n        148,\n        143,\n        142,\n        140,\n        142,\n        130,\n        137,\n        135,\n        126,\n        129,\n        128,\n        135,\n        130,\n        121,\n        132,\n        122,\n        128,\n        125,\n        118,\n        126,\n        135,\n        130,\n        124,\n        122,\n        128,\n        121,\n        124,\n        126,\n        125,\n        125,\n        125,\n        127,\n        120,\n        132,\n        131,\n        132,\n        124,\n        131\n      ],\n      "coordinate_sha256": "08459ef607f9edf3b37118cb3a3fd3f5a44d0ea2b49a53a2e52aa9e5ce9c461c",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            16854\n          ],\n          "sha256": "8ae83595b7a9ead60121ea370cf0f95177fc1fed7e2d282dd0bacfc82d33b7d8"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            16854\n          ],\n          "sha256": "7fabf9cb08cf5c880047025815336892842f38214d45518a914c8ac242e33e5d"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            16854\n          ],\n          "sha256": "9f37895c057bb442c539647199451073f8101b5d317dd2ec4335dd66157304a3"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            16854\n          ],\n          "sha256": "d11ff46bd88ca019b356176a2e44880631982374db9d133f4c5d1dfb8a88bc51"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            18036\n          ],\n          "sha256": "554d3887e7991a7a644a280d9ac507e0fa4fe5b3fca3633c176ab2d3b80ff916"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            18036\n          ],\n          "sha256": "fa8fa10a80e24543d35c6800b97db4918eef2e77f563bd26562ea2f7b63587c6"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            18036\n          ],\n          "sha256": "b043cfae2269e25d8ca35455cc04dcdaa45a645898327855a40505c6c529bf3b"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            18036\n          ],\n          "sha256": "262f79cf38aa75e19ebfdadfa6a4b32792febc844a2204c72689bb029a8ad828"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            18036\n          ],\n          "sha256": "5ce762dfdc06c1a34a6aa3de60c31376d631825bb4960e301a105ce7589e3c0c"\n        }\n      }\n    },\n    "6bba_207c6aaf": {\n      "frame_counts": [\n        69,\n        69,\n        78,\n        78,\n        85,\n        84,\n        93,\n        105,\n        107,\n        117,\n        122,\n        119,\n        129,\n        145,\n        154,\n        158,\n        174,\n        185,\n        196,\n        195,\n        199,\n        231,\n        223,\n        238,\n        240,\n        241,\n        284,\n        264,\n        275,\n        270,\n        278,\n        262,\n        268,\n        276,\n        277,\n        279,\n        315,\n        318,\n        341,\n        318,\n        312,\n        298,\n        312,\n        321,\n        338,\n        311,\n        317,\n        319,\n        319,\n        316,\n        373,\n        348,\n        345,\n        343,\n        355,\n        362,\n        331,\n        314,\n        331,\n        333,\n        349,\n        312,\n        309,\n        307,\n        308,\n        288,\n        283,\n        283,\n        287,\n        276,\n        274,\n        272,\n        268,\n        303,\n        255,\n        263,\n        263,\n        257,\n        258,\n        269,\n        272,\n        251,\n        267,\n        259,\n        222,\n        204,\n        203,\n        219,\n        217,\n        253,\n        238,\n        200,\n        188,\n        185,\n        184,\n        209,\n        181,\n        188,\n        155,\n        152\n      ],\n      "coordinate_sha256": "f227505f39baefad6de36ce0820ba83726b02a07ced10bd6809fca39d33a079e",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            17498\n          ],\n          "sha256": "f26bce3eb4e570899eb71ef1d8ab71cd66a90cf67855b291c11080a0383ccaa6"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            17498\n          ],\n          "sha256": "1eb5a1287b82d7eb3ef236245d38dae188bad004b7ae59d6d61a91d99a927f60"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            17498\n          ],\n          "sha256": "922d020a1a99b2f8fba37a198e27f7a8307a21c1e378e9ea2ba1da5511995c58"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            17498\n          ],\n          "sha256": "714741fadb87b0391a9ba944cdff11ab03679a4fd6e6dfe1fc04410ca36543c7"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            19274\n          ],\n          "sha256": "80fda221655e00350267755f78c056e8784d536fed5e75c3229e027b3ec3f196"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            19274\n          ],\n          "sha256": "2944ff72188ad95ba2f5aa6e5ae996bd2eddc9f97e47ac3acfced25775ebd0d4"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            19274\n          ],\n          "sha256": "07525610fffe59c2b5c696067db5427ae31553f2540f39749c2ab52b4761cc78"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            19274\n          ],\n          "sha256": "b3c0c98ce91978a828ac169b057598cbb13bd8453370adba46e3fd60988b0621"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            19274\n          ],\n          "sha256": "b8b6c7f3330f12c594216151b14527acbbc53b0110bf6f70e0f1fa0c395fc3fd"\n        }\n      }\n    },\n    "6bba_20852818": {\n      "frame_counts": [\n        258,\n        258,\n        253,\n        247,\n        236,\n        228,\n        225,\n        215,\n        213,\n        214,\n        205,\n        201,\n        197,\n        193,\n        191,\n        188,\n        180,\n        176,\n        171,\n        167,\n        165,\n        147,\n        144,\n        139,\n        141,\n        138,\n        130,\n        131,\n        122,\n        122,\n        124,\n        122,\n        117,\n        112,\n        110,\n        110,\n        107,\n        106,\n        106,\n        111,\n        100,\n        96,\n        97,\n        98,\n        96,\n        96,\n        96,\n        150,\n        82,\n        81,\n        78,\n        80,\n        73,\n        73,\n        75,\n        74,\n        73,\n        65,\n        65,\n        69,\n        69,\n        66,\n        65,\n        64,\n        66,\n        65,\n        66,\n        66,\n        66,\n        62,\n        63,\n        69,\n        64,\n        67,\n        65,\n        65,\n        63,\n        63,\n        66,\n        66,\n        66,\n        69,\n        62,\n        63,\n        68,\n        68,\n        72,\n        67,\n        69,\n        70,\n        74,\n        67,\n        69,\n        67,\n        71,\n        72,\n        73,\n        75,\n        71,\n        68\n      ],\n      "coordinate_sha256": "19d38bd77ac2e463c9cb83b10973cdf30bf730d3446c3bcea737bd4e62c7efcf",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            9797\n          ],\n          "sha256": "96212af6c7d1081886d7d177bb412baeadfa4e3877217a685f4e5045645c1f26"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            9797\n          ],\n          "sha256": "dd59e17503d02eeaf7aca18170c4fb961d43ecc978b45a1b700cc5f248e1ee57"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            9797\n          ],\n          "sha256": "3658c5537e7f255f74ba379f3aaeb124f2c9d7c4003bdd54e1f57bebaa95a63e"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            9797\n          ],\n          "sha256": "8dd419b26eac29e536efd49aca199167dda2009bf0682f6af276de8833df2684"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            10378\n          ],\n          "sha256": "2cda6f076dbcf2ecd217a3b1afb1c0305ad4102dbf7800cc07b1a4a34d349e37"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            10378\n          ],\n          "sha256": "5f4dd22e9b2bc72250d50eb5497aba4fda168d56c7a01e7c82c782c061821e8d"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            10378\n          ],\n          "sha256": "3feeef3d3c87cef1b49210c6a094ba64ae48603f92355514ff4ad416379e4097"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            10378\n          ],\n          "sha256": "f7b1bd0630ccb317aeb0dad89fa8a347c5ccf741d3cb2a93d76664c532aa0e69"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            10378\n          ],\n          "sha256": "8cf521d523f0b67ad113fd89359f04aa8741e473e25a2b2cc9d11b833892f03a"\n        }\n      }\n    },\n    "6bba_2312ac41": {\n      "frame_counts": [\n        252,\n        237,\n        238,\n        236,\n        232,\n        230,\n        227,\n        224,\n        223,\n        214,\n        210,\n        216,\n        207,\n        201,\n        197,\n        198,\n        190,\n        188,\n        190,\n        182,\n        181,\n        180,\n        175,\n        169,\n        168,\n        167,\n        164,\n        162,\n        151,\n        153,\n        153,\n        152,\n        165,\n        156,\n        155,\n        149,\n        154,\n        145,\n        153,\n        143,\n        144,\n        140,\n        139,\n        138,\n        132,\n        134,\n        136,\n        133,\n        135,\n        136,\n        132,\n        127,\n        128,\n        124,\n        123,\n        119,\n        118,\n        119,\n        127,\n        119,\n        120,\n        117,\n        115,\n        115,\n        112,\n        110,\n        106,\n        100,\n        101,\n        99,\n        94,\n        89,\n        89,\n        88,\n        92,\n        88,\n        83,\n        79,\n        80,\n        83,\n        79,\n        76,\n        78,\n        75,\n        78,\n        77,\n        76,\n        75,\n        76,\n        77,\n        79,\n        79,\n        76,\n        73,\n        71,\n        68,\n        67,\n        68,\n        68,\n        64\n      ],\n      "coordinate_sha256": "50c41e6e12d029fc7cf85138f7d3cf30d632da5e8eb429726a3dc01000be4f1f",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            12192\n          ],\n          "sha256": "8cbb05ae77234cbc86815980a235c04d71d8ebf9687433ca06d6b8620b6e79f8"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            12192\n          ],\n          "sha256": "0c97f9139fa781ee5be8220b32208e86cab9d1c575234ead420efe487b785e23"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            12192\n          ],\n          "sha256": "46da1f572d7b3e4a29c03e30ab6bca7a5bf6491af0e79518a35c9a92fa418612"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            12192\n          ],\n          "sha256": "21e4639f468ba2b60520ddeecaf83f16d1718659b610641c1a6cd40b49f80612"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            12697\n          ],\n          "sha256": "cf2399d166bd15092532fe53247d55e8b1bef8f1db4344096422b105dbe1630f"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            12697\n          ],\n          "sha256": "36885331d1283a8a0206131b36ebbb0e273c7b0b3dbc96e7770eb91f80d7db4b"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            12697\n          ],\n          "sha256": "5253ba7b36e076212307598cccb573d01e46b1d6e8894224fbbaab4ff7e06bf9"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            12697\n          ],\n          "sha256": "f0654b2d8e211d7c8c425ee44c9603f4555361eac33c3e50ff77a35a6890a0db"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            12697\n          ],\n          "sha256": "b173e829e81367c3d146628f479b44b699e05fac6ee30490e49e2d8fa756bc68"\n        }\n      }\n    },\n    "6bba_268e1230": {\n      "frame_counts": [\n        21,\n        19,\n        20,\n        22,\n        22,\n        22,\n        20,\n        22,\n        19,\n        20,\n        22,\n        25,\n        23,\n        24,\n        25,\n        23,\n        22,\n        27,\n        26,\n        28,\n        29,\n        28,\n        26,\n        27,\n        27,\n        28,\n        31,\n        32,\n        31,\n        52,\n        49,\n        45,\n        41,\n        43,\n        44,\n        51,\n        64,\n        60,\n        46,\n        46,\n        43,\n        42,\n        44,\n        44,\n        47,\n        45,\n        45,\n        68,\n        54,\n        50,\n        61,\n        66,\n        57,\n        60,\n        60,\n        58,\n        55,\n        53,\n        49,\n        63,\n        63,\n        72,\n        62,\n        51,\n        51,\n        53,\n        60,\n        62,\n        51,\n        51,\n        60,\n        79,\n        74,\n        80,\n        80,\n        58,\n        56,\n        59,\n        60,\n        66,\n        63,\n        79,\n        70,\n        77,\n        74,\n        74,\n        109,\n        111,\n        115,\n        122,\n        136,\n        156,\n        168,\n        171,\n        174,\n        185,\n        247,\n        258,\n        262,\n        285\n      ],\n      "coordinate_sha256": "8d33dd990dadaf760f473175b42e1ec0ab077023b9b603a121734f3a7c34c383",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            4829\n          ],\n          "sha256": "9a92f7a47fb1d07425caa09fba26a1d73edf7ecf2a411a61c9a3ed5cc5ec3ef5"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            4829\n          ],\n          "sha256": "3d23d19f16982ad04e8fe2498e7a48290d9d4328b6014d0da853f193ff0920aa"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            4829\n          ],\n          "sha256": "b42cee2ca27401af0e6d273e608fb1e075b37ef51d17f6522cc4d7612e5226e0"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            4829\n          ],\n          "sha256": "993c427a64d8d4f0c813e7fe6dbdbe623d992cde90ea2a253f12f0583c318479"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            5291\n          ],\n          "sha256": "8616f3bd2272722aae067137d036a965b8e146692ed97e592911f4c873a77ebc"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            5291\n          ],\n          "sha256": "2debb08ce755ca177e1a903d6f4279385661eb9439fe6ff6abc012b84ca02bf1"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            5291\n          ],\n          "sha256": "e711b33718ab5ebb9a5bb4358814bdf62078eba91900d2784ed77e3c1e572b21"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            5291\n          ],\n          "sha256": "a2434a9c97ca65f0b4b55c6134df0752478f90bd7b924672da3edaa8aaae5723"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            5291\n          ],\n          "sha256": "4c7de387c6178641b4b2ba4c2e3b2f1500bf4709b7391a8ccdfcac7e55978fed"\n        }\n      }\n    },\n    "6bba_2819ca14": {\n      "frame_counts": [\n        79,\n        79,\n        78,\n        79,\n        85,\n        74,\n        72,\n        77,\n        69,\n        77,\n        68,\n        77,\n        76,\n        63,\n        64,\n        61,\n        66,\n        58,\n        61,\n        61,\n        62,\n        64,\n        58,\n        53,\n        52,\n        49,\n        50,\n        51,\n        50,\n        49,\n        52,\n        58,\n        54,\n        52,\n        54,\n        55,\n        51,\n        55,\n        53,\n        53,\n        50,\n        49,\n        48,\n        47,\n        50,\n        49,\n        48,\n        48,\n        46,\n        47,\n        52,\n        49,\n        48,\n        47,\n        49,\n        50,\n        50,\n        44,\n        47,\n        46,\n        46,\n        52,\n        56,\n        55,\n        53,\n        52,\n        55,\n        53,\n        64,\n        51,\n        53,\n        51,\n        52,\n        51,\n        50,\n        51,\n        51,\n        50,\n        53,\n        53,\n        56,\n        55,\n        53,\n        51,\n        54,\n        55,\n        57,\n        49,\n        50,\n        48,\n        48,\n        50,\n        50,\n        52,\n        53,\n        50,\n        48,\n        45,\n        44,\n        46\n      ],\n      "coordinate_sha256": "f97b6c33ed2d8baa3ee48a1dd9420e5fb032c4309dc85c9d5596afb5feaa133f",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            4870\n          ],\n          "sha256": "a790d4b5361ac7b383def161f727e0be780092dcf6665d62a77369f135532af2"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            4870\n          ],\n          "sha256": "59c0ba720b2d710619d66b91d924eef719c083c05d3434a045d0971a688c9db9"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            4870\n          ],\n          "sha256": "9c59a79f0b4c0ecd3322fd5714e2a035809fcc3c5905eff0d085284baccc8374"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            4870\n          ],\n          "sha256": "646c411287380a0479ba4fdcc51e3cbb6e13f0bae6f60f286619df019aa17fe0"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            5129\n          ],\n          "sha256": "e59cdc892522e7d0436a3071cbf1a3cb0861e33d66eb6cf20d9b09b5c6cd8a51"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            5129\n          ],\n          "sha256": "0472917d149bb87dc948b1e91042e766443a27732efb2188ef5c5d279a365ae5"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            5129\n          ],\n          "sha256": "a832cee3428e49c10896c367862efb51bd16446648401300869587f0498e3eb1"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            5129\n          ],\n          "sha256": "9e37db0d5558c6bacb37e8610e528c616210c8b835d3795013ee7a4343b4793e"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            5129\n          ],\n          "sha256": "645bf9746a5c0d0aaecce3ba532014883068211327d237187d156fab5ea2f4e9"\n        }\n      }\n    },\n    "6bba_32db13fc": {\n      "frame_counts": [\n        783,\n        798,\n        822,\n        803,\n        758,\n        801,\n        784,\n        792,\n        796,\n        787,\n        752,\n        747,\n        752,\n        742,\n        718,\n        732,\n        697,\n        686,\n        681,\n        646,\n        667,\n        681,\n        659,\n        644,\n        610,\n        613,\n        600,\n        615,\n        578,\n        578,\n        578,\n        825,\n        583,\n        561,\n        544,\n        572,\n        555,\n        551,\n        559,\n        566,\n        553,\n        533,\n        490,\n        549,\n        536,\n        533,\n        506,\n        534,\n        526,\n        525,\n        513,\n        492,\n        514,\n        486,\n        494,\n        499,\n        499,\n        476,\n        462,\n        475,\n        469,\n        456,\n        440,\n        446,\n        442,\n        427,\n        424,\n        406,\n        411,\n        433,\n        412,\n        409,\n        415,\n        413,\n        400,\n        403,\n        413,\n        414,\n        394,\n        401,\n        386,\n        375,\n        377,\n        372,\n        366,\n        368,\n        359,\n        361,\n        359,\n        336,\n        341,\n        341,\n        322,\n        331,\n        327,\n        307,\n        296,\n        291,\n        289,\n        289\n      ],\n      "coordinate_sha256": "1d85a0c777a17725d80d58a6e52efdc897b7e8413b89861817cdb51ecad539b1",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            43077\n          ],\n          "sha256": "d914f2a875951c3f2c5d2efeb6acf076c32b0fae93210bdbb50b3e9e66de0d68"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            43077\n          ],\n          "sha256": "02475019a7b7cd9ceb06be2fdec44b2d9b722a6c672540105350ca1c187d5ccc"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            43077\n          ],\n          "sha256": "fd01e9203ef3dc2ab67fa095c6b762490b3cb59ac7d36254097a6930ad22351f"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            43077\n          ],\n          "sha256": "e7db1cb6170fa9555b6cbf22de5f765dfa995b9a01b99c44908866a82204711c"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            47120\n          ],\n          "sha256": "3a4b8a0265d344833570a247c708b60bed41cad6c22e44409383549a82fa2c4a"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            47120\n          ],\n          "sha256": "a401e1509ea4c3c6affadd0f64ce122c61ed59e8357720a170a9edb99e25de8e"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            47120\n          ],\n          "sha256": "350bc847fe9ad6d0fb77d04925761ce52522092cba4d2ee3b3cbdbf213c0e1ec"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            47120\n          ],\n          "sha256": "362117fccae39b8af630e3f905e1f132ce15e5607307b785139f6ec24755e3db"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            47120\n          ],\n          "sha256": "939d83a46d38bd5bae0b5024cdeb7971cd4342e69ac996edeb83a88fe8a28a72"\n        }\n      }\n    },\n    "6bba_337b1b3a": {\n      "frame_counts": [\n        165,\n        191,\n        202,\n        216,\n        231,\n        244,\n        258,\n        269,\n        282,\n        301,\n        314,\n        324,\n        335,\n        333,\n        337,\n        342,\n        335,\n        338,\n        337,\n        340,\n        340,\n        338,\n        340,\n        339,\n        335,\n        333,\n        332,\n        327,\n        317,\n        315,\n        312,\n        314,\n        303,\n        301,\n        293,\n        299,\n        300,\n        293,\n        295,\n        298,\n        300,\n        303,\n        298,\n        298,\n        301,\n        298,\n        292,\n        287,\n        285,\n        288,\n        280,\n        282,\n        285,\n        285,\n        284,\n        288,\n        285,\n        291,\n        283,\n        287,\n        279,\n        277,\n        280,\n        278,\n        287,\n        286,\n        293,\n        292,\n        295,\n        300,\n        301,\n        299,\n        309,\n        298,\n        301,\n        299,\n        298,\n        297,\n        309,\n        310,\n        317,\n        314,\n        313,\n        315,\n        311,\n        309,\n        309,\n        312,\n        321,\n        316,\n        310,\n        321,\n        317,\n        327,\n        326,\n        333,\n        331,\n        327,\n        342,\n        346\n      ],\n      "coordinate_sha256": "b0e853cacc9dd9e941b094497e04349bd031320913c5ebca4ab2bb18b94da7e3",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            28150\n          ],\n          "sha256": "b02dd013f8f8c4575a1fdfe94566d5d40e71a592c5d3293890733f6f6d9c6b51"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            28150\n          ],\n          "sha256": "637964adbaccf6d563c702e2836eb11f901aef6fb8cdaf6f43c2f7de89ec881c"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            28150\n          ],\n          "sha256": "234fbb89a70e2ae4f7583153f64daa837c7dd00ca19c0335b0c46e92062ce1c0"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            28150\n          ],\n          "sha256": "8c0151179807148f0c99a9419fae6436911616a7e429246ea3280dc1b6daf938"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            29409\n          ],\n          "sha256": "3634633e016509cdeda58b6f4f23afb58f0f8760a1ab6905ad5dbda0a5cbce5d"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            29409\n          ],\n          "sha256": "81279e45a4701e61a78e7f98e2d4befcf395cc6cd6a51cc7fc0a5f99f860d1a2"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            29409\n          ],\n          "sha256": "9891bad99637a21b9cd9205e98471c68da1cac1c08beb50a488b23bdcf6451d0"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            29409\n          ],\n          "sha256": "bbc854b1eeef9aac43fb0655a5c35622676cd5fa65b2057d07166532475defb0"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            29409\n          ],\n          "sha256": "02305b2fabbf52ded35473d0657d3ae1feb155c327819cfe4a77d28aad508215"\n        }\n      }\n    },\n    "6bba_3abfe10a": {\n      "frame_counts": [\n        276,\n        265,\n        282,\n        258,\n        289,\n        288,\n        273,\n        298,\n        274,\n        307,\n        319,\n        326,\n        333,\n        321,\n        348,\n        348,\n        386,\n        383,\n        389,\n        365,\n        406,\n        398,\n        411,\n        423,\n        399,\n        475,\n        456,\n        482,\n        507,\n        510,\n        525,\n        571,\n        583,\n        564,\n        567,\n        633,\n        643,\n        639,\n        659,\n        681,\n        677,\n        737,\n        756,\n        759,\n        726,\n        758,\n        743,\n        794,\n        745,\n        745,\n        745,\n        810,\n        792,\n        805,\n        758,\n        811,\n        794,\n        806,\n        817,\n        792,\n        797,\n        779,\n        773,\n        817,\n        819,\n        830,\n        822,\n        825,\n        810,\n        831,\n        817,\n        802,\n        804,\n        780,\n        769,\n        780,\n        770,\n        767,\n        748,\n        795,\n        767,\n        778,\n        764,\n        748,\n        773,\n        768,\n        746,\n        751,\n        766,\n        743,\n        751,\n        775,\n        763,\n        756,\n        725,\n        741,\n        732,\n        720,\n        709,\n        717\n      ],\n      "coordinate_sha256": "6a81e3452080612daab56377380191b92c04008aca43fe813169a0baf7fa7046",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            52123\n          ],\n          "sha256": "0c5439a2d2ba5881bec708b6db02322390d179c51cf93f9dd7e9c2d67faedf6b"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            52123\n          ],\n          "sha256": "7d3db91a1be4589d196eb95cdb7cdf5412a15a3f774939c6e13a792463ccef34"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            52123\n          ],\n          "sha256": "c7b92bb3a0c33de4fe875e4eff8e474ab9dc4f6c8123d7dd63b355ccd7fb7a8e"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            52123\n          ],\n          "sha256": "67ba46784b671667febe6c38094b53e4e4a294a0c77968f6e43f91effa2c9311"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            56736\n          ],\n          "sha256": "377b824e093b54d5dd9cb873281c9a44d3723efa6d4726c6c808e3428f556f25"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            56736\n          ],\n          "sha256": "192d6554c1c180f258351b0bd125529ace84f98bec4f4ff12275df98ee9c99ae"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            56736\n          ],\n          "sha256": "86fab23f4250f7d595d3d1a7c5255381a0e9a85a938111d358702487a934d1a9"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            56736\n          ],\n          "sha256": "16fdd56459f90897918de20887657e817ad8e1e9bfe24bd2339508fa440ddfbf"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            56736\n          ],\n          "sha256": "fa895d665cd2642a6c61852fbee81e6918b34541830e4c19244c720526617581"\n        }\n      }\n    },\n    "6bba_3c5691b6": {\n      "frame_counts": [\n        65,\n        66,\n        53,\n        56,\n        58,\n        56,\n        68,\n        72,\n        75,\n        74,\n        70,\n        76,\n        73,\n        74,\n        68,\n        65,\n        62,\n        59,\n        60,\n        52,\n        53,\n        54,\n        62,\n        58,\n        63,\n        62,\n        64,\n        65,\n        61,\n        59,\n        59,\n        55,\n        55,\n        58,\n        60,\n        58,\n        65,\n        69,\n        70,\n        73,\n        78,\n        77,\n        91,\n        81,\n        86,\n        83,\n        83,\n        79,\n        84,\n        84,\n        76,\n        77,\n        73,\n        71,\n        75,\n        71,\n        74,\n        72,\n        74,\n        70,\n        72,\n        70,\n        73,\n        74,\n        76,\n        73,\n        73,\n        72,\n        73,\n        71,\n        70,\n        70,\n        73,\n        72,\n        80,\n        76,\n        74,\n        80,\n        79,\n        105,\n        101,\n        89,\n        79,\n        79,\n        73,\n        71,\n        89,\n        89,\n        90,\n        92,\n        102,\n        111,\n        113,\n        81,\n        79,\n        86,\n        103,\n        80,\n        80,\n        78\n      ],\n      "coordinate_sha256": "acf6b19e95b65bf0980d7a41e96fd85cc9e190fcc53bb4054f545c69b56de2a8",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            6144\n          ],\n          "sha256": "0524bf2dca6ee0f4008c4bfebffa7f0e7990e85ba21a7b36354e40dbc0a070e9"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            6144\n          ],\n          "sha256": "a52e53102293ee82db7d946cd2c60bdc8f87cab38cfc02d81998c615664153b0"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            6144\n          ],\n          "sha256": "3a1c8d384a99310b76f11d058f099943315c7790e713b7ed2b62e59fc16c3e99"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            6144\n          ],\n          "sha256": "0619558ee066ca3069a28a87f69fcc4d8355f9490e0cf8c0571f4642d18d4f28"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            6403\n          ],\n          "sha256": "3badeb5a050f932cd94a84546123477fc2413e3d62abcbdb130cd5ec7b729580"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            6403\n          ],\n          "sha256": "e4dedc58c7ee70d32fcaa9b467740d3ec609e2013c0b2ffd821b2f98c3f5796f"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            6403\n          ],\n          "sha256": "a787a6d83134ede75083d5673a47655c3e03dcc5acbd6eb36c460c6974521b25"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            6403\n          ],\n          "sha256": "310c34bc78207baec3de59d41270b4aed009d74996999203f402f40e8aa499db"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            6403\n          ],\n          "sha256": "271700fcc610dbb69bf4abba4bfeec943be609e0cf817a22611d5d5bd59e21e4"\n        }\n      }\n    },\n    "6bba_3db54e20": {\n      "frame_counts": [\n        43,\n        44,\n        40,\n        39,\n        39,\n        40,\n        42,\n        42,\n        46,\n        41,\n        44,\n        43,\n        43,\n        42,\n        43,\n        44,\n        49,\n        48,\n        46,\n        44,\n        46,\n        53,\n        55,\n        52,\n        57,\n        58,\n        57,\n        57,\n        58,\n        59,\n        60,\n        62,\n        69,\n        63,\n        67,\n        71,\n        71,\n        84,\n        93,\n        97,\n        114,\n        114,\n        127,\n        146,\n        151,\n        166,\n        172,\n        180,\n        194,\n        201,\n        218,\n        229,\n        235,\n        246,\n        257,\n        325,\n        349,\n        339,\n        384,\n        397,\n        406,\n        417,\n        443,\n        458,\n        451,\n        469,\n        446,\n        491,\n        475,\n        524,\n        518,\n        538,\n        551,\n        562,\n        553,\n        540,\n        566,\n        560,\n        550,\n        533,\n        554,\n        551,\n        548,\n        562,\n        556,\n        561,\n        559,\n        568,\n        561,\n        533,\n        544,\n        525,\n        528,\n        517,\n        523,\n        515,\n        497,\n        493,\n        492,\n        504\n      ],\n      "coordinate_sha256": "93cbd1a9d7aacc2d776c66439c10e412f5c8a9803dd0bb16255a4dc46967c408",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            22597\n          ],\n          "sha256": "64cb8eb953f15716e0ccaf5c28bba58756a064479de2fb29c77f6eed4ce30c7a"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            22597\n          ],\n          "sha256": "fc899996d1a79856b54e0984bbd6eedf9c80dfe2b765cff701dd07bc09e8afc0"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            22597\n          ],\n          "sha256": "e20c9b7d52ed241b9d65d5aabe81c58ea09cf2bdd92d0f89bb137990196af083"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            22597\n          ],\n          "sha256": "016e35b8fcbc551b227cd821a4fd575783a72111ded09adfbb4b08b88f972b3e"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            24590\n          ],\n          "sha256": "0de33dadd62aeeff6aacfd766724a66965b02cec47b4e254ede83dd82f23cc3a"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            24590\n          ],\n          "sha256": "054cc0839d73848829a9f34cf29f42c98d5a603a3ddeb8dd4dcd06ea61cfc115"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            24590\n          ],\n          "sha256": "807a0380f17c0cd91ec1f522b7f7427697ce56555a755d90dc4e0e33293f6fe6"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            24590\n          ],\n          "sha256": "c5c3262a3391873aecded30e06a87e5175ac6a1d0de3831cae97f021eb0b38c3"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            24590\n          ],\n          "sha256": "516266ca845ede1108678982a5d59caf82db29b5b7714ebdb23d149c2b8f6ede"\n        }\n      }\n    },\n    "6bba_3fda6b25": {\n      "frame_counts": [\n        130,\n        130,\n        120,\n        110,\n        108,\n        106,\n        101,\n        110,\n        115,\n        106,\n        112,\n        120,\n        113,\n        118,\n        110,\n        105,\n        104,\n        104,\n        97,\n        95,\n        99,\n        98,\n        105,\n        99,\n        99,\n        102,\n        103,\n        108,\n        106,\n        100,\n        95,\n        101,\n        99,\n        97,\n        101,\n        97,\n        115,\n        109,\n        96,\n        96,\n        99,\n        92,\n        93,\n        99,\n        97,\n        99,\n        104,\n        105,\n        104,\n        102,\n        103,\n        96,\n        109,\n        113,\n        106,\n        110,\n        105,\n        102,\n        105,\n        97,\n        107,\n        107,\n        99,\n        102,\n        109,\n        101,\n        111,\n        97,\n        99,\n        112,\n        104,\n        102,\n        90,\n        90,\n        100,\n        92,\n        91,\n        108,\n        104,\n        98,\n        92,\n        95,\n        108,\n        100,\n        105,\n        108,\n        121,\n        106,\n        114,\n        116,\n        112,\n        109,\n        107,\n        108,\n        118,\n        113,\n        116,\n        113,\n        122,\n        131\n      ],\n      "coordinate_sha256": "e08f513d619b6c43ad7b8df7889944a9157080c147060511d6a4b3d992fc2f57",\n      "raw_graph_signature": {\n        "edge_edge_dist": {\n          "dtype": "<f8",\n          "shape": [\n            8456\n          ],\n          "sha256": "4c4fcee56faf7ad7a5e079e1f940f866303f9da60f7c43df927e8918d183c78a"\n        },\n        "edge_edge_prob": {\n          "dtype": "<f8",\n          "shape": [\n            8456\n          ],\n          "sha256": "d9ceeb47b189f3d3276281149ee18c77244f721ccb72d211a70b243ef1756243"\n        },\n        "edge_source_id": {\n          "dtype": "<i8",\n          "shape": [\n            8456\n          ],\n          "sha256": "f8cf3307874c3c7f26fc4e7717f02af7c544f3b06d9a01f7ab7518a2ed8b1f9a"\n        },\n        "edge_target_id": {\n          "dtype": "<i8",\n          "shape": [\n            8456\n          ],\n          "sha256": "e2ad18672ffd822698601c937904ef46000eb00e2464cd8183e6bcc891019170"\n        },\n        "node_node_id": {\n          "dtype": "<i8",\n          "shape": [\n            8896\n          ],\n          "sha256": "83b3545e3bcf90da123168b9e0db7a2a81364f28fedcbdd9a513a5bbbb53ada0"\n        },\n        "node_t": {\n          "dtype": "<i8",\n          "shape": [\n            8896\n          ],\n          "sha256": "cc36954552d1dc91e9cd04cdd6a529d94c775a0446e9a9050635d8be20626eff"\n        },\n        "node_x": {\n          "dtype": "<f8",\n          "shape": [\n            8896\n          ],\n          "sha256": "4d3bba761d3206b71c4f794a8b15283000cf97db5ba3d3899d88ed987d473ffd"\n        },\n        "node_y": {\n          "dtype": "<f8",\n          "shape": [\n            8896\n          ],\n          "sha256": "3f1b353a2f047f835d6a7ae7574ea84dacceac49d487879c9bb58ae6acf7f700"\n        },\n        "node_z": {\n          "dtype": "<f8",\n          "shape": [\n            8896\n          ],\n          "sha256": "7e869f72ecb81cf03649500a1607a2e68e6f55e29473480376c8605c20ff8629"\n        }\n      }\n    }\n  },\n  "groups": [\n    {\n      "group_id": "group00",\n      "stage": "diagnostic12",\n      "datasets": [\n        "44b6_12dfb391",\n        "44b6_267148e4",\n        "6bba_062c8d37",\n        "6bba_07e24132"\n      ],\n      "detectors": 118314,\n      "dense_pairs": 44972058\n    },\n    {\n      "group_id": "group01",\n      "stage": "diagnostic12",\n      "datasets": [\n        "44b6_2a2eff9f",\n        "44b6_341df25f",\n        "6bba_085bf656",\n        "6bba_09961292"\n      ],\n      "detectors": 95155,\n      "dense_pairs": 32078477\n    },\n    {\n      "group_id": "group02",\n      "stage": "diagnostic12",\n      "datasets": [\n        "44b6_587a1e22",\n        "44b6_5f15d135",\n        "6bba_0e7c0d07",\n        "6bba_12665c0e"\n      ],\n      "detectors": 83174,\n      "dense_pairs": 23379549\n    },\n    {\n      "group_id": "group03",\n      "stage": "remaining24",\n      "datasets": [\n        "44b6_706092f0",\n        "44b6_74d0c52e",\n        "6bba_1d0d8384",\n        "6bba_207c6aaf"\n      ],\n      "detectors": 91058,\n      "dense_pairs": 25136016\n    },\n    {\n      "group_id": "group04",\n      "stage": "remaining24",\n      "datasets": [\n        "44b6_7a302da0",\n        "44b6_996155de",\n        "6bba_20852818",\n        "6bba_2312ac41"\n      ],\n      "detectors": 89303,\n      "dense_pairs": 33712120\n    },\n    {\n      "group_id": "group05",\n      "stage": "remaining24",\n      "datasets": [\n        "44b6_9be80b04",\n        "44b6_a21120c2",\n        "6bba_268e1230",\n        "6bba_2819ca14"\n      ],\n      "detectors": 66502,\n      "dense_pairs": 16207798\n    },\n    {\n      "group_id": "group06",\n      "stage": "remaining24",\n      "datasets": [\n        "44b6_aaf8b0ea",\n        "44b6_c50204e0",\n        "6bba_32db13fc",\n        "6bba_337b1b3a"\n      ],\n      "detectors": 151221,\n      "dense_pairs": 67524955\n    },\n    {\n      "group_id": "group07",\n      "stage": "remaining24",\n      "datasets": [\n        "44b6_c8e2a523",\n        "44b6_d2f34f90",\n        "6bba_3abfe10a",\n        "6bba_3c5691b6"\n      ],\n      "detectors": 119578,\n      "dense_pairs": 56392786\n    },\n    {\n      "group_id": "group08",\n      "stage": "remaining24",\n      "datasets": [\n        "44b6_d5e7d891",\n        "44b6_d754aa59",\n        "6bba_3db54e20",\n        "6bba_3fda6b25"\n      ],\n      "detectors": 96647,\n      "dense_pairs": 41702315\n    }\n  ],\n  "raw_inventory": {\n    "root": "outputs/kaggle/e22_bidir030_eval36_raw/tracking_repo/predictions/unknown/unet_transformer_val/split_0",\n    "files": 1188,\n    "bytes": 10090215,\n    "sha256": "0fbc819805e63a7427b2921a8639e5433282385bcdcc5a0213847888a00974a2",\n    "serialization": "sorted relative file records; _json_bytes"\n  },\n  "source_sha256": {\n    "off": "8e7ac19e8b436d6777576b7ea2269464c4ed8842990b45448d40bf178f8f62de",\n    "on": "e2fda47e41d970a2c650bbbbb5fe068b6e1b9667fdd91cb59cc2a3ac8ba95a9c"\n  },\n  "bindings": [\n    {\n      "path": "outputs/local/e23_association_target_20260908/PARENT_ARTIFACT_AUDIT.json",\n      "bytes": 18717,\n      "sha256": "a830c98c85be60b901ca101043340b8494008588d7398891608d2f193f344dab"\n    },\n    {\n      "path": "outputs/local/e26_diagnostic/d2b_eval12_v2_202609081002/result/RESULT.json",\n      "bytes": 37489,\n      "sha256": "2df2b4c2d4cb3aad978358c52b66a893de147b4428816448b2775e090753d767"\n    },\n    {\n      "path": "outputs/kaggle/e22_bidir030_eval36_reference/detector_coordinates_harmonic_association_production_single.jsonl",\n      "bytes": 47076,\n      "sha256": "4355d4fe0c70c4152c09fb55f4988d46db434fcb924b2859c9b22fee7928d1a3"\n    },\n    {\n      "path": "outputs/local/e23_association_target_20260908/terminal/association_public4_plan.json",\n      "bytes": 18754,\n      "sha256": "a8b951b2fbf378d31054fcfcb3846509fe0f66d273ea6bfa60891b44f7880b54"\n    },\n    {\n      "path": "src/biohub/association_capture.py",\n      "bytes": 11101,\n      "sha256": "9015430194aad491bbc0df722fa966d9a21c469b29b833b090a6d445ccf4da36"\n    },\n    {\n      "path": "src/biohub/association_observer.py",\n      "bytes": 14990,\n      "sha256": "4615cce9840cc260530e965b4ccf6ecb47ed096335260a2148c67fa1eed45425"\n    },\n    {\n      "path": "src/biohub/association_instrumentation.py",\n      "bytes": 5009,\n      "sha256": "9fa79df7176d1f09ba840247dca8a71da204f62a2dbf877a2b8aa5617333b795"\n    },\n    {\n      "path": "src/biohub/association_parity.py",\n      "bytes": 12952,\n      "sha256": "3d0188c69c0b1d29c4cd0b057098df1adc8e24e3d35468b7df9f36a78557ecf7"\n    },\n    {\n      "path": "src/biohub/association_supervise.py",\n      "bytes": 4731,\n      "sha256": "457b8e3505b29bbf123c2effea4e56054a900f06777f71bc4b18b586d59692c6"\n    },\n    {\n      "path": "scripts/e23_association_bridge.py",\n      "bytes": 597,\n      "sha256": "88b4df5da2493f719145ac20a3cefc0ad18407acddbe8cac5acdbebc11fa4bcf"\n    }\n  ],\n  "limits": {\n    "group_wall_seconds": 3600,\n    "whole_job_wall_seconds": 14400,\n    "rss_bytes": 25769803776,\n    "start_free_disk_bytes": 17179869184,\n    "whole_output_bytes": 12884901888,\n    "whole_writer_seconds": 1200,\n    "whole_observer_seconds": 1200,\n    "dense_pairs_per_packet": 1000000,\n    "nodes_per_side": 2048,\n    "dense_pairs_per_run": 450000000\n  },\n  "expected_pair_packets": 3564,\n  "expected_dense_pairs": 341106074,\n  "diagnostic12": [\n    "44b6_12dfb391",\n    "44b6_267148e4",\n    "44b6_2a2eff9f",\n    "44b6_341df25f",\n    "44b6_587a1e22",\n    "44b6_5f15d135",\n    "6bba_062c8d37",\n    "6bba_07e24132",\n    "6bba_085bf656",\n    "6bba_09961292",\n    "6bba_0e7c0d07",\n    "6bba_12665c0e"\n  ],\n  "parent_only": true,\n  "seed": 23826,\n  "collection_runner_implemented": false,\n  "submission_authorized": false,\n  "gt_read": false,\n  "training_started": false,\n  "generalization_evidence": false\n}\n')
D3_REFERENCE = load_reference(D3_REFERENCE_PATH)
D3_EXPECTED_SOURCE = {'/kaggle/working/collection_payload/scripts/e23_association_bridge.py': '88b4df5da2493f719145ac20a3cefc0ad18407acddbe8cac5acdbebc11fa4bcf', '/kaggle/working/collection_payload/src/biohub/__init__.py': '5cac316fbfce0ba07ad83c869a9b7edb239df6f59c7d409c3697fe2fbd4ac24d', '/kaggle/working/collection_payload/src/biohub/association_capture.py': '9015430194aad491bbc0df722fa966d9a21c469b29b833b090a6d445ccf4da36', '/kaggle/working/collection_payload/src/biohub/association_instrumentation.py': '9fa79df7176d1f09ba840247dca8a71da204f62a2dbf877a2b8aa5617333b795', '/kaggle/working/collection_payload/src/biohub/association_observer.py': '4615cce9840cc260530e965b4ccf6ecb47ed096335260a2148c67fa1eed45425', '/kaggle/working/collection_payload/src/biohub/association_parity.py': '3d0188c69c0b1d29c4cd0b057098df1adc8e24e3d35468b7df9f36a78557ecf7', '/kaggle/working/collection_payload/src/biohub/association_supervise.py': '457b8e3505b29bbf123c2effea4e56054a900f06777f71bc4b18b586d59692c6', '/kaggle/working/tracking_repo/scripts/augmentations.py': '13db09817bf492f8d0f710a0a4d09776320b262060167055090a303fc6057f4e', '/kaggle/working/tracking_repo/scripts/dataspec.py': 'e69bf952fb985477ac50ff8598a35020c95d20a035a09b81ab4056e655dd311f', '/kaggle/working/tracking_repo/scripts/e23_observer_off.py': '8e7ac19e8b436d6777576b7ea2269464c4ed8842990b45448d40bf178f8f62de', '/kaggle/working/tracking_repo/scripts/e23_observer_on.py': 'e2fda47e41d970a2c650bbbbb5fe068b6e1b9667fdd91cb59cc2a3ac8ba95a9c', '/kaggle/working/tracking_repo/scripts/evaluate.py': '614813cc51c3581c6ccda4bb20725a19da8ecac4a27620654bfca58319cffa3c', '/kaggle/working/tracking_repo/scripts/predict_unet_transformer.py': '8e7ac19e8b436d6777576b7ea2269464c4ed8842990b45448d40bf178f8f62de', '/kaggle/working/tracking_repo/scripts/train_unet_transformer.py': 'c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea', '/kaggle/working/tracking_repo/src/biohub_tracking/__init__.py': '26a18d8da84e40da73281a48ebc3017d847a2e57431ab63e8629d2109e6e8571', '/kaggle/working/tracking_repo/src/biohub_tracking/division_metrics.py': 'd1cf1e0a43009d02174f1699ce2aa28458a2220ac4b521731d3bcf31cf8c76be', '/kaggle/working/tracking_repo/src/biohub_tracking/img_proc.py': '00e8ef0adc8b39f1aaaa547ea6197b906bf9e8c009e339d3e95f8f8dbf31be3f', '/kaggle/working/tracking_repo/src/biohub_tracking/io.py': 'efae135b088cecaab463d889f16c885ef6da3ad27b0747327d8ddc28d866b7bd', '/kaggle/working/tracking_repo/src/biohub_tracking/metrics.py': '31baf45b54c78f68bab4f65dd8f4b38bca702abb644171c6df7c46cdeef55d83', '/kaggle/working/tracking_repo/src/biohub_tracking/models/__init__.py': 'ab7587ef79856bae50d24b62e5805092d0459ee1c586522b763f9ef70c093e1d', '/kaggle/working/tracking_repo/src/biohub_tracking/models/simple_node_transformer.py': 'b97209edeb03840e80d903e3e2a8c81c520641c8ef343f6ca2904d0f80db064e', '/kaggle/working/tracking_repo/src/biohub_tracking/models/temporal_unet.py': 'd809c35d42f504161074ddeaaa7aee5b407e5bca7f9b4e1d5f9b2ff345666cac', '/kaggle/working/collection_payload/src/biohub/association_collection.py': '3e3bb41550a107a4f85d2d1465fab8cbcb372aa8538363878818a70326d3a5cb', '/kaggle/working/collection_payload/src/biohub/association_collection_supervise.py': 'cc8a38c1e96da685d964133d02faca3931cdd924dccb41901243d1b25bdb2759', '/kaggle/working/collection_payload/scripts/e23_association_collect.py': '3e7416b41a69ce91ce49de106c57f54a996231b0079e21d10d2da5ef1bebefd6'}
D3_INVENTORY = {str(p): _sha(p) for p in sorted([*REPO_DIR.rglob("*.py"), *D3_PAYLOAD.rglob("*.py")])}
assert D3_INVENTORY == D3_EXPECTED_SOURCE, "collection source differs from verified runtime"
D3_SHARED_RUNTIME = {str(_primary_materialized_path): _primary_expected_sha256,
                     str(SECONDARY_WEIGHTS_PATH): _secondary_expected_sha256,
                     str(_deepcenter_materialized_path): _deepcenter_expected_sha256,
                     str(_primary_materialized_path.parent / "config.json"): 'e9b4e396c58081bca08adf8275bd0bd1c2d3fd6eb091a1912a5116cb6de7b50a',
                     str(SECONDARY_CONFIG_PATH): 'e9b4e396c58081bca08adf8275bd0bd1c2d3fd6eb091a1912a5116cb6de7b50a',
                     str(D3_REFERENCE_PATH): _sha(D3_REFERENCE_PATH)}
for path, digest in D3_SHARED_RUNTIME.items():
    assert _sha(Path(path)) == digest, path
D3_PLANS = WORKING_DIR / "association_collection_plans"
D3_VIEWS = WORKING_DIR / "association_collection_images"
D3_PLANS.mkdir(exist_ok=False)
D3_VIEWS.mkdir(exist_ok=False)
D3_GROUP_PLANS = []
for group in D3_REFERENCE["groups"]:
    gid, names = group["group_id"], group["datasets"]
    image_view = D3_VIEWS / gid
    image_view.mkdir(exist_ok=False)
    for name in names:
        (image_view / (name + ".zarr")).symlink_to(D3_TRAIN_IMAGES / (name + ".zarr"), target_is_directory=True)
    assert set(p.name for p in image_view.iterdir()) == {n + ".zarr" for n in names}
    split_path = D3_PLANS / (gid + "_splits.json")
    with split_path.open("x") as stream:
        json.dump([{"split": 0, "train": [], "test": names}], stream)
    plan = {"schema_version": "biohub.association.collection36.group.v1", "group_id": gid,
            "reference_path": str(D3_REFERENCE_PATH), "data_dir": str(image_view),
            "datasets": {n: D3_REFERENCE["datasets"][n] for n in names},
            "splits_file": str(split_path), "primary_weights": str(_primary_materialized_path),
            "source_path": str(D3_SOURCES["on"]), "source_sha256": receipt["instrumented_sha256"],
            "source_inventory": [{"path": p, "sha256": digest} for p, digest in D3_INVENTORY.items()],
            "runtime_file_sha256": {**D3_SHARED_RUNTIME, str(split_path): _sha(split_path)},
            "submission_authorized": False}
    plan_path = D3_PLANS / (gid + ".json")
    with plan_path.open("x") as stream:
        json.dump(plan, stream, sort_keys=True, indent=2)
    D3_GROUP_PLANS.append({"group_id": gid, "path": str(plan_path), "sha256": _sha(plan_path)})
D3_MASTER = {"schema_version": "biohub.association.collection36.master.v1",
             "reference_path": str(D3_REFERENCE_PATH), "repo_dir": str(REPO_DIR),
             "expected_dependencies": {'geff': '1.2.0.1.1', 'numpy': '2.0.2', 'polars': '1.42.0', 'pyscipopt': '6.2.1', 'scipy': '1.16.3', 'torch': '2.10.0+cu128', 'tracksdata': '0.1.0rc6.dev3+g980c2d30a', 'zarr': '3.2.1'},
             "group_plans": D3_GROUP_PLANS, "submission_authorized": False}
D3_MASTER_PATH = WORKING_DIR / "association_collection_master.json"
with D3_MASTER_PATH.open("x") as stream:
    json.dump(D3_MASTER, stream, sort_keys=True, indent=2)
print("D3 COLLECTION MASTER SEALED", _sha(D3_MASTER_PATH), flush=True)


In [ ]:
from biohub.association_collection_supervise import supervise_collection
D3_COLLECTION = supervise_collection(D3_MASTER_PATH, WORKING_DIR / "association_collection_run", D3_PAYLOAD, D3_STARTED)
assert D3_COLLECTION["status"] == "COLLECTION36_COMPLETE_REFERENCE_MATCHED"
assert not (WORKING_DIR / "submission.csv").exists()
